# Project 3. 최종 모델링·평가·2026년 예측·인력 배치 (운영 전달본)

## 1. 목적과 설계 요약
궁·종묘·왕릉 23개 개소의 일 방문객을 **2024년까지 초기 학습한 뒤 2025년을 14일 단위로 순차 예측하는 rolling-origin 백테스트**(약 27개 예측 원점)로 모델 14개 행을 같은 조건에서 비교하고, 선정한 모델을 2023~2025년 전체로 다시 적용해 **2026-01-01~14 개소별 방문객과 운영 인력 배치표**를 만듭니다. 2025년 평가는 "고정된 단일 홀드아웃"이 아니라 각 원점에서 그 원점 이전 자료만 쓰는 순차 백테스트입니다. 기존 원본·전처리 파일은 수정하지 않고, 결과는 `outputs/final_operational_delivery/`에만 저장합니다.

| 구분 | 내용 |
|---|---|
| 입력 | `dataset/calendar_kr_2023_2025.csv`, `calendar_kr_2026.csv`, `heritage_visitors_daily_preprocessed_2023_2025_with_events.csv`, `weather_daily_preprocessed_2023_2025.parquet`, `sites.csv` (장기 근로자의 날 근거에만 `heritage_visitors_daily.csv` 2015~2025 사용) |
| 기간 | 초기 학습 2023-01-01~2024-12-31 → 2025 순차 백테스트(14일, 원점 이전 자료만) → 최종 학습 2023-01-01~2025-12-31 → 예측 2026-01-01~14 |
| 목표 | 총방문객 = 내국인 + 외국인(외국인 = 영어·일본어·중국어·기타 합산). 총방문객·내국인·외국인을 각각 예측하고 합계 정합성을 검산 |
| 공변량 | **휴일 묶음** `is_holiday`(공식 공휴일) + `is_labor_day`(05-01 근로자의 날), **강수 묶음** 4열(`rain_prob`, `rain_1_10_prob`, `heavy_prob`, `after_heavy_prob`), **무료개방** `free_open`. 기온·중국/일본 공휴일·국군의 날 전용 피처는 쓰지 않음 |

## 2. 모델 (실제 비교 행 수는 실행 중 자동 계산)
베이스라인 3(나이브, 계절 나이브 7일, 직전 7일 평균), ARIMA, SARIMA, 글로벌 MLP(날짜·요일·월·개소와 방문객 이력만 사용, 공변량 없음), Chronos-2 제로샷 및 공변량 조합 8종(제로샷, +휴일, +강수, +무료개방, +휴일+강수, +휴일+무료개방, +강수+무료개방, +휴일+강수+무료개방). 근로자의 날 ablation 전용으로 `+ 공식 공휴일 + 무료개방`(근로자의 날 제외) 한 모델을 추가로 계산하지만 모델 선정 대상(14행)에는 넣지 않습니다.

## 3. 강수 피처 (누수 방지)
구간 경계는 `RAIN_MM = 1.0`, `HEAVY_MM = 10.0` 두 상수만 사용합니다: `rain` = 일강수량 ≥ 1mm, `rain_1_10` = 1mm 이상 10mm 미만, `heavy` = 10mm 이상, `after_heavy` = 전날이 큰비이고 당일은 큰비가 아닌 날. 모델에는 **실제 당일 강수가 아니라 기후 평균 기반 사전 추정치**를 넣습니다: 개소별로 `월×요일 → 월 → 계절 → 전체` 순으로 표본이 모자라면 한 단계씩 축소(백오프)한 발생확률이며, 2025년 백테스트용은 **2023~2024년 자료만**, 2026년 예측용은 **2023~2025년 자료만** 사용합니다. 2025년 실제 강수는 조건별 사후 평가(비 안 옴/비 옴)에만 쓰고 "사후 평가용 실측값"으로 표시하며, 모델 선정·예측에는 쓰지 않습니다.

## 4. 평가와 선정
- 지표: **WAPE(1순위)**, RelMAE(계절 나이브 대비 보조), q80 적중률, 80% 구간 포함률, 평균 구간 길이. WAPE 분모(Σ|실제|)가 0인 묶음은 값을 비워 두고(NaN) 순위·선정에서 제외하며, 휴관일(방문객 NaN)은 학습·평가에서 일관되게 제외합니다.
- 모든 모델에 같은 split conformal 80% 구간(2024-07-03~12-31 보정)을 적용하고 **q80 상당 값**으로 인력을 산정합니다(적중률 = 실제 ≤ q80 상당 값인 비율).
- 인력: `ceil(방문객/2000)`, 개방일 최소 1명, 휴관일 0명, 부족 1인·일당 600,000원, 잉여 150,000원, 임계비율 600,000/(600,000+150,000) = 0.8 → 운영 예측은 q80. **실제 근무표가 없으므로 "실제 방문객 환산 필요 인원"과 "모델 권고 인원"으로만 표현**합니다.
- 최종 모델 선정 순서: WAPE → 총비용 → 공휴일·연휴·무료개방일의 부족 인원과 적중률 → RelMAE → 포함률(목표 80% 근접도) → 구간 길이 → 복잡도. 정확도·비용 최우선 모델과 단순성·운영 편의 실무 대안을 모두 제시하고 하나를 권고합니다.

## 4. 설정
- 모든 난수 시드를 고정하고(`SEED = 42`, MLP 반복 시드 0~29) 재현 조건은 `logs/`에 기록합니다. 캐시 폴더 `cache/final_operational_delivery`에는 정의(공변량 목록·기간·설정)와 입력 파일 해시가 같은 이전 실행의 예측만 재사용되고, 정의가 바뀐 모델(강수 묶음 4종)과 ablation 모델은 새로 계산합니다.

In [3]:
import hashlib
import platform
import sys
import time
import importlib.util
import inspect
import json
import os
import shutil
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
from IPython.display import Markdown, display
from joblib.externals.loky import get_reusable_executor
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive'   # dataset1이 들어 있는 상위 폴더
os.chdir(PROJECT_ROOT)
print(os.listdir('dataset1'))             # 파일 5개가 보이면 OK

# ---------- 실행 모드 ----------
FAST_DEBUG = os.environ.get("FINAL_FAST_DEBUG", "0") == "1"     # 1이면 3개 개소·일부 구간만 빠르게 점검 (결과는 *_debug 폴더에 저장)
FORCE_RERUN = False       # True면 저장된 예측(cache)을 무시하고 처음부터 다시 실행
N_JOBS = int(os.environ.get("FINAL_N_JOBS", "4"))                # ARIMA/SARIMA 병렬 작업 수

# ---------- 경로 (프로젝트 루트 기준 상대경로) ----------
DATA_DIR = Path("dataset1")
PATHS = dict(visitors=DATA_DIR / "heritage_visitors_daily_preprocessed_2023_2025_with_events.csv",     # 전처리 방문객 + 행사 열 (방문객 값은 행사 열이 없는 전처리 파일과 동일, 행사 열은 이번 비교에서 사용하지 않음)
             sites=DATA_DIR / "sites.csv",
             calendar_hist=DATA_DIR / "calendar_kr_2023_2025.csv",
             calendar_2026=DATA_DIR / "calendar_kr_2026.csv",
             weather=DATA_DIR / "weather_daily_preprocessed_2023_2025.parquet",
             )
SUFFIX = "_debug" if FAST_DEBUG else ""
INPUT_FILES = [PATHS[k] for k in ("visitors", "sites", "calendar_hist", "calendar_2026", "weather")]
INPUT_HASH = hashlib.md5("".join(hashlib.md5(p.read_bytes()).hexdigest() for p in INPUT_FILES if p.exists()).encode()).hexdigest()      # 입력 파일이 바뀌면 cache 서명이 달라진다
OUTPUT_DIR = Path("outputs") / f"final_operational_delivery{SUFFIX}"
OUT_ROOT = OUTPUT_DIR
CACHE_DIR = Path("cache") / f"final_operational_delivery{SUFFIX}"
for _sub in ("figures", "tables", "staffing", "summary", "logs"):
    (OUTPUT_DIR / _sub).mkdir(parents=True, exist_ok=True)
FIG_DIR, TABLE_DIR, STAFF_DIR, SUMMARY_DIR, LOG_DIR = (OUTPUT_DIR / s for s in ("figures", "tables", "staffing", "summary", "logs"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# ---------- 기간·예측 ----------
HISTORY_START, TRAIN_END = pd.Timestamp("2023-01-01"), pd.Timestamp("2024-12-31")
TEST_START, TEST_END = pd.Timestamp("2025-01-01"), pd.Timestamp("2025-12-31")
FINAL_TRAIN_END = pd.Timestamp("2025-12-31")
FORECAST_START, FORECAST_END = pd.Timestamp("2026-01-01"), pd.Timestamp("2026-01-14")
HORIZON, N_FOLDS, SEED = 14, 27, 42
N_CAL_FOLDS = 13                         # 보정(calibration) 구간 수: 2024-07-03 ~ 2024-12-31 (테스트 구간과 같은 14일 격자)
TIMELINE = pd.date_range(HISTORY_START, FORECAST_END)
TARGETS = ["domestic", "foreign_total"]          # 개별 예측 대상, total = 합산
ALL_TARGETS = ["domestic", "foreign_total", "total"]
WEEKDAY_TO_NUMBER = {"월": 0, "화": 1, "수": 2, "목": 3, "금": 4, "토": 5, "일": 6}
GROUPS = ["궁", "종묘", "왕릉"]

# ---------- 예측구간·지표 ----------
COVERAGE_LEVEL = 0.8        # 기존 Chronos-2 비교의 [q10, q90] 구간의 명목 포함률(80%)과 같은 수준, 모든 모델에 동일 적용
MIN_CAL_ROWS = 10 if FAST_DEBUG else 30     # 개소별 보정 오차가 이보다 적으면 오류로 중단 (디버그는 보정 구간이 2개뿐)
TIE_TOLERANCE = 0.1         # %p: WAPE 차이가 이 값 이내면 '유사한 수준' (기존 프로젝트의 동률 기준)
LOW_N_WARNING = 30          # 관측치가 이보다 적은 조건은 '표본 적음(참고)' 표시
METRICS = ["WAPE", "RelMAE", "hit_rate", "coverage", "mean_interval_width"]
STAFF_UNIT = 2000                     # 방문객 2,000명당 운영 인력 1명
SHORTAGE_COST_KRW, OVERSTAFF_COST_KRW = 600_000, 150_000      # 부족 1인·일당 600,000원, 잉여 150,000원 → 임계비율 0.8
MLP_SEEDS = [0, 1, 2] if FAST_DEBUG else list(range(30))      # MLP 시드 반복(0~29), 대표 시드는 SEED(42)
SIMPLICITY_TOL = 0.5                  # %p: WAPE 동률 허용 폭 (Option 2 실무 대안)
RELMAE_TOL, COST_TOL, HIT_TOL = 0.02, 0.03, 0.02     # 동률 허용 폭: RelMAE 절대값, 총비용 비율, 적중률·포함률 절대값
SITE_OVERRIDE_PP = 3.0
GOLDEN_MIN_DAYS = 4                   # 황금연휴: 공휴일을 포함해 쉬는 날(주말·공휴일)이 이 일수 이상 이어지는 구간
T0 = time.time()
PHASES = [("시작", T0)]


def mark(name):
    """단계별 실행 시간 기록 (logs/02_runtime.csv)"""
    PHASES.append((name, time.time()))

# ---------- 강수·무료개방 규칙 ----------
RAIN_MM, HEAVY_MM = 1.0, 10.0
FREE_OPEN_WEEKDAY = 2       # 수요일 → 매월 마지막 수요일(문화가 있는 날)
# 무료개방 적용 개소 범위: 프로젝트에 개소별 무료개방 적용표가 없어 다음 근거로 '전체 개소'로 둔다.
#  - 2023~2024년(학습기간) 방문객 자료에서 마지막 수요일의 무료 관람 비중이 23개 개소 모두 약 88% 이상(전체 평균 99%), 설 연휴 비중은 99% 이상 (섹션 3 출력)
#  - 적용 대상을 줄이려면 이 값을 개소 이름 리스트로 바꾸면 된다 (None = 전체 개소)
FREE_OPEN_SITE_SCOPE = None

# 무료개방 규칙 모드 (환경변수 FINAL_FREE_RULE, 기본 spec)
#  - "spec"  : 요청 명세 그대로 = 문화가 있는 날(매월 마지막 수요일) + 설 연휴. 행사성·비정기 무료개방은 포함하지 않음 (기본, 최종 비교 결과).
#  - "notice": 참고용 변형 = spec + 2025-10-30~2025-11-09 비정기 무료개방(4대궁·종묘·조선왕릉). 2025-10-29는 마지막 수요일이라 spec에도 이미 포함.
#              이 기간의 실제 무료 관람 비중은 95~99%로 확인했다 (노트북 섹션 5 출력). 결과는 *_notice_free 폴더에 따로 저장하며 기본 결과는 바뀌지 않는다.
FREE_OPEN_RULE = "file"      # 2023~2025는 방문객 파일의 검증된 is_free_admission, 2026은 달력 규칙(마지막 수요일·설 연휴)+상시 무료 개소
#  - "chuseok": 참고용 변형 = spec + 추석 연휴(달력에서 추석 연휴로 분류된 공휴일: 추석 전날·당일·다음날·대체휴일). 2023~2025년 추석 연휴의 무료 관람 비중은 95~99.7%(평균 98.7%)로 확인했다.
#              설 연휴를 spec에 넣은 것과 같은 방식이며 2023·2024·2025년에 같은 규칙을 적용한다 (결과는 *_chuseok_free 폴더).
#  - "file"  : 참고용 변형 = 2023~2025는 방문객 파일의 is_free_admission(개소·일 단위 무료개방 표시: 문화가 있는 날, 설·추석 연휴, 행사성 무료개방, 양주온릉 상시 무료 등)을 그대로 사용.
#              2026(파일에 없음)은 spec 달력 규칙(문화가 있는 날·설 연휴)에 더해 상시 무료 개소(최근 6개월 전부 무료로 표시된 개소)를 1로 둔다. 정기 휴관 우선 규칙은 동일.
#              파일 표시는 사후 정리된 정보이므로 행사성 개방 일부는 기준일 시점에 몰랐을 수 있다(사후적, 참고용). 결과는 *_file_free 폴더.
assert FREE_OPEN_RULE in ("spec", "notice", "chuseok", "file"), "FINAL_FREE_RULE은 spec, notice, chuseok, file 중 하나여야 합니다."
# 비정기 무료개방 기간. 공지 게시일을 확인하지 못했다: 해당 구간의 기준일(원점)보다 공지가 늦었다면 미래 정보이므로
# 이 모드의 결과는 '사후적(참고용)' 성능으로만 해석한다. 운영에서는 공식 공지가 확정된 뒤에만 반영할 수 있는 피처다.
FREE_OPEN_NOTICES = [("2025-10-30", "2025-11-09")]

# ---------- 모델 설정 (기존 프로젝트 재사용) ----------
SEASONAL_LAGS = (7, 14, 21, 28, 35, 42, 364)
BASELINE_BASES = {"baseline_naive": "Naive", "baseline_seasonal_naive": "Seasonal Naive", "baseline_mean7": "7-day Mean"}
REFERENCE_MODEL = "baseline_seasonal_naive"       # RelMAE 기준 모델
MLP_FEATURES = ["site", "group", "weekday", "month"]
MLP_PARAMS = dict(hidden_layer_sizes=(64, 32), activation="relu", solver="adam", alpha=0.01, learning_rate_init=0.001,
                  batch_size=128, max_iter=500, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20)
SEASONAL_PERIOD, ADF_ALPHA, EXPLOSION_FACTOR = 7, 0.05, 3
P_RANGE, Q_RANGE = range(3), range(3)
SARIMA_SEASONAL_CANDIDATES = [(1, 1), (0, 1), (1, 0)]
FALLBACK_SOURCES = {"arima": "baseline_naive", "sarima": "baseline_seasonal_naive"}     # 사전 정의한 fallback
CHRONOS_MODEL_ID = "amazon/chronos-2"
QUANTILES, QUANTILE_COLUMNS = [0.1, 0.5, 0.8, 0.9], ["q10", "q50", "q80", "q90"]
FEATURE_VERSION = "v2-staffing-inputs"      # 공변량 정의를 바꾸면 올려서 cache를 무효화

HOLIDAY_FEATURES = ["is_holiday", "is_labor_day"]      # 휴일 묶음: 공식 공휴일 + 근로자의 날(05-01 특수 수요일). 국군의 날 전용 피처는 없음
WEATHER_FEATURES = ["rain_prob", "rain_1_10_prob", "heavy_prob", "after_heavy_prob"]      # 강수 묶음 (기후 평균 기반 사전 추정 확률)
FREE_FEATURES = ["free_open"]
ALL_COVARIATES = HOLIDAY_FEATURES + FREE_FEATURES + WEATHER_FEATURES

# (model_id, 한글명, 짧은 이름, 모델군, 종류, 공변량, 비교안 번호)
_CH = "Chronos-2"
MODEL_REGISTRY = [
    dict(id="baseline_naive", label="나이브", short="나이브", family="베이스라인", kind="baseline", covariates=[], item=1),
    dict(id="baseline_seasonal_naive", label="계절 나이브", short="계절\n나이브", family="베이스라인", kind="baseline", covariates=[], item=1),
    dict(id="baseline_mean7", label="직전 7일 평균", short="7일\n평균", family="베이스라인", kind="baseline", covariates=[], item=1),
    dict(id="arima", label="ARIMA", short="ARIMA", family="ARIMA", kind="stat", covariates=[], item=2),
    dict(id="sarima", label="SARIMA", short="SARIMA", family="SARIMA", kind="stat", covariates=[], item=3),
    dict(id="mlp_global", label="글로벌 MLP", short="글로벌\nMLP", family="글로벌 MLP", kind="mlp", covariates=[], item=4),
    dict(id="chronos_zero", label="Chronos-2 제로샷", short="C2\n제로샷", family=_CH, kind="chronos", covariates=[], item=5),
    dict(id="chronos_holiday", label="Chronos-2 제로샷 + 공휴일", short="C2\n+공휴일", family=_CH, kind="chronos", covariates=HOLIDAY_FEATURES, item=6),
    dict(id="chronos_weather", label="Chronos-2 제로샷 + 날씨", short="C2\n+날씨", family=_CH, kind="chronos", covariates=WEATHER_FEATURES, item=7),
    dict(id="chronos_free", label="Chronos-2 제로샷 + 무료개방", short="C2\n+무료", family=_CH, kind="chronos", covariates=FREE_FEATURES, item=8),
    dict(id="chronos_holiday_weather", label="Chronos-2 제로샷 + 공휴일 + 날씨", short="C2 +공휴\n+날씨", family=_CH, kind="chronos", covariates=HOLIDAY_FEATURES + WEATHER_FEATURES, item=9),
    dict(id="chronos_holiday_free", label="Chronos-2 제로샷 + 공휴일 + 무료개방", short="C2 +공휴\n+무료", family=_CH, kind="chronos", covariates=HOLIDAY_FEATURES + FREE_FEATURES, item=10),
    dict(id="chronos_weather_free", label="Chronos-2 제로샷 + 날씨 + 무료개방", short="C2 +날씨\n+무료", family=_CH, kind="chronos", covariates=WEATHER_FEATURES + FREE_FEATURES, item=11),
    dict(id="chronos_holiday_weather_free", label="Chronos-2 제로샷 + 공휴일 + 날씨 + 무료개방", short="C2 +공휴\n+날씨+무료", family=_CH, kind="chronos",
         covariates=HOLIDAY_FEATURES + WEATHER_FEATURES + FREE_FEATURES, item=12)]
MODELS = [m["id"] for m in MODEL_REGISTRY]
REG = {m["id"]: m for m in MODEL_REGISTRY}
CHRONOS_MODELS = [m["id"] for m in MODEL_REGISTRY if m["kind"] == "chronos"]
assert len(MODELS) == 14 and len({m["item"] for m in MODEL_REGISTRY}) == 12, "14개 실행 결과 / 12개 비교안 구성이 아닙니다."
for _m in MODEL_REGISTRY:
    _m["label"] = _m["label"].replace("공휴일", "휴일 묶음").replace("날씨", "강수 묶음")
ABL_ID = "chronos_holiday_only_free"      # 근로자의 날 ablation 전용 모델 (모델 선정 대상 14행에 포함하지 않음)
REG[ABL_ID] = dict(id=ABL_ID, label="Chronos-2 제로샷 + 공식 공휴일 + 무료개방 (근로자의 날 ablation 전용)", short="A", family=_CH, kind="chronos", covariates=["is_holiday", "free_open"], item=0)
CHRONOS_RUN = CHRONOS_MODELS + [ABL_ID]
assert not any("temp" in c or "t2m" in c for m in MODEL_REGISTRY for c in m["covariates"]), "기온 변수는 쓰지 않습니다."


def parallel_starmap(func, jobs, n_jobs=N_JOBS):
    """작업 목록을 병렬 실행하고 입력 순서대로 결과를 반환한다 (joblib.Parallel 대신 loky 실행기를 직접 사용)."""
    executor = get_reusable_executor(max_workers=n_jobs, timeout=60)
    futures = [executor.submit(func, *job) for job in jobs]
    return [future.result() for future in futures]


print(f"모드: {'FAST_DEBUG(일부 개소·구간)' if FAST_DEBUG else '전체 실행'} | 결과 폴더 {OUTPUT_DIR.as_posix()} | cache {CACHE_DIR.as_posix()} | 모델 {len(MODELS)}개 실행 / 비교안 12개")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['calendar_kr_2023_2025.csv', 'calendar_kr_2026.csv', 'weather_daily_preprocessed_2023_2025.parquet', 'heritage_visitors_daily_preprocessed_2023_2025_with_events.csv', 'sites.csv']
모드: 전체 실행 | 결과 폴더 outputs/final_operational_delivery | cache cache/final_operational_delivery | 모델 14개 실행 / 비교안 12개


## 5. 데이터, 공변량, 모델별 예측
### 5.1 데이터 로드
- 방문객 파일의 값은 고치지 않습니다. 휴관일은 목표값 결측으로 두고 보간하지 않습니다.

In [4]:
VISITORS_ALL = None


def load_inputs():
    global VISITORS_ALL
    """전처리 완료 방문객·개소·달력·날씨를 읽고 구조를 검증한다 (값은 고치지 않음)."""
    for key, path in PATHS.items():
        if key.startswith("earlier") or key.startswith("reference"):
            continue
        if not path.exists():
            raise FileNotFoundError(f"입력 파일이 없습니다: {path}")
    visitors = pd.read_csv(PATHS["visitors"], encoding="utf-8-sig", parse_dates=["date"])
    required = ["date", "group", "site", "paid", "free", "domestic", "foreign_en", "foreign_ja", "foreign_zh", "foreign_etc", "total", "Comment", "is_free_admission", "free_admission_reason", "free_admission_scope", "free_admission_source_urls"]
    missing = [c for c in required if c not in visitors.columns]
    if missing:
        raise ValueError(f"[{PATHS['visitors']}] 필수 열이 없습니다: {missing}")
    VISITORS_ALL = visitors.copy()
    visitors["foreign_total"] = visitors[["foreign_en", "foreign_ja", "foreign_zh", "foreign_etc"]].sum(axis=1, min_count=4)
    observed = visitors["total"].notna()
    mismatch = int((observed & ((visitors["domestic"] + visitors["foreign_total"] - visitors["total"]).abs() > 1e-9)).sum())
    print(f"total == domestic + foreign_total 검증: 관측 {int(observed.sum()):,}행 중 불일치 {mismatch}행 (자동 수정 안 함)")
    assert (visitors["domestic"].isna() == visitors["total"].isna()).all() and (visitors["foreign_total"].isna() == visitors["total"].isna()).all(), "휴관 NaN 패턴이 열마다 다릅니다."
    assert visitors["date"].min() == HISTORY_START and visitors["date"].max() == TEST_END and not visitors.duplicated(["date", "site"]).any()
    sites = pd.read_csv(PATHS["sites"], encoding="utf-8-sig").drop_duplicates("site")
    assert set(visitors["site"]) == set(sites["site"]) and len(sites) == 23
    assert sites["closed_weekday"].isin(WEEKDAY_TO_NUMBER).all(), "sites.csv의 휴관 요일 값을 해석할 수 없습니다."
    calendar_hist = pd.read_csv(PATHS["calendar_hist"], encoding="utf-8-sig", parse_dates=["date"])
    calendar_2026 = pd.read_csv(PATHS["calendar_2026"], encoding="utf-8-sig", parse_dates=["date"])
    for name, frame in (("calendar_hist", calendar_hist), ("calendar_2026", calendar_2026)):
        absent = [c for c in ["date", "weekday", "is_holiday", "holiday_name", "seollal_offset", "chuseok_offset", "is_last_wednesday"] if c not in frame.columns]
        if absent:
            raise ValueError(f"[{PATHS[name]}] 필수 열이 없습니다: {absent}")
    weather = pd.read_parquet(PATHS["weather"])
    absent = [c for c in ["date", "site", "precip_mm"] if c not in weather.columns]
    if absent:
        raise ValueError(f"[{PATHS['weather']}] 필수 열이 없습니다: {absent}")
    weather = weather.assign(date=lambda d: pd.to_datetime(d["date"]))
    return visitors, sites, calendar_hist, calendar_2026, weather


VISITORS, SITES_TABLE, CALENDAR_HIST, CALENDAR_2026, WEATHER = load_inputs()
ALL_SITES = sorted(SITES_TABLE["site"])
DEBUG_SITES = ["경복궁", "종묘", "선릉"]
SITES = DEBUG_SITES if FAST_DEBUG else ALL_SITES
SITE_GROUP = VISITORS.drop_duplicates("site").set_index("site")["group"].to_dict()
SITE_CLOSED_WEEKDAY = SITES_TABLE.set_index("site")["closed_weekday"].map(WEEKDAY_TO_NUMBER).to_dict()
VISITORS = VISITORS.loc[VISITORS["site"].isin(SITES)].copy()
MATRICES = {name: VISITORS.pivot(index="date", columns="site", values=name).reindex(TIMELINE)[SITES] for name in ["domestic", "foreign_total", "total"]}
ACTUAL_LONG = pd.concat([MATRICES[t].rename_axis("date").reset_index().melt(id_vars="date", var_name="site", value_name="actual").assign(target=t) for t in ALL_TARGETS], ignore_index=True)
print(f"방문객 {len(VISITORS):,}행 / 개소 {len(SITES)}곳({'디버그 부분집합' if FAST_DEBUG else '전체'}) / 휴관·결측 행 {int(VISITORS['total'].isna().sum()):,}")
mark("데이터 로드")

total == domestic + foreign_total 검증: 관측 21,512행 중 불일치 0행 (자동 수정 안 함)
방문객 25,208행 / 개소 23곳(전체) / 휴관·결측 행 3,696


### 5.1b 데이터 검증 (결합 키, 누락·중복, 무료개방 표시)
- 결합 키: 방문객+달력은 `date`, 방문객+날씨는 `date + site`. 행 수 증가·중복·날짜 누락·개소명 불일치·결합 실패를 확인하고 하나라도 어긋나면 중단합니다.
- 2026년 달력의 노동절(2026-05-01) `is_holiday = 1`을 확인합니다(이번 14일에는 직접 들어가지 않지만 달력 전체 논리 검증 항목).
- 코리아 그랜드 페스티벌(2025-10-29~11-09) 무료개방 표시를 검증합니다. 개방한 개소는 `is_free_admission = 1`, 정기 휴관(방문객 NaN)은 0을 유지해야 하며, 이미 올바르면 파일을 건드리지 않고 검증 결과만 남깁니다. 어긋난 행이 있으면 수정 전후 행을 출력·저장하고 사유 문구는 중복 없이 병합합니다(원본 파일은 덮어쓰지 않음).

In [5]:
FESTIVAL = dict(name="코리아 그랜드 페스티벌", start=pd.Timestamp("2025-10-29"), end=pd.Timestamp("2025-11-09"), closed_check=("2025-11-04", ["경복궁", "종묘"]))
FREE_TEXT_COLUMNS = ["free_admission_reason", "free_admission_scope", "free_admission_source_urls"]


def month_last_wednesdays(first, last):
    """first~last 기간에 걸친 각 달의 마지막 수요일 (월 전체 날짜에서 계산)."""
    found = []
    for period in pd.period_range(first, last, freq="M"):
        days = pd.date_range(period.start_time, period.end_time.normalize())
        found.append(days[days.dayofweek == 2].max())
    return pd.DatetimeIndex(found)


def merge_text(existing, addition):
    """기존 문구를 보존하고 ' | ' 구분 항목이 없을 때만 추가한다 (같은 문구를 두 번 넣지 않는다)."""
    tokens = [t.strip() for t in str(existing).split("|") if t.strip()] if pd.notna(existing) else []
    if addition and addition not in tokens:
        tokens.append(addition)
    return " | ".join(tokens) if tokens else existing


def festival_violations(visitors):
    """2025-10-29~11-09 기간의 무료개방 표시 위반 행. 개방한 개소(방문객 관측)는 1, 정기 휴관(방문객 NaN)은 0이어야 한다."""
    part = visitors.loc[visitors["date"].between(FESTIVAL["start"], FESTIVAL["end"])]
    open_row, has_name = part["total"].notna(), part["free_admission_reason"].fillna("").str.contains(FESTIVAL["name"])
    flags = pd.DataFrame({"closed_but_flagged": ~open_row & (part["is_free_admission"] == 1), "open_not_flagged": open_row & (part["is_free_admission"] != 1),
                          "open_without_festival_reason": open_row & ~has_name, "closed_comment_lost": ~open_row & ~part["Comment"].fillna("").str.contains("휴관")}, index=part.index)
    return part, flags


def fix_festival_rows(visitors, flags):
    """위반 행만 고친 사본을 반환한다. 방문객 값·휴관 NaN·Comment는 바꾸지 않고, 사유·범위·URL은 기존 내용을 보존하며 중복 없이 병합한다."""
    fixed = visitors.copy()
    on_rows = flags.index[flags["open_not_flagged"] | flags["open_without_festival_reason"]]
    off_rows = flags.index[flags["closed_but_flagged"]]
    reference = visitors.loc[visitors["date"].between(FESTIVAL["start"], FESTIVAL["end"]) & visitors["free_admission_reason"].fillna("").str.contains(FESTIVAL["name"])]
    defaults = {c: (reference[c].mode().iloc[0] if reference[c].notna().any() else np.nan) for c in ("free_admission_scope", "free_admission_source_urls")}
    for index in on_rows:
        fixed.loc[index, "is_free_admission"] = 1
        fixed.loc[index, "free_admission_reason"] = merge_text(fixed.loc[index, "free_admission_reason"], FESTIVAL["name"])
        for column, value in defaults.items():
            if pd.isna(fixed.loc[index, column]):
                fixed.loc[index, column] = value
    fixed.loc[off_rows, "is_free_admission"] = 0
    return fixed


def validate_joins(visitors, sites_table, calendar_hist, calendar_2026, weather, expected_sites):
    """결합 키 검증 표: 모든 항목이 0(또는 기대값)이어야 한다."""
    full_dates = pd.date_range(HISTORY_START, TEST_END)
    calendar_all = pd.concat([calendar_hist[["date"]], calendar_2026[["date"]]], ignore_index=True)
    merged_cal = visitors[["date", "site"]].merge(pd.concat([calendar_hist, calendar_2026], ignore_index=True)[["date"]].drop_duplicates(), on="date", how="left", indicator=True)
    merged_weather = visitors[["date", "site"]].merge(weather[["date", "site"]].drop_duplicates(), on=["date", "site"], how="left", indicator=True)
    per_site_days = visitors.groupby("site")["date"].nunique()
    rows = [
        ("방문객 행 수", len(visitors), len(expected_sites) * len(full_dates)),
        ("date+site 중복 행", int(visitors.duplicated(["date", "site"]).sum()), 0),
        ("2023~2025 날짜 누락(전체 날짜 기준)", int(len(full_dates.difference(pd.DatetimeIndex(visitors["date"].unique())))), 0),
        ("개소별 날짜 누락 일수 합", int((len(full_dates) - per_site_days).abs().sum()), 0),
        ("개소명 불일치(방문객 vs sites.csv)", len(set(visitors["site"]) ^ set(sites_table["site"])), 0),
        ("개소명 불일치(방문객 vs 날씨)", len(set(visitors["site"]) ^ set(weather["site"])), 0),
        ("달력 날짜 중복", int(calendar_all["date"].duplicated().sum()), 0),
        ("2023~2025 달력 날짜 누락", int(len(full_dates.difference(pd.DatetimeIndex(calendar_hist["date"])))), 0),
        ("2026-01-01~14 달력 누락", int(len(pd.date_range(FORECAST_START, FORECAST_END).difference(pd.DatetimeIndex(calendar_2026["date"])))), 0),
        ("달력 결합 실패 건수(방문객 기준)", int((merged_cal["_merge"] == "left_only").sum()), 0),
        ("날씨 결합 실패 건수(방문객 기준)", int((merged_weather["_merge"] == "left_only").sum()), 0),
        ("결합 후 행 수 증가", int(len(merged_cal) - len(visitors)) + int(len(merged_weather) - len(visitors)), 0),
        ("날씨 date+site 중복 행", int(weather.duplicated(["date", "site"]).sum()), 0),
    ]
    table = pd.DataFrame(rows, columns=["검증 항목", "값", "기대값"])
    table["결과"] = np.where(table["값"] == table["기대값"], "PASS", "FAIL")
    return table


def run_data_validation():
    expected_sites = ALL_SITES
    # ---- 1) 결합 키·누락·중복 ----
    join_table = validate_joins(VISITORS_ALL, SITES_TABLE, CALENDAR_HIST, CALENDAR_2026, WEATHER, expected_sites)
    display(join_table)
    assert (join_table["결과"] == "PASS").all(), "결합 키 검증에 실패했습니다. 위 표에서 FAIL 항목을 확인하세요."
    # ---- 2) 2026 달력 ----
    labor = CALENDAR_2026.loc[CALENDAR_2026["date"] == pd.Timestamp("2026-05-01")]
    labor_ok = bool(len(labor) == 1 and int(labor["is_holiday"].iloc[0]) == 1)
    print(f"2026-05-01 노동절 is_holiday = {int(labor['is_holiday'].iloc[0]) if len(labor) else None} / 이름 '{labor['holiday_name'].iloc[0] if len(labor) else None}' → {'PASS' if labor_ok else 'FAIL'}")
    assert labor_ok, "2026-05-01 노동절이 is_holiday=1로 반영되어 있지 않습니다."
    last_wed_check = {y: bool(set(frame.loc[frame["is_last_wednesday"] == 1, "date"]) == set(month_last_wednesdays(frame["date"].min(), frame["date"].max())))
                      for y, frame in (("2023~2025", CALENDAR_HIST), ("2026", CALENDAR_2026))}
    print("달력의 is_last_wednesday가 '월의 마지막 수요일' 계산과 일치:", last_wed_check)
    assert all(last_wed_check.values())
    md_hist = CALENDAR_HIST["date"].dt.strftime("%m-%d")
    hist_flags_ok = bool((CALENDAR_HIST["is_labor_day"].astype(int) == md_hist.eq("05-01").astype(int)).all())
    cal26 = CALENDAR_2026.assign(is_labor_day=lambda d: d["date"].dt.strftime("%m-%d").eq("05-01").astype(int))
    row_may = cal26.loc[cal26["date"] == pd.Timestamp("2026-05-01")].iloc[0]
    labor26_ok = bool(int(row_may["is_holiday"]) == 1 and int(row_may["is_labor_day"]) == 1)
    print(f"근로자의 날 열(2023~2025 달력): 날짜 규칙(05-01)과 일치 {hist_flags_ok} | 2026은 날짜로 생성: 2026-05-01 is_holiday={int(row_may['is_holiday'])}, is_labor_day={int(row_may['is_labor_day'])} → {'PASS' if labor26_ok else 'FAIL'}")
    assert hist_flags_ok and labor26_ok, "근로자의 날 플래그 검증에 실패했습니다."
    print("달력에서 모델에 쓰는 열: date, is_holiday, is_labor_day (공휴일 블록), holiday_name, seollal_offset, chuseok_offset, is_last_wednesday (is_armed_forces_day와 east_asia_holiday_count / east_asia_holiday_any는 사용하지 않음)")
    # ---- 3) 날씨 파생 정의 대조 (새 전처리 파일의 이진 열 vs 일 강수량에서 만든 정의) ----
    w = WEATHER.sort_values(["site", "date"]).copy()
    w["prev_precip"] = w.groupby("site")["precip_mm"].shift(1)
    derived = pd.DataFrame({"is_rain": (w["precip_mm"] >= RAIN_MM).astype(int), "is_heavy_rain": (w["precip_mm"] >= HEAVY_MM).astype(int),
                            "is_after_heavy_clear": ((w["prev_precip"] >= HEAVY_MM) & (w["precip_mm"] < RAIN_MM)).astype(int)})
    weather_diff = {c: int((derived[c].to_numpy() != w[c].astype(int).to_numpy()).sum()) for c in derived.columns if c in w.columns}
    print("날씨 파일의 이진 열과 강수량 정의(rain≥1, heavy≥10, after_heavy)의 불일치 행 수:", weather_diff, "(첫날은 전날 자료가 없어 after_heavy 0)")
    # ---- 4) 코리아 그랜드 페스티벌 무료개방 표시 ----
    part, flags = festival_violations(VISITORS_ALL)
    open_part = part.loc[part["total"].notna()]
    closed_part = part.loc[part["total"].isna()]
    day, closed_sites = FESTIVAL["closed_check"]
    day_rows = part.loc[part["date"] == pd.Timestamp(day)].set_index("site")
    check_day = {"정기 휴관(경복궁·종묘) 방문객 NaN 유지": bool(day_rows.loc[closed_sites, "total"].isna().all()),
                 "정기 휴관(경복궁·종묘) 무료개방 0 유지": bool((day_rows.loc[closed_sites, "is_free_admission"] == 0).all()),
                 "정기 휴관 Comment의 휴관 표기 유지": bool(day_rows.loc[closed_sites, "Comment"].fillna("").str.contains("휴관").all()),
                 "같은 날 개방한 다른 개소 전부 무료개방 1": bool((day_rows.drop(index=closed_sites).dropna(subset=["total"])["is_free_admission"] == 1).all())}
    summary = pd.DataFrame([("기간 행 수", len(part)), ("개방 행(방문객 관측)", len(open_part)), ("개방 행 중 is_free_admission=1", int((open_part["is_free_admission"] == 1).sum())),
                            ("휴관 행(방문객 NaN)", len(closed_part)), ("휴관 행 중 is_free_admission=0", int((closed_part["is_free_admission"] == 0).sum())),
                            ("위반: 휴관인데 1", int(flags["closed_but_flagged"].sum())), ("위반: 개방인데 1 아님", int(flags["open_not_flagged"].sum())),
                            ("위반: 개방 행에 페스티벌 사유 없음", int(flags["open_without_festival_reason"].sum())), ("위반: 휴관 Comment 소실", int(flags["closed_comment_lost"].sum()))], columns=["항목", "값"])
    display(summary)
    print("2025-11-04 점검:", check_day)
    n_bad = int(flags.any(axis=1).sum())
    before_rows = after_rows = pd.DataFrame()
    if n_bad == 0 and all(check_day.values()):
        print("→ 무료개방 표시가 이미 올바르게 반영되어 있습니다. 파일·데이터를 수정하지 않았습니다 (수정 행 0).")
        fixed_rows = 0
    else:
        bad_index = flags.index[flags.any(axis=1)]
        before_rows = VISITORS_ALL.loc[bad_index].copy()
        fixed = fix_festival_rows(VISITORS_ALL, flags)
        VISITORS_ALL.loc[bad_index, ["is_free_admission"] + FREE_TEXT_COLUMNS] = fixed.loc[bad_index, ["is_free_admission"] + FREE_TEXT_COLUMNS]
        VISITORS.loc[VISITORS.index.intersection(bad_index), ["is_free_admission"] + FREE_TEXT_COLUMNS] = fixed.loc[VISITORS.index.intersection(bad_index), ["is_free_admission"] + FREE_TEXT_COLUMNS]
        after_rows = VISITORS_ALL.loc[bad_index].copy()
        fixed_rows = len(bad_index)
        print(f"→ 무료개방 표시 {fixed_rows}행을 메모리에서 수정했습니다(원본 파일은 덮어쓰지 않음). 수정 전 행:")
        display(before_rows[["date", "site", "total", "is_free_admission"] + FREE_TEXT_COLUMNS])
        print("수정 후 행:")
        display(after_rows[["date", "site", "total", "is_free_admission"] + FREE_TEXT_COLUMNS])
        pd.concat([before_rows.assign(version="수정 전"), after_rows.assign(version="수정 후")]).sort_values(["date", "site", "version"]).to_csv(OUTPUT_DIR / "tables" / "free_admission_fix_rows.csv", index=False, encoding="utf-8-sig")
        _, flags_after = festival_violations(VISITORS_ALL)
        assert not flags_after.any().any(), "수정 후에도 위반 행이 남아 있습니다."
    return dict(join_table=join_table, labor_ok=labor_ok, labor26_ok=labor26_ok, hist_flags_ok=hist_flags_ok, festival_summary=summary, festival_day_check=check_day, fixed_rows=fixed_rows, weather_diff=weather_diff, last_wed_check=last_wed_check)


DATA_CHECK = run_data_validation()
mark("데이터 검증")

,검증 항목,값,기대값,결과
0,방문객 행 수,25208,25208,PASS
1,date+site 중복 행,0,0,PASS
2,2023~2025 날짜 누락(전체 날짜 기준),0,0,PASS
3,개소별 날짜 누락 일수 합,0,0,PASS
4,개소명 불일치(방문객 vs sites.csv),0,0,PASS
5,개소명 불일치(방문객 vs 날씨),0,0,PASS
6,달력 날짜 중복,0,0,PASS
7,2023~2025 달력 날짜 누락,0,0,PASS
8,2026-01-01~14 달력 누락,0,0,PASS
9,달력 결합 실패 건수(방문객 기준),0,0,PASS


2026-05-01 노동절 is_holiday = 1 / 이름 '노동절' → PASS
달력의 is_last_wednesday가 '월의 마지막 수요일' 계산과 일치: {'2023~2025': True, '2026': True}
근로자의 날 열(2023~2025 달력): 날짜 규칙(05-01)과 일치 True | 2026은 날짜로 생성: 2026-05-01 is_holiday=1, is_labor_day=1 → PASS
달력에서 모델에 쓰는 열: date, is_holiday, is_labor_day (공휴일 블록), holiday_name, seollal_offset, chuseok_offset, is_last_wednesday (is_armed_forces_day와 east_asia_holiday_count / east_asia_holiday_any는 사용하지 않음)
날씨 파일의 이진 열과 강수량 정의(rain≥1, heavy≥10, after_heavy)의 불일치 행 수: {'is_rain': 0, 'is_heavy_rain': 3, 'is_after_heavy_clear': 0} (첫날은 전날 자료가 없어 after_heavy 0)


,항목,값
0,기간 행 수,276
1,개방 행(방문객 관측),253
2,개방 행 중 is_free_admission=1,253
3,휴관 행(방문객 NaN),23
4,휴관 행 중 is_free_admission=0,23
5,위반: 휴관인데 1,0
6,위반: 개방인데 1 아님,0
7,위반: 개방 행에 페스티벌 사유 없음,0
8,위반: 휴관 Comment 소실,0


2025-11-04 점검: {'정기 휴관(경복궁·종묘) 방문객 NaN 유지': True, '정기 휴관(경복궁·종묘) 무료개방 0 유지': True, '정기 휴관 Comment의 휴관 표기 유지': True, '같은 날 개방한 다른 개소 전부 무료개방 1': True}
→ 무료개방 표시가 이미 올바르게 반영되어 있습니다. 파일·데이터를 수정하지 않았습니다 (수정 행 0).


### 5.2 공변량 생성 (휴일 묶음 · 강수 묶음 · 무료개방)
- `is_labor_day`는 모든 연도에서 월-일이 05-01이면 1(코드로 생성), 강수 묶음은 개소별 기후 확률(월×요일→월→계절→전체 백오프)입니다.

In [6]:
def special_flag(raw, column, month_day):
    """근로자의 날: 모든 연도에서 월-일이 05-01이면 1 (달력 파일의 열이 있어도 날짜로부터 코드에서 생성한다. 원본 달력은 수정하지 않는다)."""
    return raw["date"].dt.strftime("%m-%d").eq(month_day).astype(int)


def build_calendar(calendar_hist, calendar_2026):
    """2023~2025 달력과 2026 달력을 같은 열·같은 정의로 합친다 (날짜 기준 병합)."""
    parts = [pd.DataFrame({"date": raw["date"], "is_holiday": raw["is_holiday"].astype(int), "holiday_name": raw["holiday_name"], "seollal_offset": raw["seollal_offset"], "chuseok_offset": raw["chuseok_offset"], "is_last_wednesday": raw["is_last_wednesday"].astype(int),
                           "is_labor_day": special_flag(raw, "is_labor_day", "05-01")})
             for raw in (calendar_hist, calendar_2026)]
    calendar = pd.concat(parts, ignore_index=True).sort_values("date")
    assert not calendar["date"].duplicated().any(), "달력에 중복 날짜가 있습니다."
    calendar = calendar.set_index("date").reindex(TIMELINE)
    assert calendar["is_holiday"].notna().all(), "달력에 없는 날짜가 있습니다."
    calendar["is_holiday"] = calendar["is_holiday"].astype(int)
    calendar["is_demand_special_day"] = calendar[["is_holiday", "is_labor_day"]].max(axis=1).astype(int)      # 분류용 통합 변수 (모델 입력 아님)
    return calendar


def last_wednesdays(first, last):
    """first~last 기간에 걸친 모든 달의 '마지막 수요일' (월 전체 날짜에서 계산하므로 기간 끝에서 잘린 달도 정확)."""
    found = []
    for period in pd.period_range(first, last, freq="M"):
        days = pd.date_range(period.start_time, period.end_time.normalize())
        found.append(days[days.dayofweek == FREE_OPEN_WEEKDAY].max())
    return pd.DatetimeIndex(found)


def build_free_open(calendar, sites):
    """무료개방 규칙: 문화가 있는 날(매월 마지막 수요일) + 설 연휴(달력에서 설 연휴로 분류된 공휴일). 개소별 열을 반환한다.
    휴관 우선: 정기 휴관 요일이면서 공휴일이 아닌 날은 0 (공휴일 개방 예외는 기존 프로젝트와 같이 공휴일이면 개방으로 본다). 휴관일 방문객(NaN)은 무료개방으로 간주하지 않는다."""
    rule = pd.Series(False, index=TIMELINE)
    rule = rule | (calendar["is_last_wednesday"] == 1)          # 문화가 있는 날: 달력 파일의 매월 마지막 수요일 표시
    seollal = (calendar["is_holiday"] == 1) & calendar["seollal_offset"].notna()
    rule = rule | seollal
    extended_days = pd.Series(False, index=TIMELINE)          # spec 규칙에 더해진 날짜 (notice 모드에서만)
    if FREE_OPEN_RULE == "notice":
        notice = pd.Series(False, index=TIMELINE)
        for start, end in FREE_OPEN_NOTICES:
            notice.loc[start:end] = True
        extended_days = notice & ~rule
        rule = rule | notice
    if FREE_OPEN_RULE == "chuseok":
        chuseok = (calendar["is_holiday"] == 1) & calendar["holiday_name"].fillna("").str.contains("추석")        # 추석 전날·당일·다음날·대체휴일만 (chuseok_offset은 ±15일 창이라 개천절·한글날이 섞이므로 공휴일 이름으로 판별)
        extended_days = chuseok & ~rule
        rule = rule | chuseok
    scope = set(FREE_OPEN_SITE_SCOPE) if FREE_OPEN_SITE_SCOPE else set(sites)
    file_flag = None
    if FREE_OPEN_RULE == "file":
        assert "is_free_admission" in VISITORS_ALL.columns, "방문객 파일에 is_free_admission 열이 없습니다."
        file_flag = VISITORS_ALL.pivot(index="date", columns="site", values="is_free_admission").reindex(TIMELINE)
        observed_total = VISITORS_ALL.pivot(index="date", columns="site", values="total").reindex(TIMELINE).notna()
        recent, recent_open = file_flag.loc["2025-07-01":"2025-12-31"], observed_total.loc["2025-07-01":"2025-12-31"]      # 휴관일(방문객 결측)은 제외하고 개장일만 본다
        always_free = {s for s in sites if recent_open[s].any() and (recent[s][recent_open[s]] == 1).all()}
        print(f"[file 모드] 상시 무료 개소(2025-07~12 전부 무료 표시): {sorted(always_free)}")
    free = {}
    for site in sites:
        closed = (pd.Series(TIMELINE.dayofweek, index=TIMELINE) == SITE_CLOSED_WEEKDAY[site]) & (calendar["is_holiday"] == 0)
        if file_flag is None:
            site_rule = rule
        else:
            site_rule = pd.Series(rule.to_numpy(), index=TIMELINE)           # 2026 구간: spec 달력 규칙 (+ 상시 무료 개소)
            if site in always_free:
                site_rule.loc[TEST_END + pd.Timedelta(days=1):] = True
            site_rule.loc[:TEST_END] = file_flag[site].loc[:TEST_END].fillna(0).astype(bool)      # 2023~2025: 파일 표시 그대로
        free[site] = (site_rule & ~closed & (site in scope)).astype(int)
    return free, rule, seollal, extended_days


def build_weather_daily(weather, sites):
    """개소별 일 강수량에서 강수 구간 이진 변수를 만든다. 경계값은 RAIN_MM·HEAVY_MM 상수 한 곳에서만 쓴다 (2023-01-01의 after_heavy는 전날 자료가 없으므로 0)."""
    daily = {}
    index = pd.date_range(HISTORY_START, TEST_END)
    for site in sites:
        precip = weather.loc[weather["site"] == site].drop_duplicates("date").set_index("date")["precip_mm"].reindex(index)
        assert precip.notna().all(), f"{site}: 강수량 결측이 있습니다 ({int(precip.isna().sum())}일). 임의로 채우지 않습니다."
        frame = pd.DataFrame({"precip_mm": precip})
        frame["rain"] = (precip >= RAIN_MM).astype(int)
        frame["heavy"] = (precip >= HEAVY_MM).astype(int)
        frame["rain_1_10"] = ((precip >= RAIN_MM) & (precip < HEAVY_MM)).astype(int)
        frame["after_heavy"] = ((frame["heavy"].shift(1) == 1) & (frame["heavy"] == 0)).astype(int)
        daily[site] = frame.loc[HISTORY_START:]
    return daily


SEASON_OF_MONTH = {12: "겨울", 1: "겨울", 2: "겨울", 3: "봄", 4: "봄", 5: "봄", 6: "여름", 7: "여름", 8: "여름", 9: "가을", 10: "가을", 11: "가을"}
WEATHER_BASES = ["rain", "rain_1_10", "heavy", "after_heavy"]
WEATHER_MIN_OBS = dict(month_weekday=8, month=30, season=60)       # 해당 단계의 관측일 수가 이보다 적으면 다음(더 넓은) 단계로 축소
WEATHER_LEVELS = ["월×요일", "월", "계절", "전체"]
WEATHER_BACKOFF = {}


def monthly_probabilities(daily, cutoff):
    """cutoff까지의 관측만으로 개소별 강수 구간 발생확률을 만든다. 월×요일 → 월 → 계절 → 전체 순으로 표본이 모자라면 축소(백오프)하며
    사용한 단계를 WEATHER_BACKOFF[cutoff]에 기록한다. 학습·예측 행이 같은 TIMELINE 확률표를 쓴다."""
    prob, level_rows = {}, []
    months, dows = TIMELINE.month, TIMELINE.dayofweek
    seasons = np.array([SEASON_OF_MONTH[m] for m in months])
    key_mw = pd.MultiIndex.from_arrays([months, dows])
    for site, frame in daily.items():
        used = frame.loc[HISTORY_START:cutoff]
        assert used.index.max() <= cutoff, "확률 계산에 cutoff 이후 날씨가 포함되었습니다."
        group_mw = used.groupby([used.index.month, used.index.dayofweek])
        n_mw, p_mw = group_mw.size().reindex(key_mw).fillna(0).to_numpy(), group_mw[WEATHER_BASES].mean().reindex(key_mw).to_numpy()
        group_m = used.groupby(used.index.month)
        n_m, p_m = group_m.size().reindex(months).fillna(0).to_numpy(), group_m[WEATHER_BASES].mean().reindex(months).to_numpy()
        used_season = np.array([SEASON_OF_MONTH[m] for m in used.index.month])
        group_s = used.groupby(used_season)
        n_s, p_s = group_s.size().reindex(seasons).fillna(0).to_numpy(), group_s[WEATHER_BASES].mean().reindex(seasons).to_numpy()
        p_all = np.tile(used[WEATHER_BASES].mean().to_numpy(), (len(TIMELINE), 1))
        level = np.where(n_mw >= WEATHER_MIN_OBS["month_weekday"], 0, np.where(n_m >= WEATHER_MIN_OBS["month"], 1, np.where(n_s >= WEATHER_MIN_OBS["season"], 2, 3)))
        values = np.select([level[:, None] == 0, level[:, None] == 1, level[:, None] == 2], [p_mw, p_m, p_s], default=p_all)
        assert np.isfinite(values).all() and (values >= 0).all() and (values <= 1).all(), "강수 확률 계산에 문제가 있습니다."
        prob[site] = pd.DataFrame(values, index=TIMELINE, columns=[f"{b}_prob" for b in WEATHER_BASES])
        level_rows.append(pd.Series(level).value_counts().reindex(range(4), fill_value=0).rename(site))
    counts = pd.concat(level_rows, axis=1).T
    counts.columns = WEATHER_LEVELS
    WEATHER_BACKOFF[pd.Timestamp(cutoff)] = counts
    return prob


def build_covariate_frames(mode):
    """mode='test': 강수확률을 2023~2024 날씨로, mode='final': 2023~2025 날씨로 계산. 학습·예측 행이 같은 표를 쓴다."""
    cutoff = TRAIN_END if mode == "test" else FINAL_TRAIN_END
    prob = monthly_probabilities(WEATHER_DAILY, cutoff)
    frames = {site: pd.concat([CALENDAR[HOLIDAY_FEATURES], FREE_OPEN[site].rename("free_open"), prob[site]], axis=1)[ALL_COVARIATES].astype(float) for site in SITES}
    assert all(frame.notna().all().all() for frame in frames.values()), "공변량에 결측이 있습니다."
    return frames


CALENDAR = build_calendar(CALENDAR_HIST, CALENDAR_2026)
FREE_OPEN, FREE_RULE, SEOLLAL_DAYS, EXTENDED_DAYS = build_free_open(CALENDAR, SITES)
WEATHER_DAILY = build_weather_daily(WEATHER, SITES)
COVARIATES = {mode: build_covariate_frames(mode) for mode in ("test", "final")}

# ---------- 무료개방 규칙의 근거 (학습기간 2023~2024 방문객 자료의 무료 관람 비중, 결과값 free는 근거 확인에만 사용) ----------
evidence = VISITORS.loc[VISITORS["date"].between(HISTORY_START, TRAIN_END) & (VISITORS["total"] > 0)].assign(free_share=lambda d: d["free"] / d["total"])
evidence["is_last_wed"] = evidence["date"].isin(last_wednesdays(HISTORY_START, TRAIN_END))
evidence["is_seollal"] = evidence["date"].isin(SEOLLAL_DAYS[SEOLLAL_DAYS].index)
normal_wed = evidence.loc[(evidence["date"].dt.dayofweek == FREE_OPEN_WEEKDAY) & ~evidence["is_last_wed"] & (CALENDAR["is_holiday"].reindex(evidence["date"]).to_numpy() == 0)]
print(f"무료 관람 비중 (학습기간): 마지막 수요일 {evidence.loc[evidence['is_last_wed'], 'free_share'].mean():.1%} (개소별 최저 {evidence.loc[evidence['is_last_wed']].groupby('site')['free_share'].mean().min():.1%}), "
      f"설 연휴 {evidence.loc[evidence['is_seollal'], 'free_share'].mean():.1%}, 일반 수요일 {normal_wed['free_share'].mean():.1%}")
print(f"무료개방일 수 (개소당): 2023 {int(FREE_OPEN[SITES[0]].loc['2023'].sum())}일, 2024 {int(FREE_OPEN[SITES[0]].loc['2024'].sum())}일, 2025 {int(FREE_OPEN[SITES[0]].loc['2025'].sum())}일, 2026-01-01~14 {int(FREE_OPEN[SITES[0]].loc[FORECAST_START:FORECAST_END].sum())}일 | 적용 개소 {'전체' if FREE_OPEN_SITE_SCOPE is None else FREE_OPEN_SITE_SCOPE}")
if FREE_OPEN_RULE == "notice":
    window = VISITORS.loc[VISITORS["date"].between(*map(pd.Timestamp, FREE_OPEN_NOTICES[0])) & (VISITORS["total"] > 0)]
    around = VISITORS.loc[VISITORS["date"].between("2025-10-01", "2025-10-28") & (VISITORS["total"] > 0)]
    print(f"[notice 모드] spec에 추가된 무료개방일 {int(EXTENDED_DAYS.sum())}일({FREE_OPEN_NOTICES[0][0]}~{FREE_OPEN_NOTICES[0][1]}). 이 기간 실제 무료 관람 비중 {window['free'].sum() / window['total'].sum():.1%} (직전 2025-10-01~28: {around['free'].sum() / around['total'].sum():.1%}, 근거 확인용이며 모델 입력 아님)")
if FREE_OPEN_RULE == "file":
    on = VISITORS.loc[(VISITORS["is_free_admission"] == 1) & (VISITORS["total"] > 0) & (VISITORS["site"] != "양주온릉")]
    off = VISITORS.loc[(VISITORS["is_free_admission"] == 0) & (VISITORS["total"] > 0)]
    print(f"[file 모드] 파일 표시 무료개방 개소·일 {int((VISITORS['is_free_admission'] == 1).sum()):,}건. 표시일 실제 무료 관람 비중 {on['free'].sum() / on['total'].sum():.1%} (양주온릉 제외), 미표시일 {off['free'].sum() / off['total'].sum():.1%} (근거 확인용, 모델 입력 아님)")
    print(f"[file 모드] 개소당(경복궁 기준) 무료개방일: 2023 {int(FREE_OPEN[SITES[0]].loc['2023'].sum())}일, 2024 {int(FREE_OPEN[SITES[0]].loc['2024'].sum())}일, 2025 {int(FREE_OPEN[SITES[0]].loc['2025'].sum())}일 (spec 규칙은 19일씩)")
if FREE_OPEN_RULE == "chuseok":
    chuseok_ev = VISITORS.loc[VISITORS["date"].isin(EXTENDED_DAYS[EXTENDED_DAYS].index) & (VISITORS["total"] > 0)]
    print(f"[chuseok 모드] spec에 추가된 무료개방일 {int(EXTENDED_DAYS.sum())}일(2023~2026-01-14: 추석 연휴). 해당일 실제 무료 관람 비중 {chuseok_ev['free'].sum() / chuseok_ev['total'].sum():.1%} (n={len(chuseok_ev):,}개소·일, 근거 확인용이며 모델 입력 아님)")
prob_view = pd.DataFrame({m: COVARIATES[m][SITES[0]].loc[f"2026-01-01", WEATHER_FEATURES] for m in ("test", "final")})
print(f"{SITES[0]} 1월 강수 피처: 2025 테스트용(2023~2024 기준)={prob_view['test'].round(3).to_dict()} / 2026 예측용(2023~2025 기준)={prob_view['final'].round(3).to_dict()}")
mark("공변량 생성")

[file 모드] 상시 무료 개소(2025-07~12 전부 무료 표시): ['양주온릉']
무료 관람 비중 (학습기간): 마지막 수요일 99.0% (개소별 최저 87.7%), 설 연휴 99.6%, 일반 수요일 66.1%
무료개방일 수 (개소당): 2023 21일, 2024 26일, 2025 34일, 2026-01-01~14 0일 | 적용 개소 전체
[file 모드] 파일 표시 무료개방 개소·일 2,704건. 표시일 실제 무료 관람 비중 98.5% (양주온릉 제외), 미표시일 57.3% (근거 확인용, 모델 입력 아님)
[file 모드] 개소당(경복궁 기준) 무료개방일: 2023 21일, 2024 26일, 2025 34일 (spec 규칙은 19일씩)
경복궁 1월 강수 피처: 2025 테스트용(2023~2024 기준)={'rain_prob': 0.0, 'rain_1_10_prob': 0.0, 'heavy_prob': 0.0, 'after_heavy_prob': 0.0} / 2026 예측용(2023~2025 기준)={'rain_prob': 0.0, 'rain_1_10_prob': 0.0, 'heavy_prob': 0.0, 'after_heavy_prob': 0.0}


### 5.3 백테스트 구간
- 테스트 27개 구간(14일 단위), 보정용 13개 구간(2024-07-03~2024-12-31), 2026 최종 예측 구간. 2025 평가는 고정된 단일 홀드아웃이 아니라 2024년까지 초기 학습 후 2025년 순차 rolling-origin 백테스트입니다.

In [7]:
def create_test_windows():
    starts = pd.date_range(TEST_START, TEST_END, freq=f"{HORIZON}D")
    windows = pd.DataFrame({"fold": [f"fold_{i:02d}" for i in range(1, len(starts) + 1)], "origin": starts - pd.Timedelta(days=1), "test_start": starts,
                            "test_end": [min(s + pd.Timedelta(days=HORIZON - 1), TEST_END) for s in starts]})
    assert len(windows) == N_FOLDS and ((windows["test_end"] - windows["test_start"]).dt.days + 1).iloc[-1] == 1
    return windows.assign(phase="test")


def create_calibration_windows():
    """보정용 rolling-origin 구간: 테스트와 같은 14일 격자의 2024-07-03~2024-12-31 (13개). 모두 학습기간 안이다."""
    starts = [TEST_START - pd.Timedelta(days=HORIZON * k) for k in range(N_CAL_FOLDS, 0, -1)]
    windows = pd.DataFrame({"fold": [f"cal_{i:02d}" for i in range(1, N_CAL_FOLDS + 1)], "origin": [s - pd.Timedelta(days=1) for s in starts], "test_start": starts,
                            "test_end": [s + pd.Timedelta(days=HORIZON - 1) for s in starts]})
    assert windows["test_end"].max() == TRAIN_END and windows["test_start"].min() == pd.Timestamp("2024-07-03")
    return windows.assign(phase="cal_test")


FULL_TEST_WINDOWS, FULL_CAL_WINDOWS = create_test_windows(), create_calibration_windows()
FINAL_WINDOW = pd.DataFrame({"fold": ["final_01"], "origin": [FINAL_TRAIN_END], "test_start": [FORECAST_START], "test_end": [FORECAST_END], "phase": "final"})
if FAST_DEBUG:      # 디버그: 테스트 4개 구간(처음 2·마지막 2), 보정 2개 구간
    TEST_WINDOWS, CAL_WINDOWS = FULL_TEST_WINDOWS.iloc[[0, 8, 19, 26]].reset_index(drop=True), FULL_CAL_WINDOWS.iloc[-2:].reset_index(drop=True)
else:
    TEST_WINDOWS, CAL_WINDOWS = FULL_TEST_WINDOWS, FULL_CAL_WINDOWS
CAL_FINAL_FOLDS = TEST_WINDOWS.loc[TEST_WINDOWS["test_start"] >= pd.Timestamp("2025-07-01"), "fold"].tolist()      # 2026용 보정: 2025-07 이후 테스트 구간의 오차
CAL_FINAL_CUTOFF = TEST_WINDOWS.loc[TEST_WINDOWS["fold"] == CAL_FINAL_FOLDS[0], "origin"].iloc[0]                  # 2026 보정용 MLP는 이 날짜까지만 학습
CAL_TEST_CUTOFF = CAL_WINDOWS["origin"].min()                                                                       # 테스트 보정용 MLP는 이 날짜까지만 학습
RUN_WINDOWS = pd.concat([CAL_WINDOWS, TEST_WINDOWS], ignore_index=True)
print(f"창 구성: 보정 {len(CAL_WINDOWS)}개({CAL_WINDOWS['test_start'].min().date()}~{CAL_WINDOWS['test_end'].max().date()}), 테스트 {len(TEST_WINDOWS)}개, 최종 예측 {FORECAST_START.date()}~{FORECAST_END.date()}; "
      f"2026 보정용 테스트 구간 {len(CAL_FINAL_FOLDS)}개(MLP 학습 종료 {CAL_FINAL_CUTOFF.date()}), 테스트 보정용 MLP 학습 종료 {CAL_TEST_CUTOFF.date()}")

창 구성: 보정 13개(2024-07-03~2024-12-31), 테스트 27개, 최종 예측 2026-01-01~2026-01-14; 2026 보정용 테스트 구간 14개(MLP 학습 종료 2025-07-01), 테스트 보정용 MLP 학습 종료 2024-07-02


### 5.4 베이스라인 3종 (나이브 · 계절 나이브 7일 · 직전 7일 평균)
- 구간 안에서는 앞서 만든 예측값만 재사용하고 테스트 실제값은 쓰지 않습니다.

In [8]:
PRED_COLUMNS = ["model_id", "phase", "fold", "horizon", "date", "site", "target", "prediction", "q10", "q50", "q80", "q90"]


def baseline_next_value(base, target_date, history, last_valid):
    """한 날짜의 베이스라인 예측값 (기존 Project3_Baseline_MLP.ipynb와 같은 규칙). history는 {날짜: 값}(휴관일 NaN 유지, 앞서 예측한 값 포함)."""
    if base == "Naive":
        return last_valid
    if base == "Seasonal Naive":
        for lag in SEASONAL_LAGS:
            value = history.get(target_date - pd.Timedelta(days=lag), np.nan)
            if pd.notna(value):
                return value
        return last_valid
    if base == "7-day Mean":
        previous = [history.get(target_date - pd.Timedelta(days=k), np.nan) for k in range(1, 8)]
        previous = [v for v in previous if np.isfinite(v)]
        return float(np.mean(previous)) if previous else last_valid
    raise ValueError(f"지원하지 않는 베이스라인입니다: {base}")


def forecast_baselines(windows_df):
    """모든 구간·개소·타깃의 베이스라인 재귀 예측. 각 구간은 origin 이하 값만 이력으로 쓰고, 구간 안에서는 앞서 만든 예측값만 재사용한다 (테스트 실제값 누수 없음)."""
    rows = []
    for target in TARGETS:
        matrix = MATRICES[target]
        for w in windows_df.itertuples():
            dates = pd.date_range(w.test_start, w.test_end)
            for site in SITES:
                observed = matrix[site].loc[:w.origin]
                assert observed.index.max() == w.origin, "이력이 origin까지가 아닙니다."
                last0 = float(observed.dropna().iloc[-1])
                for model_id, base in BASELINE_BASES.items():
                    history, last = observed.to_dict(), last0
                    for horizon, date in enumerate(dates, start=1):
                        value = max(float(baseline_next_value(base, date, history, last)), 0.0)
                        history[date], last = value, value
                        rows.append((model_id, w.phase, w.fold, horizon, date, site, target, value))
    frame = pd.DataFrame(rows, columns=PRED_COLUMNS[:8])
    return frame.assign(q10=np.nan, q50=np.nan, q80=np.nan, q90=np.nan)[PRED_COLUMNS]


print("베이스라인 예측 중 (보정·테스트·2026 구간, 재귀 규칙)...")
BASELINE_PRED = forecast_baselines(pd.concat([RUN_WINDOWS, FINAL_WINDOW], ignore_index=True))
print(f"베이스라인 예측 {len(BASELINE_PRED):,}행")
mark("베이스라인")

베이스라인 예측 중 (보정·테스트·2026 구간, 재귀 규칙)...
베이스라인 예측 77,418행


In [9]:
CACHE_INFO = CACHE_DIR / "run_info.json"
FIT_LOGS = []


NEG_COLUMNS = ["model_id", "phase", "fold", "date", "site", "target", "raw_value"]
NEG_ROWS = {}
WEATHER_SPEC = f"rain>={RAIN_MM}mm, rain_1_10=[{RAIN_MM},{HEAVY_MM}), heavy>={HEAVY_MM}mm, after_heavy=prev heavy & today not heavy; site-level climatological probability with backoff month x weekday -> month -> season -> all (min obs 8/30/60); test uses 2023-2024, final uses 2023-2025"


def record_negative(model_id, frame, raw_column="raw_prediction"):
    """0 미만이어서 0으로 보정한 예측(보정 전 값)을 기록한다. frame에는 phase, fold, date, site, target, raw 열이 있어야 한다."""
    neg = frame.loc[frame[raw_column] < 0, ["phase", "fold", "date", "site", "target", raw_column]].rename(columns={raw_column: "raw_value"})
    if len(neg):
        NEG_ROWS.setdefault(model_id, []).append(neg.assign(model_id=model_id)[NEG_COLUMNS])


def neg_frame(model_id):
    parts = NEG_ROWS.get(model_id, [])
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=NEG_COLUMNS)


def read_info():
    return json.loads(CACHE_INFO.read_text(encoding="utf-8")) if CACHE_INFO.exists() else {}


def write_info(model_id, **values):
    info = read_info()
    info[model_id] = {**info.get(model_id, {}), **values}
    CACHE_INFO.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")


def model_signature(model_id):
    """모델 정의·기간·개소·공변량 정의의 서명. 같으면 저장된 예측을 다시 쓴다."""
    spec = REG[model_id]
    extra = {"free_variant": [FREE_OPEN_RULE, FREE_OPEN_NOTICES] + ([int(pd.util.hash_pandas_object(pd.concat([FREE_OPEN[s] for s in SITES]), index=False).sum() % (2**31))] if FREE_OPEN_RULE == "file" else [])} if FREE_OPEN_RULE != "spec" and "free_open" in spec["covariates"] else {}      # 무료개방을 쓰는 모델만 규칙 변형이 서명에 반영된다
    if set(spec["covariates"]) & set(WEATHER_FEATURES):
        extra = {**extra, "weather_spec": WEATHER_SPEC}      # 강수 묶음을 쓰는 모델만 강수 확률 정의가 서명에 반영된다
    payload = json.dumps(dict(model=model_id, covariates=spec["covariates"], sites=SITES, feature_version=FEATURE_VERSION, seed=SEED, inputs=INPUT_HASH, mlp_seeds=MLP_SEEDS, targets=TARGETS, **extra,
                              windows=RUN_WINDOWS[["fold", "origin", "test_start", "test_end"]].astype(str).values.tolist(),
                              train_end=str(TRAIN_END.date()), final_train_end=str(FINAL_TRAIN_END.date()), cal_final_cutoff=str(CAL_FINAL_CUTOFF.date()), cal_test_cutoff=str(CAL_TEST_CUTOFF.date()),
                              mlp=MLP_PARAMS, ts=[list(P_RANGE), list(Q_RANGE), SARIMA_SEASONAL_CANDIDATES, ADF_ALPHA], chronos=[CHRONOS_MODEL_ID, QUANTILES],
                              rain=[RAIN_MM, HEAVY_MM], free=[FREE_OPEN_WEEKDAY, str(FREE_OPEN_SITE_SCOPE)]), sort_keys=True, default=str)
    return hashlib.md5(payload.encode()).hexdigest()


def pred_path(model_id):
    return CACHE_DIR / f"pred_{model_id}.csv"


MAIN_CACHE_DIR = Path("cache") / "final_model_comparison"


def seed_cache_from_main():
    """notice 모드: 무료개방 공변량을 쓰지 않는 모델은 입력이 spec 실행과 같으므로 저장된 spec 예측(서명이 같은 것)을 복사해 재사용한다."""
    main_info_path = MAIN_CACHE_DIR / "run_info.json"
    if FREE_OPEN_RULE == "spec" or FAST_DEBUG or not main_info_path.exists():
        return []
    main_info, seeded = json.loads(main_info_path.read_text(encoding="utf-8")), []
    for model_id in MODELS:
        source = MAIN_CACHE_DIR / f"pred_{model_id}.csv"
        if "free_open" in REG[model_id]["covariates"] or pred_path(model_id).exists() or not source.exists() or model_id not in main_info:
            continue
        if main_info[model_id].get("signature") != model_signature(model_id):
            continue
        for name in (f"pred_{model_id}.csv", f"orders_{model_id}.csv", f"fit_log_{model_id}.csv"):
            if (MAIN_CACHE_DIR / name).exists():
                shutil.copy(MAIN_CACHE_DIR / name, CACHE_DIR / name)
        write_info(model_id, **main_info[model_id])
        seeded.append(model_id)
    return seeded


def load_cached(model_id):
    if FORCE_RERUN or not pred_path(model_id).exists() or read_info().get(model_id, {}).get("signature") != model_signature(model_id):
        return None
    neg_path = CACHE_DIR / f"neg_{model_id}.csv"
    if neg_path.exists():
        loaded = pd.read_csv(neg_path, encoding="utf-8-sig", parse_dates=["date"])
        NEG_ROWS[model_id] = [loaded] if len(loaded) else []
    return pd.read_csv(pred_path(model_id), encoding="utf-8-sig", parse_dates=["date"])


def save_cached(model_id, frame, **info):
    frame[PRED_COLUMNS].to_csv(pred_path(model_id), index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
    neg_frame(model_id).to_csv(CACHE_DIR / f"neg_{model_id}.csv", index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
    write_info(model_id, signature=model_signature(model_id), rows=int(len(frame)), **info)

### 5.5 글로벌 MLP
- 날짜·요일·월·개소와 방문객 이력만 사용(공변량 없음), log1p. 대표 시드 42와 시드 0~29 반복을 모두 저장합니다.

In [ ]:
def mlp_features(frame):
    """기존 MLP 노트북과 같은 피처: 개소·유형(범주) + 요일·월. 날짜·개소 정보만 사용하며 공휴일·강수·무료개방은 넣지 않는다."""
    date = pd.to_datetime(frame["date"])
    return pd.DataFrame({"site": frame["site"].to_numpy(), "group": frame["group"].to_numpy(),
                         "weekday": date.dt.dayofweek.astype(str).to_numpy(), "month": date.dt.month.astype(str).to_numpy()})[MLP_FEATURES]


def fit_mlp(target, cutoff, seed=SEED):
    """cutoff 이하·방문객이 관측된 행으로 학습 (log1p 타깃 + 개소·유형 원핫: 개소 간 규모 차이를 반영하는 기존 방식)."""
    rows = VISITORS.loc[(VISITORS["date"] <= cutoff) & VISITORS[target].notna()].sort_values(["date", "site"])
    assert rows["date"].max() <= cutoff, "MLP 학습 데이터가 cutoff 이후를 포함합니다."
    pipeline = Pipeline([("preprocessing", ColumnTransformer([("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False), MLP_FEATURES)], remainder="drop", sparse_threshold=0)),
                         ("regressor", MLPRegressor(**MLP_PARAMS, random_state=seed, verbose=False))])
    model = TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1, check_inverse=False)
    model.fit(mlp_features(rows), rows[target].to_numpy(dtype=float))
    assert list(mlp_features(rows).columns) == ["site", "group", "weekday", "month"]
    return model, len(rows)


def predict_mlp(model, windows_df, target, phase=None, record=True):
    records = []
    for w in windows_df.itertuples():
        for site in SITES:
            for horizon, date in enumerate(pd.date_range(w.test_start, w.test_end), start=1):
                records.append((phase or w.phase, w.fold, horizon, date, site, SITE_GROUP[site]))
    frame = pd.DataFrame(records, columns=["phase", "fold", "horizon", "date", "site", "group"])
    frame["raw_prediction"] = model.predict(mlp_features(frame))
    if record:
        record_negative("mlp_global", frame.assign(target=target))
    frame["prediction"] = np.clip(frame["raw_prediction"], 0, None)
    return frame.assign(model_id="mlp_global", target=target, q10=np.nan, q50=np.nan, q80=np.nan, q90=np.nan)[PRED_COLUMNS]


def run_mlp():
    """학습 4종: 테스트(≤2024-12-31), 테스트 보정(≤보정 시작 전날), 2026 보정(≤2025-07 이전), 최종(≤2025-12-31). 같은 시드(42)·같은 설정."""
    cached = load_cached("mlp_global")
    if cached is not None:
        print("글로벌 MLP: cache의 저장된 예측을 불러옵니다.")
        return cached
    parts, counts = [], {}
    for target in TARGETS:
        for label, cutoff, windows_df, phase in [("test", TRAIN_END, TEST_WINDOWS, None), ("cal_test", CAL_TEST_CUTOFF, CAL_WINDOWS, None),
                                                   ("cal_final", CAL_FINAL_CUTOFF, TEST_WINDOWS.loc[TEST_WINDOWS["fold"].isin(CAL_FINAL_FOLDS)], "cal_final"),
                                                   ("final", FINAL_TRAIN_END, FINAL_WINDOW, None)]:
            model, n_rows = fit_mlp(target, cutoff)
            counts[f"{target}|{label}"] = dict(train_end=str(cutoff.date()), n_train=int(n_rows))
            parts.append(predict_mlp(model, windows_df, target, phase))
            print(f"  MLP 학습 완료: {target} / {label} (학습 종료 {cutoff.date()}, {n_rows:,}행)", flush=True)
    frame = pd.concat(parts, ignore_index=True)
    save_cached("mlp_global", frame, training=counts)
    return frame


print("글로벌 MLP 학습·예측 중...")
MLP_PRED = run_mlp()
print(f"글로벌 MLP 예측 {len(MLP_PRED):,}행 (phase: {sorted(MLP_PRED['phase'].unique())})")


def run_mlp_seeds():
    """시드 반복(테스트용·보정용 모델만): 대표 시드(42) 결과와 구분해 저장한다."""
    path = CACHE_DIR / "pred_mlp_global_seeds.csv"
    info = read_info().get("mlp_global_seeds", {})
    if not FORCE_RERUN and path.exists() and info.get("signature") == model_signature("mlp_global"):
        print("MLP 시드 반복: cache의 저장된 예측을 불러옵니다.")
        return pd.read_csv(path, encoding="utf-8-sig", parse_dates=["date"])
    parts = []
    for seed in MLP_SEEDS:
        for target in TARGETS:
            for cutoff, windows_df in ((TRAIN_END, TEST_WINDOWS), (CAL_TEST_CUTOFF, CAL_WINDOWS)):
                model, _ = fit_mlp(target, cutoff, seed)
                parts.append(predict_mlp(model, windows_df, target, record=False).assign(seed=seed))
        print(f"  MLP 시드 {seed} 완료", flush=True)
    frame = pd.concat(parts, ignore_index=True)
    frame.to_csv(path, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
    write_info("mlp_global_seeds", signature=model_signature("mlp_global"), seeds=MLP_SEEDS)
    return frame


MLP_SEED_PRED = run_mlp_seeds()
print(f"MLP 시드 반복 예측 {len(MLP_SEED_PRED):,}행")
mark("MLP")

글로벌 MLP 학습·예측 중...
  MLP 학습 완료: domestic / test (학습 종료 2024-12-31, 14,308행)
  MLP 학습 완료: domestic / cal_test (학습 종료 2024-07-02, 10,833행)
  MLP 학습 완료: domestic / cal_final (학습 종료 2025-07-01, 17,899행)
  MLP 학습 완료: domestic / final (학습 종료 2025-12-31, 21,512행)
  MLP 학습 완료: foreign_total / test (학습 종료 2024-12-31, 14,308행)
  MLP 학습 완료: foreign_total / cal_test (학습 종료 2024-07-02, 10,833행)
  MLP 학습 완료: foreign_total / cal_final (학습 종료 2025-07-01, 17,899행)
  MLP 학습 완료: foreign_total / final (학습 종료 2025-12-31, 21,512행)
글로벌 MLP 예측 34,224행 (phase: ['cal_final', 'cal_test', 'final', 'test'])
  MLP 시드 0 완료
  MLP 시드 1 완료
  MLP 시드 2 완료
  MLP 시드 3 완료
  MLP 시드 4 완료
  MLP 시드 5 완료
  MLP 시드 6 완료
  MLP 시드 7 완료
  MLP 시드 8 완료
  MLP 시드 9 완료
  MLP 시드 10 완료
  MLP 시드 11 완료
  MLP 시드 12 완료
  MLP 시드 13 완료
  MLP 시드 14 완료
  MLP 시드 15 완료
  MLP 시드 16 완료
  MLP 시드 17 완료
  MLP 시드 18 완료
  MLP 시드 19 완료
  MLP 시드 20 완료


### 5.6 ARIMA · SARIMA
- log1p, 개소·타깃별 ADF·AIC 차수 선택(≤2024-12-31), 구간별 재적합. 실패 시 사전 정의 fallback(ARIMA→나이브, SARIMA→계절 나이브).

In [ ]:
def fit_arima(y, order, seasonal_order):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        result = ARIMA(y, order=order, seasonal_order=seasonal_order).fit()
    return result, not any(issubclass(w.category, ConvergenceWarning) for w in caught)


def select_site_order(model_id, site, target, y, cutoff):
    """학습 이력(≤ cutoff)만으로 차수 선택: d는 ADF 검정, 나머지는 AIC 최소 (기존 프로젝트와 같은 후보). 테스트(2025) 결과는 쓰지 않는다."""
    assert y.index.max() <= cutoff, "차수 선택에 cutoff 이후 데이터가 포함되었습니다."
    seasonal = model_id == "sarima"
    base = y.diff(SEASONAL_PERIOD) if seasonal else y
    d = 0 if adfuller(base.dropna())[1] < ADF_ALPHA else 1
    if seasonal:
        candidates = [((p, d, q), (P, 1, Q, SEASONAL_PERIOD)) for p in P_RANGE for q in Q_RANGE for P, Q in SARIMA_SEASONAL_CANDIDATES]
    else:
        candidates = [((p, d, q), (0, 0, 0, 0)) for p in P_RANGE for q in Q_RANGE]
    best, failed = None, 0
    for order, seasonal_order in candidates:
        try:
            aic = fit_arima(y, order, seasonal_order)[0].aic
        except Exception:
            failed += 1
            continue
        if np.isfinite(aic) and (best is None or aic < best["aic"]):
            best = dict(order=order, seasonal_order=seasonal_order, aic=aic)
    if best is None:
        return dict(model_id=model_id, site=site, target=target, order=None, seasonal_order=None, aic=np.nan, n_candidates=len(candidates), n_failed_candidates=failed, cutoff=str(cutoff.date()))
    return dict(model_id=model_id, site=site, target=target, d=d, n_candidates=len(candidates), n_failed_candidates=failed, cutoff=str(cutoff.date()), **best)


def select_orders(model_id, cutoff):
    jobs = [(model_id, site, target, np.log1p(MATRICES[target][site].loc[:cutoff]), cutoff) for target in TARGETS for site in SITES]
    return pd.DataFrame(parallel_starmap(select_site_order, jobs))


def forecast_job(model_id, site, target, phase, fold, y, steps, order, seasonal_order):
    """한 개소·타깃·구간 예측. 실패하면 values=None과 사유를 반환한다 (호출부에서 fallback 적용·기록)."""
    try:
        result, converged = fit_arima(y, order, seasonal_order)
        values = np.expm1(np.asarray(result.forecast(steps), dtype=float))
        if not np.isfinite(values).all() or (values > EXPLOSION_FACTOR * np.expm1(y.max())).any():
            return dict(model_id=model_id, site=site, target=target, phase=phase, fold=fold, values=None, reason="비정상 예측(비유한 또는 과거 최대 3배 초과)", converged=converged)
        return dict(model_id=model_id, site=site, target=target, phase=phase, fold=fold, values=np.clip(values, 0, None), raw=values, reason="", converged=converged)
    except Exception as error:
        return dict(model_id=model_id, site=site, target=target, phase=phase, fold=fold, values=None, reason=f"{type(error).__name__}: {error}"[:150], converged=False)


def forecast_stat(model_id, windows_df, orders, baseline_pred):
    """ARIMA/SARIMA 예측. 개소별 차수 선택이 실패했거나 예측이 실패하면 사전에 정의한 fallback(ARIMA→나이브, SARIMA→계절 나이브)을 적용하고 로그에 남긴다."""
    order_map = {(r.site, r.target): (r.order, r.seasonal_order) for r in orders.itertuples()}
    jobs, skipped = [], []
    for target in TARGETS:
        for w in windows_df.itertuples():
            steps = (w.test_end - w.test_start).days + 1
            for site in SITES:
                order, seasonal_order = order_map[(site, target)]
                if not isinstance(order, tuple):
                    skipped.append(dict(model_id=model_id, site=site, target=target, phase=w.phase, fold=w.fold, values=None, reason="차수 선택 실패(모든 후보 적합 실패)", converged=False))
                    continue
                y = np.log1p(MATRICES[target][site].loc[:w.origin])
                assert y.index.max() == w.origin, "이력이 origin까지가 아닙니다."
                jobs.append((model_id, site, target, w.phase, w.fold, y, steps, order, seasonal_order))
    results = skipped + parallel_starmap(forecast_job, jobs)
    fallback = baseline_pred.loc[baseline_pred["model_id"] == FALLBACK_SOURCES[model_id]]
    fallback_map = {key: part.sort_values("horizon")["prediction"].to_numpy() for key, part in fallback.groupby(["phase", "fold", "site", "target"])}
    rows, log = [], []
    for r in results:
        dates = pd.date_range(*windows_df.loc[windows_df["fold"] == r["fold"], ["test_start", "test_end"]].iloc[0])
        values = r["values"] if r["values"] is not None else fallback_map[(r["phase"], r["fold"], r["site"], r["target"])]
        raw_values = r["raw"] if r["values"] is not None else values
        rows.append(pd.DataFrame({"model_id": model_id, "phase": r["phase"], "fold": r["fold"], "horizon": np.arange(1, len(dates) + 1), "date": dates, "site": r["site"], "target": r["target"], "prediction": values, "raw_prediction": raw_values}))
        log.append(dict(model_id=model_id, phase=r["phase"], fold=r["fold"], site=r["site"], target=r["target"], status="정상" if r["values"] is not None else "fallback",
                        fallback_to=FALLBACK_SOURCES[model_id] if r["values"] is None else "", reason=r["reason"], converged=r["converged"]))
    frame = pd.concat(rows, ignore_index=True)
    record_negative(model_id, frame)
    frame = frame.assign(q10=np.nan, q50=np.nan, q80=np.nan, q90=np.nan)[PRED_COLUMNS]
    return frame, pd.DataFrame(log)


def run_stat(model_id):
    cached = load_cached(model_id)
    log_path = CACHE_DIR / f"fit_log_{model_id}.csv"
    if cached is not None and log_path.exists():
        print(f"{REG[model_id]['label']}: cache의 저장된 예측을 불러옵니다.")
        FIT_LOGS.append(pd.read_csv(log_path, encoding="utf-8-sig"))
        return cached
    print(f"{REG[model_id]['label']}: 개소·타깃별 차수 선택 (학습기간 ≤ {TRAIN_END.date()}) ...", flush=True)
    orders = select_orders(model_id, TRAIN_END)
    orders.to_csv(CACHE_DIR / f"orders_{model_id}.csv", index=False, encoding="utf-8-sig")
    print(f"  차수 선택 실패 {int(orders['order'].isna().sum())}건 / {len(orders)}건. 구간별 재적합·예측 중 (작업 {len(orders) * (len(RUN_WINDOWS))}개) ...", flush=True)
    frame, log = forecast_stat(model_id, RUN_WINDOWS, orders, BASELINE_PRED)
    log.to_csv(log_path, index=False, encoding="utf-8-sig")
    FIT_LOGS.append(log)
    save_cached(model_id, frame, orders_cutoff=str(TRAIN_END.date()), n_fallback=int((log["status"] == "fallback").sum()), n_jobs=int(len(log)))
    print(f"  완료: fallback {int((log['status'] == 'fallback').sum())}/{len(log)}건, 수렴 경고 {int((~log['converged'].astype(bool)).sum())}건", flush=True)
    return frame


STAT_PRED = {model_id: run_stat(model_id) for model_id in ("arima", "sarima")}
mark("ARIMA·SARIMA")

### 5.7 Chronos-2 제로샷 8종 + ablation 모델 A
- 모델은 한 번만 로드해 재사용합니다. 서명에 공변량 목록·입력 해시(강수 묶음은 강수 확률 정의 포함)가 들어갑니다.

In [ ]:
def check_chronos():
    """Chronos-2 사용 가능 여부 확인. 패키지는 자동 설치하지 않는다."""
    if importlib.util.find_spec("chronos") is None or importlib.util.find_spec("torch") is None:
        return dict(ok=False, reason="chronos-forecasting 또는 torch가 설치되어 있지 않습니다.", fix="환경에 `pip install \"chronos-forecasting>=2.0\" torch`를 설치한 뒤 노트북을 처음부터 다시 실행하세요.")
    import chronos
    from chronos import Chronos2Pipeline
    if "future_df" not in inspect.signature(Chronos2Pipeline.predict_df).parameters:
        return dict(ok=False, reason=f"chronos-forecasting {chronos.__version__}의 predict_df가 future_df(미래 공변량)를 지원하지 않습니다.", fix="chronos-forecasting>=2.0 으로 업그레이드하세요.")
    return dict(ok=True, version=chronos.__version__)


_PIPELINE = {}


def get_pipeline():
    """모델은 한 번만 로드해 재사용한다 (CPU, zero-shot)."""
    if "pipeline" not in _PIPELINE:
        import torch
        from chronos import Chronos2Pipeline
        torch.manual_seed(SEED)
        np.random.seed(SEED)
        _PIPELINE["pipeline"] = Chronos2Pipeline.from_pretrained(CHRONOS_MODEL_ID, device_map="cpu")
    return _PIPELINE["pipeline"]


def build_target_frames(target, mode):
    """개소별 일자 표: target(휴관 NaN) + 공변량. mode에 따라 강수확률 기준 기간이 다르다 (test: 2023~2024, final: 2023~2025)."""
    return {site: COVARIATES[mode][site].assign(target=MATRICES[target][site].reindex(TIMELINE)) for site in SITES}


def build_chronos_frames(covariates, window, site_frames):
    """한 구간의 (컨텍스트 df, 미래 공변량 future_df). origin 이후의 목표값은 컨텍스트에 포함하지 않는다."""
    dates = pd.date_range(window.test_start, window.test_end)
    contexts, futures = [], []
    for site, frame in site_frames.items():
        past = frame.loc[:window.origin, ["target"] + covariates].rename_axis("timestamp").reset_index().assign(id=site)
        assert past["timestamp"].max() == window.origin, "컨텍스트가 origin을 넘습니다."
        contexts.append(past[["id", "timestamp", "target"] + covariates])
        if covariates:
            futures.append(frame.loc[dates, covariates].rename_axis("timestamp").reset_index().assign(id=site)[["id", "timestamp"] + covariates])
    return pd.concat(contexts, ignore_index=True), (pd.concat(futures, ignore_index=True) if covariates else None)


def run_chronos(pipeline, covariates, windows_df, site_frames, audit=None):
    """한 모델·한 타깃의 모든 구간 예측. 반환: phase, fold, horizon, date, site, q10, q50, q80, q90."""
    results = []
    for w in windows_df.itertuples():
        df, future_df = build_chronos_frames(covariates, w, site_frames)
        if audit is not None:
            audit.append(dict(context_columns=[c for c in df.columns if c not in ("id", "timestamp", "target")], future_columns=[] if future_df is None else [c for c in future_df.columns if c not in ("id", "timestamp")]))
        steps = (w.test_end - w.test_start).days + 1
        output = pipeline.predict_df(df, future_df=future_df, prediction_length=steps, quantile_levels=QUANTILES, id_column="id", timestamp_column="timestamp", target="target").sort_values(["id", "timestamp"])
        results.append(pd.DataFrame({"phase": w.phase, "fold": w.fold, "horizon": output.groupby("id").cumcount().to_numpy() + 1, "date": output["timestamp"].to_numpy(), "site": output["id"].to_numpy(),
                                     **{name: output[str(q)].to_numpy(dtype=float) for name, q in zip(QUANTILE_COLUMNS, QUANTILES)}}))
    return pd.concat(results, ignore_index=True)


NORMALIZED_ROWS = {}


def normalize_chronos(frame):
    """기존 Chronos-2 비교 노트북과 같은 처리: 방문객은 0 미만이 될 수 없으므로 분위수를 0으로 자르고 행 안에서 오름차순으로 맞춘 뒤 점 예측을 q50으로 둔다.
    모델 출력 자체는 바꾸지 않으며 조정된 행 수를 기록한다 (여러 번 적용해도 결과가 같다)."""
    raw = frame[QUANTILE_COLUMNS].to_numpy(dtype=float)
    negative_median = frame.assign(raw_prediction=raw[:, QUANTILE_COLUMNS.index("q50")])
    for model_id, part in negative_median.groupby("model_id"):
        record_negative(model_id, part)
    fixed = np.sort(np.clip(raw, 0, None), axis=1)
    changed = (np.abs(fixed - raw) > 1e-12).any(axis=1)
    out = frame.copy()
    out[QUANTILE_COLUMNS] = fixed
    out["prediction"] = out["q50"]
    for model_id, part in out.assign(changed=changed).groupby("model_id"):
        NORMALIZED_ROWS[model_id] = dict(rows=int(len(part)), adjusted_rows=int(part["changed"].sum()))
    return out


def chronos_frame(model_id, windows_df, mode, audit=None):
    covariates = REG[model_id]["covariates"]
    parts = []
    for target in TARGETS:
        if audit is not None:
            audit[target] = []
        print(f"  예측 중: {model_id} / {target} ({len(windows_df)}개 구간)", flush=True)
        part = run_chronos(get_pipeline(), covariates, windows_df, build_target_frames(target, mode), None if audit is None else audit[target])
        parts.append(part.assign(model_id=model_id, target=target, prediction=part["q50"]))
    return normalize_chronos(pd.concat(parts, ignore_index=True)[PRED_COLUMNS])


def run_chronos_model(model_id):
    cached = load_cached(model_id)
    if cached is not None:
        print(f"{REG[model_id]['label']}: cache의 저장된 예측을 불러옵니다.")
        return normalize_chronos(cached)
    print(f"{REG[model_id]['label']}: 보정 {len(CAL_WINDOWS)}개 + 테스트 {len(TEST_WINDOWS)}개 구간 예측 (공변량 {REG[model_id]['covariates'] or '없음'})", flush=True)
    audit = {}
    frame = chronos_frame(model_id, RUN_WINDOWS, "test", audit)
    save_cached(model_id, frame, covariates=REG[model_id]["covariates"],
                covariate_audit={t: dict(context_columns=sorted(set(sum([x["context_columns"] for x in v], []))), future_columns=sorted(set(sum([x["future_columns"] for x in v], [])))) for t, v in audit.items()})
    return frame


CHRONOS_STATUS = check_chronos()
print("Chronos-2 사용 가능:", CHRONOS_STATUS["ok"], {k: v for k, v in CHRONOS_STATUS.items() if k != "ok"})
assert CHRONOS_STATUS["ok"], f"Chronos-2를 사용할 수 없어 최종 비교를 만들 수 없습니다 (가짜 결과 없음): {CHRONOS_STATUS.get('reason')} / {CHRONOS_STATUS.get('fix', '')}"
CHRONOS_PRED = {model_id: run_chronos_model(model_id) for model_id in CHRONOS_RUN}
print("Chronos-2 8개 모델 예측 준비 완료 | 0 미만 절단·분위수 정렬로 조정된 행(모델 출력의 일부):", {m: f"{v['adjusted_rows']:,}/{v['rows']:,}" for m, v in NORMALIZED_ROWS.items()})
mark("Chronos-2")

In [ ]:
def add_total(pred):
    """내국인·외국인 예측을 같은 (모델·구간·날짜·개소)끼리 합쳐 total 예측(점 예측·분위수 합)을 만든다."""
    keys = ["model_id", "phase", "fold", "horizon", "date", "site"]
    values = ["prediction", "q10", "q50", "q80", "q90"]
    dom = pred.loc[pred["target"] == "domestic"].set_index(keys)[values].sort_index()
    frn = pred.loc[pred["target"] == "foreign_total"].set_index(keys)[values].sort_index()
    assert dom.index.equals(frn.index), "내국인·외국인 예측 행이 다릅니다."
    total = (dom + frn).reset_index().assign(target="total")
    return pd.concat([pred, total[PRED_COLUMNS]], ignore_index=True)


ALL_PRED = add_total(pd.concat([BASELINE_PRED.loc[BASELINE_PRED["phase"] != "final"], MLP_PRED.loc[MLP_PRED["phase"] != "final"], *STAT_PRED.values(), *[f.loc[f["phase"] != "final"] for f in CHRONOS_PRED.values()]], ignore_index=True))
ALL_PRED["date"] = pd.to_datetime(ALL_PRED["date"])
print(f"전체 예측 표: {len(ALL_PRED):,}행 | 모델 {ALL_PRED['model_id'].nunique()}개 | phase {sorted(ALL_PRED['phase'].unique())} | 타깃 {sorted(ALL_PRED['target'].unique())}")
assert set(ALL_PRED["model_id"]) == set(MODELS) | {ABL_ID}, "예측이 없는 모델이 있습니다."

## 6. 모든 모델 공통 예측구간 (split conformal, 목표 포함률 80%)
- **보정 오차**: 학습기간(2024-07-03~2024-12-31)의 rolling-origin 구간(13개)에서 만든 개소·타깃별 오차 `실제 − 예측`만 사용합니다. 테스트(2025) 실제값은 쓰지 않습니다. 글로벌 MLP는 보정 구간 이전(2024-07-02)까지만 학습한 모델로 보정 오차를 만들어 학습 오차(과소 추정)를 피했습니다.
- **구간**: `하한 = max(예측 − Q_abs, 0)`, `상한 = 예측 + Q_abs`, `Q_abs`는 개소별 절대오차의 conformal 분위수 `ceil((n+1)·0.8)/n`. **q80 상당 값**(적중률용) = `max(예측 + S_up, 0)`, `S_up`은 부호 있는 오차의 같은 분위수. 점 예측값은 바꾸지 않습니다.
- **2026 예측용 보정**: 2025-07 이후 테스트 구간의 오차(과거 실적)로 같은 방식의 구간을 만듭니다 (MLP는 2025-07-01까지 학습한 모델의 오차).

In [ ]:
def conformal_value(values, level=COVERAGE_LEVEL):
    """split conformal 분위수: n개 오차의 ceil((n+1)·level)번째 값 (표본이 모자라면 최댓값)."""
    v = np.sort(np.asarray(values, dtype=float))
    assert len(v) > 0, "보정 오차가 없습니다."
    return float(v[min(int(np.ceil((len(v) + 1) * level)), len(v)) - 1])


def calibration_frame(for_final, pred=None):
    """보정용 (예측, 실제) 표. for_final=False: 학습기간 보정 구간. for_final=True: 2025-07 이후 테스트 구간(MLP는 해당 시점까지 학습한 모델의 예측)."""
    pred = ALL_PRED if pred is None else pred
    if not for_final:
        cal = pred.loc[pred["phase"] == "cal_test"]
        assert cal["date"].max() <= TRAIN_END, "보정 오차에 학습기간 이후 날짜가 포함되었습니다."
    else:
        cal = pd.concat([pred.loc[(pred["phase"] == "test") & pred["fold"].isin(CAL_FINAL_FOLDS) & (pred["model_id"] != "mlp_global")],
                         pred.loc[(pred["phase"] == "cal_final") & (pred["model_id"] == "mlp_global")]], ignore_index=True)
        assert cal["date"].max() <= TEST_END, "2026 보정 오차에 2025년 이후 날짜가 포함되었습니다."
    cal = cal.merge(ACTUAL_LONG, on=["date", "site", "target"], how="left", validate="many_to_one").dropna(subset=["actual"])
    return cal.assign(residual=lambda d: d["actual"] - d["prediction"])


def conformal_params(cal):
    rows = []
    for (model_id, target, site), part in cal.groupby(["model_id", "target", "site"]):
        assert len(part) >= MIN_CAL_ROWS, f"{model_id}/{target}/{site}: 보정 오차 {len(part)}개 (< {MIN_CAL_ROWS})"
        rows.append(dict(model_id=model_id, target=target, site=site, q_abs=conformal_value(part["residual"].abs()), s_up=conformal_value(part["residual"]), n_cal=len(part)))
    return pd.DataFrame(rows)


def apply_intervals(frame, params):
    """점 예측에 개소·타깃별 conformal 보정값을 붙여 하한·상한·q80 상당 값을 만든다 (하한·q80은 0 미만이 되지 않게)."""
    out = frame.merge(params, on=["model_id", "target", "site"], how="left", validate="many_to_one")
    assert out["q_abs"].notna().all(), "보정값이 없는 행이 있습니다."
    out["lower_80"] = (out["prediction"] - out["q_abs"]).clip(lower=0)
    out["upper_80"] = out["prediction"] + out["q_abs"]
    out["q80_equiv"] = (out["prediction"] + out["s_up"]).clip(lower=0)
    return out


CAL_TEST = calibration_frame(for_final=False)
PARAMS_TEST = conformal_params(CAL_TEST)

# ---------- 평가 표 (테스트 구간) ----------
EVAL = ALL_PRED.loc[ALL_PRED["phase"] == "test"].merge(ACTUAL_LONG, on=["date", "site", "target"], how="left", validate="many_to_one")
reference = EVAL.loc[EVAL["model_id"] == REFERENCE_MODEL, ["date", "site", "target", "prediction"]].rename(columns={"prediction": "reference"})
EVAL = EVAL.merge(reference, on=["date", "site", "target"], how="left", validate="many_to_one")
EVAL = apply_intervals(EVAL, PARAMS_TEST)
EVAL["group"] = EVAL["site"].map(SITE_GROUP)
EVAL["is_closed"] = EVAL["actual"].isna()
print(f"평가 표 {len(EVAL):,}행 | 모델당 평가 행(휴관 제외, total) {int(((EVAL['model_id'] == MODELS[0]) & (EVAL['target'] == 'total') & ~EVAL['is_closed']).sum()):,} | 보정 오차 개소·타깃·모델 조합 {len(PARAMS_TEST):,}개 (개소별 최소 {int(PARAMS_TEST['n_cal'].min())}개)")
mark("conformal 구간")

## 7. 주 단위 조건 분류 (2025년, 여섯 조건)
- 주는 월~일. 2025년 시작·끝의 **불완전한 주는 제외**하고(`2025-01-06`~`2025-12-28` 완전한 주 51개), 예측은 일 단위로 두고 각 날짜에 해당 개소·주의 조건을 붙여 일별 행으로 지표를 계산합니다.
- **강수 주**: 해당 개소에서 그 주에 하루라도 `precip_mm ≥ 1`이면 *비 오는 주*, 7일 모두 `< 1`이면 *비 안 오는 주*. **실제 강수는 이 평가 그룹에만 쓰고 모델 입력에는 쓰지 않습니다.**
- **공휴일 주** (우선순위 `공휴일+주말 연휴가 있는 주 > 평일 공휴일 > 공휴일 없음`): 토·일 또는 법정·대체공휴일로 이어지는 연속 휴일 중 **공휴일과 주말이 모두 포함되고 3일 이상**인 구간의 날짜가 한 주에 하루라도 걸치면 *공휴일+주말 연휴가 있는 주*, 아니면 월~금에 공휴일이 있으면 *평일 공휴일이 있는 주*, 아니면 *공휴일이 없는 주*.

In [ ]:
RAIN_LABELS = ["비 안 오는 주", "비 오는 주"]
HOLIDAY_LABELS = ["공휴일이 없는 주", "평일 공휴일이 있는 주", "공휴일+주말 연휴가 있는 주"]
CONDITIONS = [f"{r} · {h}" for r in RAIN_LABELS for h in HOLIDAY_LABELS]


def holiday_week_table(calendar):
    """월~일 주별 공휴일 조건. 연속 휴일(주말+공휴일) 블록 중 3일 이상이며 공휴일과 주말이 모두 있으면 '공휴일+주말 연휴' 블록."""
    off = pd.Series((calendar.index.dayofweek >= 5) | (calendar["is_holiday"].to_numpy() == 1), index=calendar.index)
    block = (~off).cumsum()
    qualifying = pd.Series(False, index=calendar.index)
    for _, days in calendar.loc[off].groupby(block[off]):
        if len(days) >= 3 and (days["is_holiday"] == 1).any() and (days.index.dayofweek >= 5).any():
            qualifying.loc[days.index] = True
    first_monday = TEST_START + pd.Timedelta(days=(7 - TEST_START.dayofweek) % 7)
    rows = []
    for start in pd.date_range(first_monday, TEST_END, freq="7D"):
        days = pd.date_range(start, start + pd.Timedelta(days=6))
        if days.max() > TEST_END:
            continue                                                 # 불완전한 주 제외
        weekday_holiday = ((days.dayofweek < 5) & (calendar.loc[days, "is_holiday"].to_numpy() == 1)).any()
        label = "공휴일+주말 연휴가 있는 주" if qualifying.loc[days].any() else ("평일 공휴일이 있는 주" if weekday_holiday else "공휴일이 없는 주")
        rows.append(dict(week_start=start, holiday_week=label))
    return pd.DataFrame(rows)


def build_week_conditions():
    holiday_weeks = holiday_week_table(CALENDAR)
    rows = []
    for site in SITES:
        rain = WEATHER_DAILY[site]["rain"]
        for w in holiday_weeks.itertuples():
            days = pd.date_range(w.week_start, w.week_start + pd.Timedelta(days=6))
            rain_label = RAIN_LABELS[int(rain.loc[days].max() == 1)]
            for date in days:
                rows.append((site, date, w.week_start, rain_label, w.holiday_week, f"{rain_label} · {w.holiday_week}"))
    return pd.DataFrame(rows, columns=["site", "date", "week_start", "rain_week", "holiday_week", "condition"]), holiday_weeks


WEEK_CONDITIONS, HOLIDAY_WEEKS = build_week_conditions()
EVAL = EVAL.merge(WEEK_CONDITIONS, on=["site", "date"], how="left", validate="many_to_one")
print("완전한 주", len(HOLIDAY_WEEKS), "개 | 공휴일 주 분포:", HOLIDAY_WEEKS["holiday_week"].value_counts().to_dict())
cond_counts = WEEK_CONDITIONS.drop_duplicates(["site", "week_start"]).groupby("condition").size().reindex(CONDITIONS, fill_value=0)
print("개소·주 수(여섯 조건):", cond_counts.to_dict())

## 8. 지표 계산 (기존 프로젝트 정의, 다섯 가지만)
- **WAPE (%)** = Σ|실제 − 예측| / Σ|실제| × 100 (1순위) · **RelMAE** = 모델 MAE / 계절 나이브 MAE (같은 행, 보조) · **적중률** = 실제 ≤ q80 상당 값인 행 비율 · **포함률** = 실제가 conformal 80% 구간 `[하한, 상한]` 안인 행 비율 · **평균 구간 길이** = (상한 − 하한)의 평균(명).
- 전체 개소 통합 / 개소별 / 유형별(궁·종묘·왕릉) / 여섯 주간 조건별 / 개소별·주간 조건별을 모두 계산하고, 조건별 결과에는 관측치 수(`n_observations`, 평가 지표가 아니라 해석 참고값)를 함께 둡니다.

In [ ]:
def calculate_metrics(frame, keys):
    """keys 단위 WAPE·RelMAE·적중률·포함률·평균 구간 길이 + 관측치 수 (휴관일 제외). 정의는 기존 Chronos-2 비교 노트북과 같다."""
    valid = frame.loc[~frame["is_closed"]]
    work = valid[keys].assign(abs_error=(valid["actual"] - valid["prediction"]).abs(), ref_error=(valid["actual"] - valid["reference"]).abs(), abs_actual=valid["actual"].abs(),
                              hit=(valid["actual"] <= valid["q80_equiv"]).astype(float), covered=((valid["actual"] >= valid["lower_80"]) & (valid["actual"] <= valid["upper_80"])).astype(float),
                              width=valid["upper_80"] - valid["lower_80"])
    out = work.groupby(keys, observed=True).agg(n_observations=("abs_error", "size"), abs_error=("abs_error", "sum"), ref_error=("ref_error", "sum"), abs_actual=("abs_actual", "sum"),
                                                hit_rate=("hit", "mean"), coverage=("covered", "mean"), mean_interval_width=("width", "mean"))
    out["WAPE"] = out["abs_error"] / out["abs_actual"].where(out["abs_actual"] > 0) * 100
    out["RelMAE"] = out["abs_error"] / out["ref_error"].where(out["ref_error"] > 0)
    out = out[["n_observations"] + METRICS].reset_index()
    out["low_n_note"] = np.where(out["n_observations"] < LOW_N_WARNING, "표본 적음(참고)", "")
    return out


def decorate(table, first):
    """모델 이름·모델군·비교안 번호를 붙이고 열 순서를 정리한다."""
    out = table.assign(model_label=table["model_id"].map(lambda m: REG[m]["label"]), model_family=table["model_id"].map(lambda m: REG[m]["family"]),
                       comparison_item=table["model_id"].map(lambda m: REG[m]["item"]))
    columns = ["model_id", "model_label", "model_family", "comparison_item"] + [c for c in first if c != "model_id"] + ["n_observations"] + [c for c in ("n_site_weeks",) if c in out.columns] + METRICS + ["low_n_note"]
    out["_order"] = out["model_id"].map({m: i for i, m in enumerate(MODELS)})
    return out.sort_values([c for c in first if c != "model_id"] + ["_order"])[columns].reset_index(drop=True)


def sort_by(table, column, order):
    return table.assign(**{column: pd.Categorical(table[column], order, ordered=True)}).sort_values([column]).assign(**{column: lambda d: d[column].astype(str)})


OVERALL = decorate(calculate_metrics(EVAL, ["model_id", "target"]), ["target"])
SITE_METRICS = decorate(calculate_metrics(EVAL.assign(group=EVAL["site"].map(SITE_GROUP)), ["model_id", "target", "site", "group"]), ["target", "group", "site"])
TYPE_METRICS = decorate(calculate_metrics(EVAL, ["model_id", "target", "group"]), ["target", "group"])
WEEK_EVAL = EVAL.dropna(subset=["condition"])
week_pooled = calculate_metrics(WEEK_EVAL, ["model_id", "target", "condition", "rain_week", "holiday_week"])
n_site_weeks = WEEK_EVAL.drop_duplicates(["site", "week_start"]).groupby("condition").size()
week_pooled["n_site_weeks"] = week_pooled["condition"].map(n_site_weeks).fillna(0).astype(int)
WEEKLY_METRICS = decorate(week_pooled, ["target", "condition", "rain_week", "holiday_week"])
WEEKLY_METRICS["_c"] = pd.Categorical(WEEKLY_METRICS["condition"], CONDITIONS, ordered=True)
WEEKLY_METRICS = WEEKLY_METRICS.sort_values(["target", "_c", "comparison_item", "model_id"]).drop(columns="_c").reset_index(drop=True)
SITE_WEEKLY_METRICS = decorate(calculate_metrics(WEEK_EVAL.assign(group=WEEK_EVAL["site"].map(SITE_GROUP)), ["model_id", "target", "site", "group", "condition", "rain_week", "holiday_week"]),
                               ["target", "group", "site", "condition", "rain_week", "holiday_week"])
print(f"지표 표: 전체 {len(OVERALL)}행, 개소별 {len(SITE_METRICS)}행, 유형별 {len(TYPE_METRICS)}행, 주간 조건별 {len(WEEKLY_METRICS)}행, 개소별·주간 조건별 {len(SITE_WEEKLY_METRICS)}행")
display(OVERALL.loc[OVERALL["target"] == "total", ["model_label", "n_observations"] + METRICS].round(3).reset_index(drop=True))
mark("지표 계산")

### 6. 인력 환산 함수와 날짜 속성
- 모든 모델·표에서 같은 함수로 `ceil(방문객/2000)`(개방일 최소 1명, 휴관일 0명), 부족 600,000원·잉여 150,000원을 적용합니다. 날짜 유형(상호 배타): 설·추석 명절 > 황금연휴 > 일반 공휴일 > 근로자의 날 > 무료개방일 > 일반 주말 > 일반 평일.

In [ ]:
# ---------- 인력 환산 함수 (모든 모델·모든 표에서 같은 함수를 쓴다) ----------
STAFF_RULE_TEXT = {"closed": "정기 휴관일: 0명", "palace": "궁·종묘: q80 방문객 기준 max(1, ceil(q80/2,000))",
                   "tomb_normal": "왕릉 일반 개방일: q80 ≤ 2,000명이면 1명", "tomb_over": "왕릉 q80 > 2,000명: ceil(q80/2,000) (2명 이상)"}
CRITICAL_RATIO = SHORTAGE_COST_KRW / (SHORTAGE_COST_KRW + OVERSTAFF_COST_KRW)
assert abs(CRITICAL_RATIO - COVERAGE_LEVEL) < 1e-12, "임계비율이 0.8이 아닙니다."


def to_visitors_int(values):
    return np.rint(np.clip(np.asarray(values, dtype=float), 0, None)).astype(np.int64)


def staff_formula(visitors, unit=None):
    """ceil(방문객 / 2,000) 정수 연산 (2,000명 → 1, 2,001명 → 2, 3,000명 → 2). 0명이면 0."""
    unit = STAFF_UNIT if unit is None else unit
    return (np.asarray(visitors, dtype=np.int64) + unit - 1) // unit


def staff_rule(visitors, closed, unit=None):
    """개방일 최소 1명, 정기 휴관일 0명."""
    return np.where(np.asarray(closed, dtype=bool), 0, np.maximum(1, staff_formula(visitors, unit))).astype(np.int64)


# ---------- 날짜 속성: 공휴일명·근로자의 날·설추석 명절·황금연휴·무료개방·실측 강수 유형 ----------
RAW_CAL = pd.concat([CALENDAR_HIST, CALENDAR_2026], ignore_index=True).set_index("date").sort_index()
assert RAW_CAL.index.is_unique
off_days = (RAW_CAL.index.dayofweek >= 5) | (RAW_CAL["is_holiday"].astype(int) == 1)
block_id = pd.Series((~off_days).cumsum(), index=RAW_CAL.index)
GOLDEN_FLAG = pd.Series(False, index=RAW_CAL.index)
for _, days in RAW_CAL.loc[off_days].groupby(block_id[off_days]):
    if len(days) >= GOLDEN_MIN_DAYS and (days["is_holiday"].astype(int) == 1).any():
        GOLDEN_FLAG.loc[days.index] = True               # 황금연휴: 공휴일을 포함해 쉬는 날(주말·공휴일)이 4일 이상 이어지는 구간의 모든 날
reason_text = VISITORS_ALL["free_admission_reason"].fillna("")
MYEONGJEOL_FLAG = pd.Series(RAW_CAL.index.isin(VISITORS_ALL.loc[reason_text.str.contains("설 연휴|추석 연휴"), "date"].unique()), index=RAW_CAL.index)
MYEONGJEOL_FLAG |= (RAW_CAL["is_holiday"].astype(int) == 1) & RAW_CAL["holiday_name"].fillna("").str.contains("설날|추석")      # 달력 이름 기준(2026 등 방문객 자료가 없는 날짜)
FREE_LONG = pd.concat([FREE_OPEN[s].rename("free_open").rename_axis("date").reset_index().assign(site=s) for s in SITES], ignore_index=True)
WEATHER_LONG = pd.concat([WEATHER_DAILY[s][["precip_mm"]].rename_axis("date").reset_index().assign(site=s) for s in SITES], ignore_index=True)
DATE_TYPES = ["일반 평일", "일반 주말", "일반 공휴일", "황금연휴", "설·추석 명절", "무료개방일", "근로자의 날"]


def add_date_attributes(frame):
    """날짜 유형(상호 배타, 우선순위: 설·추석 명절 > 황금연휴 > 일반 공휴일 > 근로자의 날 > 무료개방일 > 일반 주말 > 일반 평일)과 비배타 플래그를 붙인다."""
    out = frame.merge(FREE_LONG, on=["date", "site"], how="left", validate="many_to_one").merge(WEATHER_LONG, on=["date", "site"], how="left", validate="many_to_one")
    out["is_holiday"] = RAW_CAL["is_holiday"].reindex(out["date"]).to_numpy().astype(int)
    out["is_labor_day"] = out["date"].dt.strftime("%m-%d").eq("05-01").astype(int).to_numpy()
    out["holiday_name"] = RAW_CAL["holiday_name"].reindex(out["date"]).to_numpy()
    out["is_golden_holiday"] = GOLDEN_FLAG.reindex(out["date"]).to_numpy()
    out["is_myeongjeol"] = MYEONGJEOL_FLAG.reindex(out["date"]).to_numpy()
    out["is_weekend"] = (out["date"].dt.dayofweek >= 5).astype(int)
    out["weekday_ko"] = out["date"].dt.dayofweek.map({0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"})
    out["date_type"] = np.select([out["is_myeongjeol"], out["is_golden_holiday"], out["is_holiday"] == 1, out["is_labor_day"] == 1, out["free_open"] == 1, out["is_weekend"] == 1],
                                 ["설·추석 명절", "황금연휴", "일반 공휴일", "근로자의 날", "무료개방일", "일반 주말"], default="일반 평일")
    out["rain_type_actual"] = np.select([out["precip_mm"] >= HEAVY_MM, out["precip_mm"] >= RAIN_MM], ["10mm 이상 큰비", "1~10mm 비"], default="비 안 옴")      # 사후 평가용 실측값
    return out


EVAL_BASE = add_date_attributes(EVAL)
assert EVAL_BASE["date_type"].isin(DATE_TYPES).all()
SPECIAL_DEMAND_TYPES = ["일반 공휴일", "황금연휴", "설·추석 명절", "무료개방일"]       # 선정 기준 3: 공휴일·연휴·무료개방일

STAFF_VARIANTS = {"point": "점 예측 기준", "q80": "q80 기준 (운영 정책)"}


def build_eval_staff(frame):
    """총방문객·개방일 행에 '실제 방문객 환산 필요 인원', 점 예측/q80 기준 모델 권고 인원, 부족·잉여 인원과 비용을 붙인다 (부족 = max(필요 − 권고, 0), 잉여 = max(권고 − 필요, 0))."""
    out = frame.loc[(frame["target"] == "total") & ~frame["is_closed"]].copy()
    out["actual_visitors"] = to_visitors_int(out["actual"])
    out["predicted_visitors"] = to_visitors_int(out["prediction"])
    out["prediction_q80"] = to_visitors_int(out["q80_equiv"])
    out["required_staff_actual"] = staff_formula(out["actual_visitors"])
    open_flags = np.zeros(len(out), dtype=bool)
    out["staff_point"] = staff_rule(out["predicted_visitors"], open_flags)
    out["staff_q80"] = staff_rule(out["prediction_q80"], open_flags)
    for variant in STAFF_VARIANTS:
        gap = out[f"staff_{variant}"] - out["required_staff_actual"]
        out[f"shortage_{variant}"] = (-gap).clip(lower=0)
        out[f"surplus_{variant}"] = gap.clip(lower=0)
        out[f"shortage_cost_{variant}"] = out[f"shortage_{variant}"] * SHORTAGE_COST_KRW
        out[f"surplus_cost_{variant}"] = out[f"surplus_{variant}"] * OVERSTAFF_COST_KRW
        out[f"total_cost_{variant}"] = out[f"shortage_cost_{variant}"] + out[f"surplus_cost_{variant}"]
    return out


EVAL_STAFF = build_eval_staff(EVAL_BASE)
mark("인력 환산 함수·날짜 속성")
print(f"평가 행(총방문객, 개방일) {len(EVAL_STAFF):,}행 | 날짜 유형 분포(모델 1개 기준):", EVAL_STAFF.loc[EVAL_STAFF["model_id"] == MODELS[0], "date_type"].value_counts().to_dict())

## 7. 모델 비교 표 (2025 순차 백테스트, 총방문객)
- 모든 표는 **휴관일(방문객 NaN)을 제외한 개방일 행**만 평가하며, 같은 모델 조건·같은 원점·같은 개소·같은 날짜로 비교합니다. WAPE 분모(Σ|실제|)가 0이면 값을 NaN으로 두고 순위에서 제외합니다.
- 인력 열은 **q80 기준 운영 정책**으로 계산한 `실제 방문객 환산 필요 인원`과 `모델 권고 인원`의 차이입니다(부족 = max(필요 − 권고, 0), 잉여 = max(권고 − 필요, 0)). 직전 7일 평균의 point 방식과 q80 방식은 서로 다른 기준이라 같은 표 안에서 섞지 않고 `staffing_basis` 열로 구분합니다.

In [ ]:
FORECAST_MODELS = list(MODELS)               # 모델 선정 대상 (ablation 전용 모델은 제외)
assert len(FORECAST_MODELS) == len(MODEL_REGISTRY), "모델 등록부와 선정 대상 수가 다릅니다."
N_MODEL_ROWS = len(FORECAST_MODELS)
print(f"실제 비교 행(모델) 수: {N_MODEL_ROWS}개 (베이스라인 {sum(REG[m]['kind'] == 'baseline' for m in FORECAST_MODELS)} + ARIMA·SARIMA·MLP {sum(REG[m]['kind'] in ('stat', 'mlp') for m in FORECAST_MODELS)} + Chronos-2 {sum(REG[m]['kind'] == 'chronos' for m in FORECAST_MODELS)})")
STAFF_SUM_COLS = ["observed_visitors", "predicted_visitors_sum", "q80_visitors_sum", "required_staff_actual", "recommended_staff", "shortage_person_days", "surplus_person_days", "shortage_cost_krw", "surplus_cost_krw", "total_cost_krw"]


def staff_by(keys, variant="q80", models=None, frame=None):
    frame = EVAL_STAFF if frame is None else frame
    frame = frame.loc[frame["model_id"].isin(FORECAST_MODELS if models is None else models)]
    agg = frame.groupby(["model_id"] + keys).agg(observed_visitors=("actual_visitors", "sum"), predicted_visitors_sum=("predicted_visitors", "sum"), q80_visitors_sum=("prediction_q80", "sum"),
                                                 required_staff_actual=("required_staff_actual", "sum"), recommended_staff=(f"staff_{variant}", "sum"), shortage_person_days=(f"shortage_{variant}", "sum"),
                                                 surplus_person_days=(f"surplus_{variant}", "sum"), shortage_cost_krw=(f"shortage_cost_{variant}", "sum"), surplus_cost_krw=(f"surplus_cost_{variant}", "sum")).reset_index()
    agg["total_cost_krw"] = agg["shortage_cost_krw"] + agg["surplus_cost_krw"]
    return agg


def full_table(keys, models=None, variant="q80", frame=None):
    models = FORECAST_MODELS if models is None else models
    base = (EVAL_BASE if frame is None else frame)
    base = base.loc[(base["target"] == "total") & base["model_id"].isin(models)]
    metrics = calculate_metrics(base, ["model_id"] + keys)
    table = metrics.merge(staff_by(keys, variant, models), on=["model_id"] + keys, how="left", validate="one_to_one")
    table.insert(1, "model_label", table["model_id"].map(lambda m: REG[m]["label"]))
    table.insert(2, "model_family", table["model_id"].map(lambda m: REG[m]["family"]))
    table["model_order"] = table["model_id"].map({m: i for i, m in enumerate(MODELS + [ABL_ID])})
    return table


def money_columns(table):
    """원 단위 비용 열 옆에 억 원 단위 열을 함께 둔다."""
    out = table.copy()
    for column in [c for c in out.columns if c.endswith("_krw")]:
        out[column.replace("_krw", "_eok_won")] = out[column] / 1e8
    return out


COMPLEXITY_KIND = {"baseline": 1, "stat": 2, "mlp": 4}


def model_complexity(model_id):
    spec = REG[model_id]
    if spec["kind"] == "baseline":
        return 1
    if spec["kind"] == "stat":
        return 2 if model_id == "arima" else 3
    if spec["kind"] == "mlp":
        return 4
    return 5 + sum(bool(set(block) & set(spec["covariates"])) for block in (HOLIDAY_FEATURES, WEATHER_FEATURES, FREE_FEATURES))


OVERALL = full_table([])
GROUP_T = full_table(["group"])
SITE_T = full_table(["site", "group"])
ORIGIN_T = full_table(["fold"]).merge(TEST_WINDOWS[["fold", "origin", "test_start", "test_end"]], on="fold", how="left")
DATE_TYPE_T = full_table(["date_type"])
EVAL_WEEK = EVAL_BASE.dropna(subset=["condition"]).copy()
CROSS_T = full_table(["rain_week", "holiday_week"], frame=EVAL_WEEK)
display(OVERALL.sort_values("WAPE")[["model_label", "n_observations"] + METRICS + ["shortage_person_days", "surplus_person_days", "total_cost_krw"]].round(3).reset_index(drop=True))
mark("모델 비교 표")

## 8. 최종 모델 선정
선정 순서: ① WAPE ② 인력 총비용(q80 기준) ③ 공휴일·연휴(황금연휴·설추석 명절)·무료개방일의 부족 인원과 적중률 ④ RelMAE ⑤ 포함률이 목표 80%에 얼마나 가까운지 ⑥ 구간 길이 ⑦ 모델 복잡도.
- **정확도·비용 최우선 모델(Option 1)**: 위 순서를 허용오차 없이 그대로 적용한 1위입니다(허용오차 안에 있다는 이유만으로 더 좋은 모델을 제외하지 않음).
- **단순성·운영 편의 실무 대안(Option 2)**: 각 단계에서 *차이가 작은* 후보(WAPE 0.5%p, 총비용 3%, 특별일 부족 5%·적중률 0.02, RelMAE 0.02, 포함률 거리 0.02, 구간 길이 5%)를 동률로 보고 끝까지 남은 후보 중 가장 단순한 모델입니다.
- **최종 권고안**: 두 모델이 다르면 차이를 수치로 보이고, Option 1의 WAPE 우위가 0.5%p 이상이거나 총비용 우위가 3% 이상이면 Option 1(정확도·비용 우선), 그렇지 않으면 Option 2(단순한 모델)를 권고합니다.

In [ ]:
special_rows = EVAL_STAFF.loc[EVAL_STAFF["date_type"].isin(SPECIAL_DEMAND_TYPES) & EVAL_STAFF["model_id"].isin(FORECAST_MODELS)]
special_stats = special_rows.groupby("model_id").agg(special_n=("actual", "size"), special_shortage_person_days=("shortage_q80", "sum"), special_hit_rate=("actual", lambda s: np.nan))
special_stats["special_hit_rate"] = special_rows.assign(hit=lambda d: (d["actual"] <= d["q80_equiv"]).astype(float)).groupby("model_id")["hit"].mean()
SEL_T = OVERALL.set_index("model_id")[["model_label", "n_observations", "WAPE", "RelMAE", "hit_rate", "coverage", "mean_interval_width", "total_cost_krw", "shortage_person_days", "surplus_person_days"]].join(special_stats)
SEL_T["coverage_gap"] = (SEL_T["coverage"] - COVERAGE_LEVEL).abs()
SEL_T["complexity"] = [model_complexity(m) for m in SEL_T.index]
SEL_T = SEL_T.loc[SEL_T["WAPE"].notna()]
STRICT_ORDER = ["WAPE", "total_cost_krw", "special_shortage_person_days", "special_hit_rate_desc", "RelMAE", "coverage_gap", "mean_interval_width", "complexity"]
SEL_T["special_hit_rate_desc"] = -SEL_T["special_hit_rate"]
SEL_T["rank_strict"] = SEL_T.sort_values(STRICT_ORDER).reset_index().reset_index().set_index("model_id")["index"].reindex(SEL_T.index) + 1
OPT1 = SEL_T.sort_values(STRICT_ORDER).index[0]
TOL_STEPS = [("WAPE", "abs", SIMPLICITY_TOL), ("total_cost_krw", "rel", COST_TOL), ("special_shortage_person_days", "rel", 0.05), ("special_hit_rate_desc", "abs", HIT_TOL), ("RelMAE", "abs", RELMAE_TOL), ("coverage_gap", "abs", HIT_TOL), ("mean_interval_width", "rel", 0.05)]
pool, TRACE = SEL_T.copy(), []
for column, kind, tol in TOL_STEPS:
    best = pool[column].min()
    pool = pool.loc[pool[column] <= (best + tol if kind == "abs" else best * (1 + tol) + (1e-9 if best == 0 else 0))]
    TRACE.append(dict(step=column, tolerance=f"{tol:g}{'%p' if column == 'WAPE' else (' 비율' if kind == 'rel' else '')}", candidates_left=len(pool), candidates=", ".join(pool.index)))
OPT2 = pool.sort_values(["complexity", "WAPE"]).index[0]
w1, w2 = SEL_T.loc[OPT1, "WAPE"], SEL_T.loc[OPT2, "WAPE"]
c1, c2 = SEL_T.loc[OPT1, "total_cost_krw"], SEL_T.loc[OPT2, "total_cost_krw"]
wape_gap, cost_gain = w2 - w1, (c2 - c1) / c2
if OPT1 == OPT2:
    FINAL_MODEL, FINAL_REASON = OPT1, "정확도·비용 최우선 모델과 단순성 실무 대안이 같은 모델이다."
elif wape_gap >= SIMPLICITY_TOL or cost_gain >= COST_TOL:
    FINAL_MODEL, FINAL_REASON = OPT1, f"실무 대안 대비 WAPE {wape_gap:+.2f}%p 우위, 총비용 {cost_gain:+.1%} 우위로 허용 폭(WAPE {SIMPLICITY_TOL}%p 또는 비용 {COST_TOL:.0%}) 이상이어서 정확도·비용 최우선 모델을 권고한다."
else:
    FINAL_MODEL, FINAL_REASON = OPT2, f"두 모델 차이가 WAPE {wape_gap:+.2f}%p, 총비용 {cost_gain:+.1%}로 허용 폭 안이어서 더 단순한 실무 대안을 권고한다."
SELECTION_TRACE = pd.DataFrame(TRACE)
OPTION_TABLE = SEL_T.loc[sorted({OPT1, OPT2, "baseline_mean7"}, key=lambda m: MODELS.index(m)), ["model_label", "WAPE", "RelMAE", "hit_rate", "coverage", "mean_interval_width", "total_cost_krw", "shortage_person_days", "surplus_person_days", "special_shortage_person_days", "special_hit_rate"]].copy()
OPTION_TABLE.insert(0, "role", [("정확도·비용 최우선(Option 1)" if m == OPT1 else "") + (" / " if m == OPT1 and m == OPT2 else "") + ("단순성·운영 편의 실무 대안(Option 2)" if m == OPT2 else "") + ("운영 기준선(직전 7일 평균)" if m == "baseline_mean7" and m not in (OPT1, OPT2) else "") for m in OPTION_TABLE.index])
OPTION_TABLE.insert(1, "is_final_recommendation", OPTION_TABLE.index == FINAL_MODEL)
print("Option 1 (정확도·비용 최우선):", REG[OPT1]["label"], "| Option 2 (단순성 실무 대안):", REG[OPT2]["label"])
print("최종 권고:", REG[FINAL_MODEL]["label"], "-", FINAL_REASON)
display(OPTION_TABLE.round(3))
display(SELECTION_TRACE)
SEL_ORDER_T = SEL_T.sort_values(STRICT_ORDER).reset_index()
mark("최종 모델 선정")

## 9. 직전 7일 평균 대비 비용과 FVA
운영 기준선은 직전 7일 평균이며, 최종 비교의 **주 기준은 모든 모델에 같은 q80 인력 정책**입니다(point 기준은 참고 행으로 따로 표시). 절감액 = 직전 7일 평균 총비용 − 모델 총비용, 절감률 = 절감액 / 직전 7일 평균 총비용. 기존 결과와의 검산: 직전 7일 평균 q80 총비용 ≈ 1,210,650,000원, 기존 최종 후보 ≈ 767,550,000원, 절감액 ≈ 443,100,000원.

In [ ]:
def fva_rows(variant):
    table = staff_by([], variant).merge(OVERALL[["model_id", "model_label", "n_observations"] + METRICS], on="model_id", how="left")
    base = table.loc[table["model_id"] == "baseline_mean7"].iloc[0]
    table["staffing_basis"] = "q80 기준 (주 기준)" if variant == "q80" else "point 기준 (참고, q80과 혼용 금지)"
    table["saving_vs_mean7_krw"] = base["total_cost_krw"] - table["total_cost_krw"]
    table["saving_rate_vs_mean7"] = table["saving_vs_mean7_krw"] / base["total_cost_krw"]
    table["wape_improvement_pp_vs_mean7"] = OVERALL.set_index("model_id").loc["baseline_mean7", "WAPE"] - table["model_id"].map(OVERALL.set_index("model_id")["WAPE"])
    table["relmae_improvement_vs_mean7"] = OVERALL.set_index("model_id").loc["baseline_mean7", "RelMAE"] - table["model_id"].map(OVERALL.set_index("model_id")["RelMAE"])
    return table


FVA_T = pd.concat([fva_rows("q80"), fva_rows("point")], ignore_index=True)
FVA_T["model_order"] = FVA_T["model_id"].map({m: i for i, m in enumerate(MODELS)})
FVA_T = FVA_T.sort_values(["staffing_basis", "model_order"], ascending=[False, True]).reset_index(drop=True)
FVA_Q80 = FVA_T.loc[FVA_T["staffing_basis"].str.startswith("q80")].set_index("model_id")
base_cost, final_cost = FVA_Q80.loc["baseline_mean7", "total_cost_krw"], FVA_Q80.loc[FINAL_MODEL, "total_cost_krw"]
print(f"직전 7일 평균 q80 총비용 {base_cost:,.0f}원 ({base_cost / 1e8:.4f}억) | 최종 권고 모델 {final_cost:,.0f}원 ({final_cost / 1e8:.4f}억) | 절감 {base_cost - final_cost:,.0f}원 ({(base_cost - final_cost) / base_cost:.1%})")
REF_MEAN7, REF_PREV_FINAL = 1_210_650_000, 767_550_000
print(f"기존 결과 대비: 직전 7일 평균 {base_cost - REF_MEAN7:+,.0f}원 (기존 {REF_MEAN7:,}원), 기존 최종 후보 대비 {final_cost - REF_PREV_FINAL:+,.0f}원 (기존 {REF_PREV_FINAL:,}원)")
display(FVA_Q80[["model_label", "WAPE", "RelMAE", "hit_rate", "coverage", "mean_interval_width", "shortage_person_days", "surplus_person_days", "total_cost_krw", "saving_vs_mean7_krw", "saving_rate_vs_mean7"]].round(3))
mark("FVA·비용 비교")

## 10. 2025년 인력 배치 오류 탐지 (최종 권고 모델)
**실제 근무표가 없으므로 "실제 인력"이라는 표현을 쓰지 않습니다.** `required_staff_actual` = 실제 방문객 환산 필요 인원, `staff_q80` = 모델 권고 인원(q80), `staff_point` = 점 예측 기반 환산 인원입니다. 인력 부족 10건, 비용 최대 10건, 방문객 과소예측 최대 10건, 잉여 최대 10건과 황금연휴·명절 집중도를 만듭니다.

In [ ]:
FINAL_ROWS = EVAL_STAFF.loc[EVAL_STAFF["model_id"] == FINAL_MODEL].copy()
FINAL_ROWS["under_forecast_visitors"] = FINAL_ROWS["actual_visitors"] - FINAL_ROWS["predicted_visitors"]
SITE_CV = {s: float(MATRICES["total"][s].loc[TEST_START:TEST_END].std() / MATRICES["total"][s].loc[TEST_START:TEST_END].mean()) for s in SITES}
HIGH_VAR_SITES = {s for s, v in SITE_CV.items() if v >= np.quantile(list(SITE_CV.values()), 0.75)}
COMMENT_LOOKUP = VISITORS_ALL.set_index(["site", "date"])["Comment"]


def probable_cause(row, side):
    """규칙 기반 추정 원인 (단정이 아닌 후보 분류, 사후 진단용). side = 'shortage' 또는 'surplus'."""
    comment = COMMENT_LOOKUP.get((row["site"], row["date"]), np.nan)
    if row["predicted_visitors"] == 0 or (isinstance(comment, str) and "중앙값" in comment):
        return "휴관·결측 처리 문제"
    history = MATRICES["total"][row["site"]].loc[:row["date"] - pd.Timedelta(days=1)].dropna().tail(28)
    ratio = row["actual_visitors"] / history.mean() if len(history) and history.mean() > 0 else np.nan
    if side == "shortage":
        if row["date_type"] == "설·추석 명절":
            return "설·추석 명절 수요 급증"
        if row["date_type"] == "황금연휴":
            return "황금연휴 수요 급증"
        if row["date_type"] == "일반 공휴일":
            return "공휴일 수요 급증"
        if row["date_type"] == "근로자의 날":
            return "근로자의 날 특수 수요"
        if row["date_type"] == "무료개방일":
            return "무료개방 수요 급증"
        if row["date_type"] == "일반 주말":
            return "주말 수요 급증"
        if np.isfinite(ratio) and ratio >= 1.5:
            return "직전 기간 대비 비정상 급증"
        return "개소별 변동성" if row["site"] in HIGH_VAR_SITES else "기타 또는 확인 필요"
    if row["rain_type_actual"] != "비 안 옴":
        return "강수로 인한 방문 감소 가능(사후 실측)"
    if np.isfinite(ratio) and ratio <= 0.67:
        return "직전 기간 대비 비정상 감소"
    return "개소별 변동성" if row["site"] in HIGH_VAR_SITES else "기타 또는 확인 필요"


CASE_COLUMNS = ["date", "weekday_ko", "site", "group", "date_type", "holiday_name", "is_labor_day", "free_open", "rain_type_actual", "actual_visitors", "predicted_visitors", "prediction_q80", "required_staff_actual",
                "staff_point", "staff_q80", "shortage_q80", "surplus_q80", "total_cost_q80", "probable_cause"]
CASE_RENAME = {"weekday_ko": "weekday", "free_open": "is_free_open", "rain_type_actual": "rain_type_observed", "predicted_visitors": "predicted_point", "prediction_q80": "predicted_q80", "staff_q80": "recommended_staff_q80",
               "shortage_q80": "staff_shortage", "surplus_q80": "staff_surplus", "total_cost_q80": "cost_krw", "required_staff_actual": "required_staff_from_actual_visitors"}


def top_cases(sort_column, side, label, n=10):
    top = FINAL_ROWS.loc[FINAL_ROWS[sort_column] > 0].sort_values([sort_column, "actual_visitors"], ascending=[False, False]).head(n).copy()
    top["probable_cause"] = [probable_cause(r, side) for _, r in top.iterrows()]
    top["date"] = top["date"].dt.strftime("%Y-%m-%d")
    top["holiday_name"] = top["holiday_name"].fillna("")
    top["cost_eok_won"] = top["total_cost_q80"] / 1e8
    out = top[CASE_COLUMNS + ["cost_eok_won"]].rename(columns=CASE_RENAME)
    out.insert(0, "rank", np.arange(1, len(out) + 1))
    out.insert(0, "case_type", label)
    return out.reset_index(drop=True)


TOP_SHORTAGE = top_cases("shortage_q80", "shortage", "인력 부족 상위 10건")
TOP_COST = top_cases("total_cost_q80", "shortage", "비용 상위 10건")
TOP_COST["probable_cause"] = [probable_cause(FINAL_ROWS.loc[(FINAL_ROWS["site"] == r["site"]) & (FINAL_ROWS["date"] == pd.Timestamp(r["date"])) ].iloc[0], "shortage" if r["staff_shortage"] > 0 else "surplus") for _, r in TOP_COST.iterrows()]
TOP_UNDER = top_cases("under_forecast_visitors", "shortage", "방문객 과소예측 상위 10건")
TOP_SURPLUS = top_cases("surplus_q80", "surplus", "잉여 인력 상위 10건")
TOP_OTHER = pd.concat([TOP_UNDER, TOP_SURPLUS], ignore_index=True)
total_shortage = FINAL_ROWS["shortage_q80"].sum()
golden_mask = FINAL_ROWS["is_golden_holiday"] | FINAL_ROWS["is_myeongjeol"]
GOLDEN_SHORTAGE_SHARE = float(FINAL_ROWS.loc[golden_mask, "shortage_q80"].sum() / total_shortage)
DATE_TYPE_SHORTAGE = FINAL_ROWS.groupby("date_type").agg(site_days=("shortage_q80", "size"), shortage_person_days=("shortage_q80", "sum"), surplus_person_days=("surplus_q80", "sum")).reindex(DATE_TYPES).fillna(0)
DATE_TYPE_SHORTAGE["shortage_share"] = DATE_TYPE_SHORTAGE["shortage_person_days"] / total_shortage
DATE_TYPE_SHORTAGE["site_day_share"] = DATE_TYPE_SHORTAGE["site_days"] / DATE_TYPE_SHORTAGE["site_days"].sum()
print(f"최종 권고 모델 2025: 필요 인원 합 {int(FINAL_ROWS['required_staff_actual'].sum()):,}인일, 권고(q80) {int(FINAL_ROWS['staff_q80'].sum()):,}인일, 부족 {int(total_shortage):,}인일, 잉여 {int(FINAL_ROWS['surplus_q80'].sum()):,}인일")
print(f"황금연휴·설추석 명절(전체 {int(golden_mask.sum()):,}행, 개방 행의 {golden_mask.mean():.1%})이 부족 인원의 {GOLDEN_SHORTAGE_SHARE:.1%}를 차지")
display(DATE_TYPE_SHORTAGE.round(3))
display(TOP_SHORTAGE[["rank", "date", "weekday", "site", "date_type", "actual_visitors", "predicted_q80", "required_staff_from_actual_visitors", "recommended_staff_q80", "staff_shortage", "cost_krw", "probable_cause"]])
mark("오류 탐지")

## 11. 황금연휴 운영 시나리오
모델에 검증되지 않은 임의 가중치를 넣지 않고, **2025년 순차 백테스트 예측(최종 권고 모델)** 위에서 황금연휴(공휴일을 포함해 쉬는 날이 4일 이상 이어지는 구간)와 설·추석 명절의 개방 행에 대해 운영 시나리오를 비교합니다: 기본 q80, q80 + 1명, q80 + 2명, 황금연휴에만 q90. 서비스 수준 = 부족이 없는 개소·일의 비율, q90 시나리오의 적중률 = 실제 ≤ q90 상당 값인 비율입니다. q90은 같은 conformal 절차(2024-07-03~12-31 보정 오차)로 만듭니다. 2023~2024년 황금연휴는 보정 구간이 일부 겹쳐 표본 내(in-sample) 성격이라 평가에 넣지 않고 2025년 표본(황금연휴·명절 행)만 비교합니다. 운영 가능성은 "추가 배치 인원이 기본 q80 인원의 30%를 넘지 않는다"를 가정한 상한으로 반영합니다(가정, 실제 인력 제약 자료 없음).

In [ ]:
def signed_quantile(level):
    rows = [(m, t, s, conformal_value(part["residual"], level)) for (m, t, s), part in CAL_TEST.groupby(["model_id", "target", "site"])]
    return pd.DataFrame(rows, columns=["model_id", "target", "site", "s_up_level"])


S_UP90 = signed_quantile(0.9)
golden_rows = FINAL_ROWS.loc[FINAL_ROWS["is_golden_holiday"] | FINAL_ROWS["is_myeongjeol"]].drop(columns=["s_up"]).merge(S_UP90, on=["model_id", "target", "site"], how="left", validate="many_to_one")
assert golden_rows["s_up_level"].notna().all()
golden_rows["q90_visitors"] = to_visitors_int(np.clip(golden_rows["prediction"] + golden_rows["s_up_level"], 0, None))
golden_rows["staff_q90"] = staff_rule(golden_rows["q90_visitors"], np.zeros(len(golden_rows), dtype=bool))
FEASIBLE_EXTRA_RATIO = 0.30
scenario_rows = []
base_person_days = int(golden_rows["staff_q80"].sum())
palace_mask = golden_rows["group"].isin(["궁", "종묘"]).to_numpy()
for apply_to, mask in (("전 개소", np.ones(len(golden_rows), dtype=bool)), ("궁·종묘만", palace_mask)):
    for name, add, use_q90 in (("기본 q80", 0, False), ("황금연휴 q80 + 1명", 1, False), ("황금연휴 q80 + 2명", 2, False), ("황금연휴 q90", 0, True)):
        if name == "기본 q80" and apply_to != "전 개소":
            continue
        base = golden_rows["staff_q90"] if use_q90 else golden_rows["staff_q80"]
        staff = np.where(mask, base + add, golden_rows["staff_q80"]).astype(int)
        gap = staff - golden_rows["required_staff_actual"].to_numpy()
        shortage, surplus = np.clip(-gap, 0, None), np.clip(gap, 0, None)
        scenario_rows.append(dict(scenario=name, apply_to="전 개소" if name == "기본 q80" else apply_to, n_site_days=len(golden_rows), service_level_no_shortage=float((shortage == 0).mean()), q90_hit_rate=float((golden_rows["actual"] <= golden_rows["prediction"] + golden_rows["s_up_level"]).mean()) if use_q90 else np.nan,
                                  q80_hit_rate=float((golden_rows["actual"] <= golden_rows["q80_equiv"]).mean()), shortage_person_days=int(shortage.sum()), surplus_person_days=int(surplus.sum()), shortage_cost_krw=int(shortage.sum()) * SHORTAGE_COST_KRW,
                                  surplus_cost_krw=int(surplus.sum()) * OVERSTAFF_COST_KRW, total_cost_krw=int(shortage.sum()) * SHORTAGE_COST_KRW + int(surplus.sum()) * OVERSTAFF_COST_KRW, recommended_person_days=int(staff.sum())))
GOLDEN_T = pd.DataFrame(scenario_rows)
base_row = GOLDEN_T.iloc[0]
GOLDEN_T["extra_person_days_vs_base"] = GOLDEN_T["recommended_person_days"] - int(base_row["recommended_person_days"])
GOLDEN_T["extra_ratio_vs_base"] = GOLDEN_T["extra_person_days_vs_base"] / int(base_row["recommended_person_days"])
GOLDEN_T["service_level_change_pp"] = (GOLDEN_T["service_level_no_shortage"] - base_row["service_level_no_shortage"]) * 100
GOLDEN_T["cost_change_krw"] = GOLDEN_T["total_cost_krw"] - base_row["total_cost_krw"]
GOLDEN_T["shortage_cost_change_krw"] = GOLDEN_T["shortage_cost_krw"] - base_row["shortage_cost_krw"]
GOLDEN_T["operationally_feasible"] = GOLDEN_T["extra_ratio_vs_base"] <= FEASIBLE_EXTRA_RATIO
GOLDEN_T["model_id"] = FINAL_MODEL
feasible = GOLDEN_T.loc[GOLDEN_T["operationally_feasible"] & (GOLDEN_T["scenario"] != "기본 q80")]
GOLDEN_RECOMMEND = feasible.sort_values(["total_cost_krw", "service_level_no_shortage"], ascending=[True, False]).iloc[0] if len(feasible) else base_row
GOLDEN_T["is_recommended"] = (GOLDEN_T["scenario"] == GOLDEN_RECOMMEND["scenario"]) & (GOLDEN_T["apply_to"] == GOLDEN_RECOMMEND["apply_to"])
GOLDEN_ADD = dict(scenario=GOLDEN_RECOMMEND["scenario"], apply_to=GOLDEN_RECOMMEND["apply_to"])
print(f"황금연휴·명절 개방 행 {len(golden_rows)}개 (기본 q80 {base_person_days}인일). 운영 가능성 상한 {FEASIBLE_EXTRA_RATIO:.0%} 안에서 총비용이 가장 낮은 시나리오: {GOLDEN_ADD}")
display(GOLDEN_T[["scenario", "apply_to", "service_level_no_shortage", "shortage_person_days", "surplus_person_days", "total_cost_krw", "extra_person_days_vs_base", "extra_ratio_vs_base", "service_level_change_pp", "operationally_feasible", "is_recommended"]].round(3))
mark("황금연휴 시나리오")

## 12. 근로자의 날 (장기 근거와 ablation)
- **유지 근거(장기 자료)**: `heritage_visitors_daily.csv` 2015~2025년에서 5월 1일 방문객을 평상시 일평균과 비교합니다. 민감도: ① 전체 2015~2025 ② 코로나 영향 2020~2022년 제외 ③ 같은 연도 5월 같은 요일 평균과 비교(봄철 계절성·요일 효과 완화) ④ 개소 수가 균형인 2019~2025년만. 장기 자료에는 봄철 계절성·요일·코로나 기간이 섞여 있어 인과가 아니라 반복 관찰된 특수 수요 신호입니다. 근로자의 날은 "법정공휴일"이 아니라 "반복적으로 관찰된 특수 수요일" 피처로 설명합니다.
- **ablation**: `+ 공식 공휴일 + 무료개방`(근로자의 날 제외, 모델 A)과 `+ 휴일 묶음(공식 공휴일 + 근로자의 날) + 무료개방`(모델 B)을 같은 조건으로 비교합니다. 기존 `outputs/labor_day_ablation/` 결과와 일치하는지 재현 여부를 확인합니다. 2025년 백테스트에서 독립적인 근로자의 날 표본은 2025-05-01 하루뿐이어서 그 하루의 오차만으로 장기 효과를 단정하지 않습니다.

In [ ]:
long = pd.read_csv("dataset/heritage_visitors_daily.csv", encoding="utf-8-sig", parse_dates=["date"])
assert {"date", "site", "group", "total"} <= set(long.columns) and long["date"].min() == pd.Timestamp("2015-01-01") and long["date"].max() == pd.Timestamp("2025-12-31")
long = long.dropna(subset=["total"]).copy()
long["md"], long["year"] = long["date"].dt.strftime("%m-%d"), long["date"].dt.year
may1 = long.loc[long["md"] == "05-01"]
site_mean_all = long.groupby("site")["total"].mean()
site_group_map = long.drop_duplicates("site").set_index("site")["group"]
per_site = may1.groupby("site")["total"].agg(["mean", "size"]).rename(columns={"mean": "may1_visitors", "size": "n_years_observed"})
per_site["baseline_visitors"] = site_mean_all.reindex(per_site.index)
per_site["ratio"] = per_site["may1_visitors"] / per_site["baseline_visitors"]
per_site["higher_than_baseline"] = per_site["ratio"] > 1
daily_total = long.groupby("date")["total"].sum()
d_md, d_year = daily_total.index.strftime("%m-%d"), daily_total.index.year
may1_total = daily_total[d_md == "05-01"]
rows = []
for site, r in per_site.reset_index().set_index("site").iterrows():
    rows.append(dict(analysis="개소별(2015~2025 평균)", year="2015~2025", site=site, group=site_group_map.get(site, ""), n_days_or_years=int(r["n_years_observed"]), may1_visitors=r["may1_visitors"], baseline_visitors=r["baseline_visitors"], ratio=r["ratio"], higher_than_baseline=bool(r["higher_than_baseline"]), note="기준 = 해당 개소의 전체 일평균(2015~2025)"))
rows.append(dict(analysis="전체 개소 합계(주 분석)", year="2015~2025", site="전체", group="전체", n_days_or_years=int(len(may1_total)), may1_visitors=may1_total.mean(), baseline_visitors=daily_total.mean(), ratio=may1_total.mean() / daily_total.mean(),
                 higher_than_baseline=bool(may1_total.mean() > daily_total.mean()), note="일별 전체 개소 합계의 평균 대비 5월 1일 합계의 평균. 연도별 보고 개소 수가 다름(2015~2018 4곳, 2019~ 22~23곳)"))
for year in sorted(may1_total.index.year.unique()):
    rows.append(dict(analysis="전체 개소 합계 연도별", year=str(year), site="전체", group="전체", n_days_or_years=int((d_year == year).sum()), may1_visitors=float(may1_total[may1_total.index.year == year].iloc[0]), baseline_visitors=float(daily_total[d_year == year].mean()),
                     ratio=float(may1_total[may1_total.index.year == year].iloc[0] / daily_total[d_year == year].mean()), higher_than_baseline=bool(may1_total[may1_total.index.year == year].iloc[0] > daily_total[d_year == year].mean()), note="기준 = 같은 연도 일평균. 2025년은 1 미만"))
keep = ~np.isin(d_year, [2020, 2021, 2022])
rows.append(dict(analysis="민감도: 코로나 2020~2022 제외", year="2015~2019, 2023~2025", site="전체", group="전체", n_days_or_years=int((~np.isin(may1_total.index.year, [2020, 2021, 2022])).sum()), may1_visitors=may1_total[~np.isin(may1_total.index.year, [2020, 2021, 2022])].mean(),
                 baseline_visitors=daily_total[keep].mean(), ratio=may1_total[~np.isin(may1_total.index.year, [2020, 2021, 2022])].mean() / daily_total[keep].mean(), higher_than_baseline=bool(may1_total[~np.isin(may1_total.index.year, [2020, 2021, 2022])].mean() > daily_total[keep].mean()), note="코로나 영향 기간 제외"))
same_weekday = []
for date, value in may1_total.items():
    window = daily_total[(daily_total.index.year == date.year) & (daily_total.index.month == 5) & (daily_total.index.dayofweek == date.dayofweek) & (daily_total.index != date)]
    same_weekday.append((date.year, value, window.mean(), value / window.mean(), len(window)))
for year, value, base, ratio, n in same_weekday:
    rows.append(dict(analysis="민감도: 같은 연도 5월 같은 요일 평균", year=str(year), site="전체", group="전체", n_days_or_years=int(n), may1_visitors=float(value), baseline_visitors=float(base), ratio=float(ratio), higher_than_baseline=bool(ratio > 1), note="기준 = 같은 해 5월의 5월 1일과 같은 요일 평균(5월 1일 제외)"))
rows.append(dict(analysis="민감도: 같은 연도 5월 같은 요일 평균(요약)", year="2015~2025", site="전체", group="전체", n_days_or_years=len(same_weekday), may1_visitors=float(np.mean([s[1] for s in same_weekday])), baseline_visitors=float(np.mean([s[2] for s in same_weekday])),
                 ratio=float(np.mean([s[3] for s in same_weekday])), higher_than_baseline=bool(np.mean([s[3] for s in same_weekday]) > 1), note=f"연도별 비율 평균. 5월 1일이 같은 요일 평균보다 높은 연도 {sum(s[3] > 1 for s in same_weekday)}/{len(same_weekday)}"))
bal = np.isin(may1_total.index.year, range(2019, 2026))
rows.append(dict(analysis="민감도: 개소 수 균형 2019~2025", year="2019~2025", site="전체", group="전체", n_days_or_years=int(bal.sum()), may1_visitors=may1_total[bal].mean(), baseline_visitors=daily_total[d_year >= 2019].mean(), ratio=may1_total[bal].mean() / daily_total[d_year >= 2019].mean(),
                 higher_than_baseline=bool(may1_total[bal].mean() > daily_total[d_year >= 2019].mean()), note="2019년 이후(22~23개소)만"))
LABOR_EVIDENCE = pd.DataFrame(rows)
N_SITES_HIGHER, N_SITES_LONG = int(per_site["higher_than_baseline"].sum()), int(len(per_site))
RATIO_ALL = float(LABOR_EVIDENCE.loc[LABOR_EVIDENCE["analysis"] == "전체 개소 합계(주 분석)", "ratio"].iloc[0])
RATIO_EX_COVID = float(LABOR_EVIDENCE.loc[LABOR_EVIDENCE["analysis"] == "민감도: 코로나 2020~2022 제외", "ratio"].iloc[0])
RATIO_SAME_WEEKDAY = float(LABOR_EVIDENCE.loc[LABOR_EVIDENCE["analysis"] == "민감도: 같은 연도 5월 같은 요일 평균(요약)", "ratio"].iloc[0])
RATIO_BALANCED = float(LABOR_EVIDENCE.loc[LABOR_EVIDENCE["analysis"] == "민감도: 개소 수 균형 2019~2025", "ratio"].iloc[0])
YEAR_RATIOS = LABOR_EVIDENCE.loc[LABOR_EVIDENCE["analysis"] == "전체 개소 합계 연도별"].set_index("year")["ratio"]
print(f"5월 1일 > 개소 전체 일평균인 개소 {N_SITES_HIGHER}/{N_SITES_LONG} | 전체 합계 비율 {RATIO_ALL:.3f}배 | 코로나 제외 {RATIO_EX_COVID:.3f}배 | 같은 요일 5월 평균 대비 {RATIO_SAME_WEEKDAY:.3f}배 | 2019~2025 {RATIO_BALANCED:.3f}배")
print("연도별 비율:", YEAR_RATIOS.round(2).to_dict())

# ---------- ablation: A(공식 공휴일 + 무료개방) vs B(휴일 묶음 = 공식 공휴일 + 근로자의 날, + 무료개방) ----------
B_ID = "chronos_holiday_free"
assert REG[B_ID]["covariates"] == ["is_holiday", "is_labor_day", "free_open"] and REG[ABL_ID]["covariates"] == ["is_holiday", "free_open"]
ABL_MODELS = [ABL_ID, B_ID]
ABL_LABEL = {ABL_ID: "A: Chronos-2 제로샷 + 공식 공휴일 + 무료개방", B_ID: "B: Chronos-2 제로샷 + 공식 공휴일 + 근로자의 날 + 무료개방"}
abl_rows = []
for scope, mask_fn in (("전체 2025", lambda f: np.ones(len(f), dtype=bool)), ("궁", lambda f: (f["group"] == "궁").to_numpy()), ("종묘", lambda f: (f["group"] == "종묘").to_numpy()), ("왕릉", lambda f: (f["group"] == "왕릉").to_numpy()),
                       ("2025-05-01 하루(23개소)", lambda f: (f["date"] == pd.Timestamp("2025-05-01")).to_numpy())):
    for model_id in ABL_MODELS:
        base = EVAL_BASE.loc[(EVAL_BASE["model_id"] == model_id) & (EVAL_BASE["target"] == "total")]
        base = base.loc[mask_fn(base)]
        stf = EVAL_STAFF.loc[EVAL_STAFF["model_id"] == model_id]
        stf = stf.loc[mask_fn(stf)]
        m = calculate_metrics(base.assign(scope=scope), ["model_id", "scope"]).iloc[0]
        abl_rows.append(dict(scope=scope, model_id=model_id, model_label=ABL_LABEL[model_id], n_observations=int(m["n_observations"]), **{k: float(m[k]) for k in METRICS}, observed_visitors=float(stf["actual_visitors"].sum()), predicted_visitors_sum=float(stf["predicted_visitors"].sum()),
                             q80_visitors_sum=float(stf["prediction_q80"].sum()), required_staff_actual=float(stf["required_staff_actual"].sum()), recommended_staff=float(stf["staff_q80"].sum()), shortage_person_days=float(stf["shortage_q80"].sum()), surplus_person_days=float(stf["surplus_q80"].sum()),
                             shortage_cost_krw=float(stf["shortage_cost_q80"].sum()), surplus_cost_krw=float(stf["surplus_cost_q80"].sum()), total_cost_krw=float(stf["total_cost_q80"].sum())))
ABLATION = pd.DataFrame(abl_rows)
value_cols = ["n_observations"] + METRICS + ["observed_visitors", "predicted_visitors_sum", "q80_visitors_sum", "required_staff_actual", "recommended_staff", "shortage_person_days", "surplus_person_days", "shortage_cost_krw", "surplus_cost_krw", "total_cost_krw"]
diff_rows = []
for scope, part in ABLATION.groupby("scope", sort=False):
    a, b = part.set_index("model_id").loc[ABL_ID], part.set_index("model_id").loc[B_ID]
    diff_rows.append(dict(scope=scope, model_id="diff_B_minus_A", model_label="차이 (B − A)", **{c: float(b[c] - a[c]) for c in value_cols}))
ABLATION = pd.concat([ABLATION, pd.DataFrame(diff_rows)], ignore_index=True)
ABLATION["_s"] = ABLATION["scope"].map({"전체 2025": 0, "궁": 1, "종묘": 2, "왕릉": 3, "2025-05-01 하루(23개소)": 4})
ABLATION["_o"] = ABLATION["model_id"].map({ABL_ID: 0, B_ID: 1, "diff_B_minus_A": 2})
ABLATION = ABLATION.sort_values(["_s", "_o"]).drop(columns=["_s", "_o"]).reset_index(drop=True)
PREV_ABL = Path("outputs") / "labor_day_ablation" / "tables" / "labor_feature_model_comparison_overall.csv"
ABL_REPRO = {}
if PREV_ABL.exists():
    prev = pd.read_csv(PREV_ABL, encoding="utf-8-sig")
    for model_id, key in ((ABL_ID, "chronos_holiday_only_free"), (B_ID, "chronos_holiday_labor_free")):
        row = prev.loc[prev["model_id"] == key].iloc[0]
        new = ABLATION.loc[(ABLATION["scope"] == "전체 2025") & (ABLATION["model_id"] == model_id)].iloc[0]
        ABL_REPRO[model_id] = dict(previous_wape=float(row["WAPE"]), new_wape=float(new["WAPE"]), previous_cost=float(row["total_cost_krw"]), new_cost=float(new["total_cost_krw"]))
print("ablation 재현 확인(기존 outputs/labor_day_ablation 대비):", {k: {a: round(b, 3) for a, b in v.items()} for k, v in ABL_REPRO.items()})
display(ABLATION.loc[ABLATION["scope"].isin(["전체 2025", "2025-05-01 하루(23개소)"]), ["scope", "model_id"] + METRICS + ["shortage_person_days", "surplus_person_days", "total_cost_krw"]].round(3))
mark("근로자의 날 근거·ablation")

## 13. 2026-01-01~14 최종 예측
선정된 최종 권고 모델을 **2023-01-01~2025-12-31 전체 이력**으로 다시 적용합니다(ARIMA·SARIMA는 2025-12-31까지의 이력으로 차수 재선택·재적합, MLP는 2025-12-31까지 재학습, Chronos-2는 전체를 컨텍스트로 사용). 강수 확률은 2023~2025년 자료로 만든 월×요일(→월→계절→전체) 발생확률이며 2026년 실제 기상 관측값은 쓰지 않습니다. q80 구간 보정은 2023~2024년 내부 백테스트로만 하고 2025년 테스트 오차는 보정에 쓰지 않습니다. 총방문객·내국인·외국인을 각각 예측하고 합계 정합성을 검산합니다. 정기 휴관 요일(공휴일 제외)은 0입니다.

In [ ]:
def final_forecast(model_id):
    """모델을 2023~2025년 전체(학습 종료 2025-12-31)로 다시 학습·적용해 2026-01-01~14를 예측한다 (domestic, foreign_total, total)."""
    kind = REG[model_id]["kind"]
    if kind == "baseline":
        frame = BASELINE_PRED.loc[(BASELINE_PRED["phase"] == "final") & (BASELINE_PRED["model_id"] == model_id)]
    elif kind == "mlp":
        frame = MLP_PRED.loc[MLP_PRED["phase"] == "final"]
    elif kind == "stat":
        orders = select_orders(model_id, FINAL_TRAIN_END)
        FINAL_ORDERS[model_id] = orders
        frame, log = forecast_stat(model_id, FINAL_WINDOW, orders, BASELINE_PRED)
    else:
        frame = chronos_frame(model_id, FINAL_WINDOW, "final")
    frame = frame.loc[frame["model_id"] == model_id]
    assert (frame["phase"] == "final").all() and frame["date"].min() == FORECAST_START and frame["date"].max() == FORECAST_END
    return add_total(frame)


FINAL_ORDERS = {}
print(f"2026 예측: {REG[FINAL_MODEL]['label']} ({FINAL_MODEL}), 공변량 {REG[FINAL_MODEL]['covariates'] or '없음'}", flush=True)
FINAL_RAW = final_forecast(FINAL_MODEL)
FINAL_RAW["date"] = pd.to_datetime(FINAL_RAW["date"])
FINAL_FULL = apply_intervals(FINAL_RAW, PARAMS_TEST.loc[PARAMS_TEST["model_id"] == FINAL_MODEL])      # q80 보정: 2023~2024 내부 백테스트(PARAMS_TEST)만
assert CAL_TEST["date"].max() <= TRAIN_END
HOLIDAY_SET = set(RAW_CAL.index[RAW_CAL["is_holiday"].astype(int) == 1])
FINAL_FULL["group"] = FINAL_FULL["site"].map(SITE_GROUP)
FINAL_FULL["is_regular_closed"] = [(pd.Timestamp(d).dayofweek == SITE_CLOSED_WEEKDAY[s]) and (pd.Timestamp(d) not in HOLIDAY_SET) for d, s in zip(FINAL_FULL["date"], FINAL_FULL["site"])]
for column in ["prediction", "lower_80", "upper_80", "q80_equiv"]:
    FINAL_FULL.loc[FINAL_FULL["is_regular_closed"], column] = 0.0
assert (FINAL_FULL[["prediction", "q80_equiv", "lower_80", "upper_80"]] >= 0).all().all(), "예측에 음수가 남아 있습니다."
neg_final = neg_frame(FINAL_MODEL)
N_NEG_2026 = int(neg_final.loc[neg_final["phase"] == "final"].drop_duplicates(["date", "site"]).shape[0]) if len(neg_final) else 0
wide = FINAL_FULL.pivot_table(index=["date", "site", "group", "is_regular_closed"], columns="target", values=["prediction", "q80_equiv", "lower_80", "upper_80"]).reset_index()
wide.columns = ["_".join([c for c in col if c]) for col in wide.columns]
neg_keys = set(zip(pd.to_datetime(neg_final["date"]), neg_final["site"])) if len(neg_final) else set()
F13 = pd.DataFrame({"date": wide["date"].dt.strftime("%Y-%m-%d"), "weekday": wide["date"].dt.dayofweek.map({0: "월", 1: "화", 2: "수", 3: "목", 4: "금", 5: "토", 6: "일"}), "group": wide["group"], "site": wide["site"], "model_id": FINAL_MODEL,
                    "is_regular_closed": wide["is_regular_closed"], "predicted_total_point": wide["prediction_total"].round(1), "predicted_total_lower80": wide["lower_80_total"].round(1), "predicted_total_q80": wide["q80_equiv_total"].round(1),
                    "predicted_total_upper80": wide["upper_80_total"].round(1), "predicted_domestic": wide["prediction_domestic"].round(1), "predicted_foreign": wide["prediction_foreign_total"].round(1),
                    "predicted_domestic_q80": wide["q80_equiv_domestic"].round(1), "predicted_foreign_q80": wide["q80_equiv_foreign_total"].round(1)})
F13["negative_corrected_before"] = [(pd.Timestamp(d), s) in neg_keys for d, s in zip(F13["date"], F13["site"])]
F13["_g"] = F13["group"].map({"궁": 0, "종묘": 1, "왕릉": 2})
F13 = F13.sort_values(["date", "_g", "site"]).drop(columns="_g").reset_index(drop=True)
assert np.allclose(F13["predicted_total_point"], F13["predicted_domestic"] + F13["predicted_foreign"], atol=0.25), "total과 domestic+foreign 합이 어긋납니다."
print(f"2026 예측 행 {len(F13)}개(개소 {F13['site'].nunique()} × 날짜 {F13['date'].nunique()}), 총 예상 방문객(점 예측) {F13['predicted_total_point'].sum():,.0f}명, q80 합계 {F13['predicted_total_q80'].sum():,.0f}명, 보정 전 음수 개소·일 {N_NEG_2026}건")
mark("2026 예측")

## 14. 2026-01-01~14 운영팀 전달용 인력 배치표
- `required_staff_point` = ceil(점 예측 / 2000), `recommended_staff_q80` = max(1, ceil(q80 / 2000)) (휴관일 0명), `final_operational_staff` = 궁·종묘·왕릉 운영 규칙 + 황금연휴 권고 반영 인원. 왕릉은 개방일 최소 1명이며 q80이 2,000명을 넘으면 2명 이상입니다. 황금연휴 추가 배치 권고(섹션 11)는 이번 14일에 황금연휴가 없어 적용되지 않았습니다.
- 내국인·외국인 담당 인력을 별도로 강제 배정하지 않고, 예측 비중에 따른 **참고 열**(domestic_share, foreign_share, recommended_domestic/foreign_service_share와 인원)을 제공합니다. 총 인원이 1명이면 "통합 응대"로 표시하고, 2명 이상이면 큰 비중 순 반올림(최대잔여법)으로 나누어 합계가 총 권고 인원과 일치합니다.
- **권고 인원은 실제 근무표 인원이 아니라 예측 방문객(q80)과 운영 가정(2,000명당 1명)으로 산출한 필요 인원입니다.**

In [ ]:
def split_staff(total_staff, domestic_share):
    """내국인·외국인 응대 참고 인원: 최대잔여법으로 반올림해 합계를 총 권고 인원과 일치시킨다. 1명 이하이면 분리하지 않는다(통합 응대)."""
    if total_staff <= 1 or not np.isfinite(domestic_share):
        return np.nan, np.nan
    raw = np.array([domestic_share, 1 - domestic_share]) * total_staff
    base = np.floor(raw).astype(int)
    for k in np.argsort(-(raw - base))[: int(total_staff - base.sum())]:
        base[k] += 1
    return int(base[0]), int(base[1])


S90 = S_UP90.loc[(S_UP90["model_id"] == FINAL_MODEL) & (S_UP90["target"] == "total"), ["site", "s_up_level"]]
f14 = F13.copy()
f14["_date"] = pd.to_datetime(f14["date"])
f14["holiday_name"] = RAW_CAL["holiday_name"].reindex(f14["_date"]).fillna("").to_numpy()
f14["is_holiday"] = RAW_CAL["is_holiday"].reindex(f14["_date"]).astype(int).to_numpy()
f14["is_labor_day"] = f14["_date"].dt.strftime("%m-%d").eq("05-01").astype(int).to_numpy()
f14["is_free_open"] = [int(FREE_OPEN[s].loc[d]) for s, d in zip(f14["site"], f14["_date"])]
for column in ("rain_prob", "rain_1_10_prob", "heavy_prob", "after_heavy_prob"):
    f14[column] = [float(COVARIATES["final"][s].loc[d, column]) for s, d in zip(f14["site"], f14["_date"])]


def expected_rain_class(row):
    options = {"비 안 옴": 1 - row["rain_prob"], "1~10mm 비": row["rain_1_10_prob"], "10mm 이상 큰비": row["heavy_prob"]}
    label = max(options, key=options.get)
    return f"{label} 우세(기후 확률 {options[label]:.0%})"


f14["expected_rain_class"] = f14.apply(expected_rain_class, axis=1)
f14["is_golden_holiday"] = GOLDEN_FLAG.reindex(f14["_date"]).to_numpy()
f14["is_myeongjeol"] = MYEONGJEOL_FLAG.reindex(f14["_date"]).to_numpy()
f14["predicted_total_point"] = to_visitors_int(f14["predicted_total_point"])
f14["predicted_total_q80"] = to_visitors_int(f14["predicted_total_q80"])
f14["predicted_domestic"] = to_visitors_int(f14["predicted_domestic"])
f14["predicted_foreign"] = (f14["predicted_total_point"] - f14["predicted_domestic"]).clip(lower=0)
f14["predicted_total_point"] = f14["predicted_domestic"] + f14["predicted_foreign"]
closed = f14["is_regular_closed"].to_numpy()
f14["required_staff_point"] = staff_rule(f14["predicted_total_point"], closed)
f14["recommended_staff_q80"] = staff_rule(f14["predicted_total_q80"], closed)
q90_by_site = f14[["site"]].merge(S90, on="site", how="left")["s_up_level"].to_numpy()
raw_total_point = F13["predicted_total_point"].to_numpy()
f14["predicted_total_q90"] = to_visitors_int(np.clip(raw_total_point + q90_by_site, 0, None) * (~closed))
f14["staff_q90"] = staff_rule(f14["predicted_total_q90"], closed)
scenario, apply_to = GOLDEN_ADD["scenario"], GOLDEN_ADD["apply_to"]
golden_day = (f14["is_golden_holiday"] | f14["is_myeongjeol"]).to_numpy() & ~closed
applies = golden_day & (f14["group"].isin(["궁", "종묘"]).to_numpy() if apply_to == "궁·종묘만" else True)
if "q90" in scenario:
    f14["final_operational_staff"] = np.where(applies, f14["staff_q90"], f14["recommended_staff_q80"])
elif "+ 1" in scenario or "+ 2" in scenario:
    f14["final_operational_staff"] = f14["recommended_staff_q80"] + np.where(applies, 1 if "+ 1" in scenario else 2, 0)
else:
    f14["final_operational_staff"] = f14["recommended_staff_q80"]
f14["final_operational_staff"] = f14["final_operational_staff"].astype(int)
f14["golden_addon_applied"] = applies


def rule_label(row):
    if row["is_regular_closed"]:
        return STAFF_RULE_TEXT["closed"]
    if row["group"] != "왕릉":
        return STAFF_RULE_TEXT["palace"]
    return STAFF_RULE_TEXT["tomb_over"] if row["predicted_total_q80"] > STAFF_UNIT else STAFF_RULE_TEXT["tomb_normal"]


f14["staffing_rule"] = f14.apply(rule_label, axis=1)
total_points = (f14["predicted_domestic"] + f14["predicted_foreign"]).replace(0, np.nan)
f14["domestic_share"] = (f14["predicted_domestic"] / total_points).round(4)
f14["foreign_share"] = (f14["predicted_foreign"] / total_points).round(4)
splits = [split_staff(s, d) for s, d in zip(f14["final_operational_staff"], f14["domestic_share"])]
f14["recommended_domestic_service_staff"] = [a for a, _ in splits]
f14["recommended_foreign_service_staff"] = [b for _, b in splits]
f14["recommended_domestic_service_share"] = (f14["recommended_domestic_service_staff"] / f14["final_operational_staff"]).round(4)
f14["recommended_foreign_service_share"] = (f14["recommended_foreign_service_staff"] / f14["final_operational_staff"]).round(4)
f14["service_mode"] = np.where(f14["is_regular_closed"], "휴관", np.where(f14["final_operational_staff"] <= 1, "통합 응대", "내·외국인 분담 참고"))
rank_in_day = f14.loc[~f14["is_regular_closed"]].groupby("date")["predicted_total_q80"].rank(ascending=False, method="first")
f14["high_demand_rank_in_day"] = rank_in_day.reindex(f14.index)


def risk_note(row):
    notes = []
    if row["is_regular_closed"]:
        return "정기 휴관일 (0명)"
    if row["holiday_name"]:
        notes.append(f"공휴일({row['holiday_name']})")
    if row["is_labor_day"] == 1:
        notes.append("근로자의 날(특수 수요일)")
    if row["is_free_open"] == 1:
        notes.append("무료개방")
    if row["golden_addon_applied"]:
        notes.append("황금연휴·명절 추가 배치 적용")
    if row["predicted_total_point"] > 0 and row["predicted_total_q80"] / row["predicted_total_point"] >= 1.5:
        notes.append("불확실성 큼(q80/점예측 ≥ 1.5)")
    if row["group"] == "왕릉" and row["final_operational_staff"] >= 2:
        notes.append("왕릉 q80 2,000명 초과 → 2명 이상")
    if row["high_demand_rank_in_day"] <= 5 and row["group"] != "왕릉":
        notes.append(f"당일 고수요 {int(row['high_demand_rank_in_day'])}위")
    return "; ".join(notes) if notes else "특이사항 없음"


f14["risk_note"] = f14.apply(risk_note, axis=1)
STAFF_COLS_ORDER = ["date", "weekday", "group", "site", "holiday_name", "is_holiday", "is_labor_day", "is_free_open", "expected_rain_class", "predicted_total_point", "predicted_total_q80", "predicted_domestic", "predicted_foreign",
                    "required_staff_point", "recommended_staff_q80", "final_operational_staff", "staffing_rule", "risk_note", "domestic_share", "foreign_share", "recommended_domestic_service_share", "recommended_foreign_service_share",
                    "recommended_domestic_service_staff", "recommended_foreign_service_staff", "service_mode", "is_regular_closed", "rain_prob", "rain_1_10_prob", "heavy_prob", "after_heavy_prob"]
F14 = f14[STAFF_COLS_ORDER].copy()
# ---------- 요약: 일별 / 개소별 / 유형별 / 고수요 Top5 / 특별관리일 ----------
daily_g = f14.groupby("date")
F15 = daily_g.agg(weekday=("weekday", "first"), holiday_name=("holiday_name", lambda s: ", ".join(sorted(set(x for x in s if x)))), open_sites=("is_regular_closed", lambda s: int((~s).sum())), predicted_total_point=("predicted_total_point", "sum"),
                  predicted_total_q80=("predicted_total_q80", "sum"), required_staff_point=("required_staff_point", "sum"), recommended_staff_q80=("recommended_staff_q80", "sum"), final_operational_staff=("final_operational_staff", "sum")).reset_index()
for g_name, key in (("궁", "palace_staff"), ("종묘", "jongmyo_staff"), ("왕릉", "royal_tomb_staff")):
    F15[key] = F15["date"].map(f14.loc[f14["group"] == g_name].groupby("date")["final_operational_staff"].sum()).fillna(0).astype(int)
top5 = f14.loc[~f14["is_regular_closed"] & (f14["high_demand_rank_in_day"] <= 5)].sort_values(["date", "high_demand_rank_in_day"])
F15["high_demand_top5_sites"] = F15["date"].map(top5.groupby("date").apply(lambda d: ", ".join(f"{r['site']}({int(r['final_operational_staff'])}명)" for _, r in d.iterrows()), include_groups=False))
F15["special_note"] = F15["date"].map(f14.groupby("date")["risk_note"].apply(lambda s: "; ".join(sorted({n for note in s for n in note.split("; ") if n.startswith(("공휴일", "근로자", "무료개방", "황금연휴"))}))))
F15["special_note"] = F15["special_note"].fillna("")
F16_site = f14.groupby(["group", "site"]).agg(open_days=("is_regular_closed", lambda s: int((~s).sum())), closed_days=("is_regular_closed", "sum"), predicted_total_point_sum=("predicted_total_point", "sum"), predicted_total_q80_sum=("predicted_total_q80", "sum"),
                                              person_days_point=("required_staff_point", "sum"), person_days_q80=("recommended_staff_q80", "sum"), person_days_final=("final_operational_staff", "sum"), max_daily_staff=("final_operational_staff", "max"),
                                              mean_daily_staff_open_days=("final_operational_staff", lambda s: float(s[s > 0].mean()))).reset_index()
peak = f14.sort_values(["site", "final_operational_staff", "predicted_total_q80"], ascending=[True, False, False]).drop_duplicates("site").set_index("site")["date"]
F16_site["peak_date"] = F16_site["site"].map(peak)
F16_site["row_type"] = "개소"
F16_group = f14.groupby("group").agg(open_days=("is_regular_closed", lambda s: int((~s).sum())), closed_days=("is_regular_closed", "sum"), predicted_total_point_sum=("predicted_total_point", "sum"), predicted_total_q80_sum=("predicted_total_q80", "sum"),
                                     person_days_point=("required_staff_point", "sum"), person_days_q80=("recommended_staff_q80", "sum"), person_days_final=("final_operational_staff", "sum"), max_daily_staff=("final_operational_staff", "max")).reset_index()
F16_group["site"] = "유형 합계"
F16_group["row_type"] = "유형"
F16_total = pd.DataFrame([dict(group="전체", site="전체 합계", row_type="전체", open_days=int((~f14["is_regular_closed"]).sum()), closed_days=int(f14["is_regular_closed"].sum()), predicted_total_point_sum=int(f14["predicted_total_point"].sum()),
                               predicted_total_q80_sum=int(f14["predicted_total_q80"].sum()), person_days_point=int(f14["required_staff_point"].sum()), person_days_q80=int(f14["recommended_staff_q80"].sum()), person_days_final=int(f14["final_operational_staff"].sum()))])
F16 = pd.concat([F16_site, F16_group, F16_total], ignore_index=True)
F16["_o"] = F16["row_type"].map({"개소": 0, "유형": 1, "전체": 2})
F16["_g"] = F16["group"].map({"궁": 0, "종묘": 1, "왕릉": 2, "전체": 3})
F16 = F16.sort_values(["_o", "_g", "site"]).drop(columns=["_o", "_g"]).reset_index(drop=True)
F16 = F16[["row_type", "group", "site", "open_days", "closed_days", "predicted_total_point_sum", "predicted_total_q80_sum", "person_days_point", "person_days_q80", "person_days_final", "max_daily_staff", "mean_daily_staff_open_days", "peak_date"]]
SPECIAL_DAYS_26 = F15.loc[(F15["special_note"] != "") | (F15["final_operational_staff"] >= F15["final_operational_staff"].quantile(0.75)), ["date", "weekday", "holiday_name", "special_note", "predicted_total_point", "predicted_total_q80", "final_operational_staff", "high_demand_top5_sites"]].copy()
SPECIAL_DAYS_26["memo"] = np.where(SPECIAL_DAYS_26["special_note"] != "", "특별관리일: " + SPECIAL_DAYS_26["special_note"], "고수요일(일별 총 인원 상위 25%): 궁 인력 우선 점검")
TOP5_T = top5[["date", "weekday", "high_demand_rank_in_day", "group", "site", "predicted_total_point", "predicted_total_q80", "final_operational_staff", "risk_note"]].reset_index(drop=True)
TOMB_MULTI = F14.loc[(F14["group"] == "왕릉") & (F14["final_operational_staff"] >= 2), ["date", "site", "predicted_total_point", "predicted_total_q80", "final_operational_staff"]]
print(f"2026 운영 인력: 총 {int(F14['final_operational_staff'].sum())} person-days (궁 {int(F14.loc[F14['group'] == '궁', 'final_operational_staff'].sum())}, 종묘 {int(F14.loc[F14['group'] == '종묘', 'final_operational_staff'].sum())}, 왕릉 {int(F14.loc[F14['group'] == '왕릉', 'final_operational_staff'].sum())}) | 점 예측 환산 {int(F14['required_staff_point'].sum())} | 왕릉 2명 이상 개소·일 {len(TOMB_MULTI)}")
display(F15[["date", "weekday", "holiday_name", "predicted_total_point", "predicted_total_q80", "final_operational_staff", "palace_staff", "jongmyo_staff", "royal_tomb_staff"]])
mark("2026 운영 인력 배치표")

In [ ]:
def write_staffing_excel(path):
    """운영팀용 Excel: 제목행 고정, 필터, 천 단위 표시, 조건부 서식."""
    with pd.ExcelWriter(path, engine="xlsxwriter") as writer:
        wb = writer.book
        head = wb.add_format({"bold": True, "bg_color": "#1F5E4B", "font_color": "white", "border": 1, "text_wrap": True, "valign": "vcenter", "align": "center"})
        num = wb.add_format({"num_format": "#,##0"})
        pct = wb.add_format({"num_format": "0.0%"})
        title = wb.add_format({"bold": True, "font_size": 14})
        wrap = wb.add_format({"text_wrap": True, "valign": "top"})
        red = wb.add_format({"bg_color": "#F8D7DA", "font_color": "#9C0006", "bold": True})
        amber = wb.add_format({"bg_color": "#FFF2CC"})
        green_header = wb.add_format({"bold": True, "bg_color": "#DCEBE4", "border": 1})
        # 안내 시트
        ws = wb.add_worksheet("안내")
        writer.sheets["안내"] = ws
        ws.set_column(0, 0, 120)
        notes = ["Project 3 2026-01-01~14 운영 인력 배치표 (운영팀 전달용)", "",
                 "※ 권고 배치 인원은 실제 근무표상의 배치 인원이 아니라, 예측 방문객(q80)과 운영 기준(2,000명당 1명)으로 산출한 관람객 대응 필요 인원입니다.",
                 f"최종 권고 모델: {REG[FINAL_MODEL]['label']} ({FINAL_MODEL}) | 학습 2023-01-01~2025-12-31, 예측 2026-01-01~14 | 2025 순차 백테스트(14일, {len(TEST_WINDOWS)}개 원점) 기준 WAPE {OVERALL.set_index('model_id').loc[FINAL_MODEL, 'WAPE']:.2f}%",
                 "규칙: 필요 인원 = ceil(방문객/2,000), 개방일 최소 1명, 정기 휴관일 0명. 운영 예측은 q80(부족 1인·일 600,000원 : 잉여 150,000원 → 임계비율 0.8).",
                 "왕릉은 개방일 최소 1명이며 q80이 2,000명을 넘으면 2명 이상. 궁·종묘는 q80 기준 정밀 배치.",
                 "내·외국인 응대 인원은 강제 배정이 아니라 예측 비중 기반 참고값이며, 총 1명이면 '통합 응대'.",
                 "강수는 기후 평균 기반 사전 추정치이며 인력 감축의 단독 근거로 쓰지 말고 보조 신호로 사용하세요. 실제 기상예보가 확보되면 배치 확정 3일 전에 갱신하세요.",
                 "시트: 일별 요약 / 개소별 요약 / 개소×일 배치표(필터) / 배치 히트맵(개소×날짜) / 고수요 Top5 / 특별관리일"]
        for i, text in enumerate(notes):
            ws.write(i, 0, text, title if i == 0 else wrap)

        def add_table(sheet, frame, widths, number_cols=(), pct_cols=(), start_note=None):
            frame.to_excel(writer, sheet_name=sheet, index=False, startrow=0)
            w = writer.sheets[sheet]
            for j, column in enumerate(frame.columns):
                w.write(0, j, column, head)
                fmt = num if column in number_cols else (pct if column in pct_cols else None)
                w.set_column(j, j, widths.get(column, 14), fmt)
            w.freeze_panes(1, 0)
            w.autofilter(0, 0, len(frame), len(frame.columns) - 1)
            w.set_row(0, 32)
            return w

        d = F15.rename(columns={"palace_staff": "궁_인원", "jongmyo_staff": "종묘_인원", "royal_tomb_staff": "왕릉_인원"})
        w = add_table("일별 요약", d, {"date": 12, "holiday_name": 18, "high_demand_top5_sites": 60, "special_note": 40}, number_cols=("predicted_total_point", "predicted_total_q80", "required_staff_point", "recommended_staff_q80", "final_operational_staff", "궁_인원", "종묘_인원", "왕릉_인원"))
        idx = list(d.columns).index("final_operational_staff")
        w.conditional_format(1, idx, len(d), idx, {"type": "3_color_scale", "min_color": "#FFFFFF", "mid_color": "#FFE699", "max_color": "#F4B183"})
        s = F16.copy()
        w = add_table("개소별 요약", s, {"site": 16, "peak_date": 12}, number_cols=("predicted_total_point_sum", "predicted_total_q80_sum", "person_days_point", "person_days_q80", "person_days_final"))
        idx = list(s.columns).index("person_days_final")
        w.conditional_format(1, idx, len(s), idx, {"type": "data_bar", "bar_color": "#7FB5A0"})
        t = F14[["date", "weekday", "group", "site", "holiday_name", "is_holiday", "is_labor_day", "is_free_open", "expected_rain_class", "predicted_total_point", "predicted_total_q80", "predicted_domestic", "predicted_foreign",
                 "required_staff_point", "recommended_staff_q80", "final_operational_staff", "service_mode", "recommended_domestic_service_staff", "recommended_foreign_service_staff", "staffing_rule", "risk_note"]]
        w = add_table("개소×일 배치표", t, {"date": 12, "weekday": 6, "site": 14, "holiday_name": 14, "expected_rain_class": 26, "staffing_rule": 40, "risk_note": 50, "service_mode": 16},
                      number_cols=("predicted_total_point", "predicted_total_q80", "predicted_domestic", "predicted_foreign", "required_staff_point", "recommended_staff_q80", "final_operational_staff"))
        idx = list(t.columns).index("final_operational_staff")
        w.conditional_format(1, idx, len(t), idx, {"type": "cell", "criteria": ">=", "value": 2, "format": amber})
        w.conditional_format(1, idx, len(t), idx, {"type": "cell", "criteria": ">=", "value": 10, "format": red})
        w.conditional_format(1, 0, len(t), len(t.columns) - 1, {"type": "formula", "criteria": f'=$E2<>""', "format": wb.add_format({"font_color": "#C0392B"})})
        pivot = F14.pivot(index="site", columns="date", values="final_operational_staff").reindex(index=[s_ for g_ in GROUPS for s_ in sorted(F14.loc[F14["group"] == g_, "site"].unique())])
        pivot.insert(0, "유형", [SITE_GROUP[s_] for s_ in pivot.index])
        pivot.reset_index().to_excel(writer, sheet_name="배치 히트맵", index=False)
        w = writer.sheets["배치 히트맵"]
        for j, column in enumerate(["site"] + list(pivot.columns)):
            w.write(0, j, "개소" if column == "site" else str(column), head)
        w.set_column(0, 0, 16)
        w.set_column(1, 1, 8)
        w.set_column(2, len(pivot.columns), 11, num)
        w.freeze_panes(1, 2)
        w.conditional_format(1, 2, len(pivot), len(pivot.columns), {"type": "3_color_scale", "min_color": "#FFFFFF", "mid_color": "#FFE699", "max_color": "#C55A11"})
        tomb_rows = [i for i, s_ in enumerate(pivot.index, start=1) if SITE_GROUP[s_] == "왕릉"]
        for i in tomb_rows:
            w.conditional_format(i, 2, i, len(pivot.columns), {"type": "cell", "criteria": ">=", "value": 2, "format": wb.add_format({"bold": True, "border": 2, "border_color": "#C0392B"})})
        add_table("고수요 Top5", TOP5_T, {"date": 12, "site": 14, "risk_note": 60}, number_cols=("predicted_total_point", "predicted_total_q80", "final_operational_staff"))
        add_table("특별관리일", SPECIAL_DAYS_26, {"date": 12, "holiday_name": 16, "special_note": 40, "high_demand_top5_sites": 60, "memo": 60}, number_cols=("predicted_total_point", "predicted_total_q80", "final_operational_staff"))

## 15. PPT·보고서용 최종 시각화 (16개, 300dpi)
메시지형 제목, 축 단위·기간·표본 수 표기, 색맹 친화 팔레트(진한 초록 + 절제된 단청 계열 포인트), 16:9 비율, 최소 글자 크기 10pt 이상. 그림은 검산이 모두 PASS한 뒤 저장 단계에서 파일로 만들고, 그림별 핵심 해석·PPT/보고서 위치는 `18_figure_manifest.csv`와 `summary/ppt_report_placement_guide.md`에 정리합니다.

In [ ]:
def setup_korean_font():
    """Noto Sans KR(가변 폰트는 보통·굵은 굵기의 정적 폰트로 변환) → 맑은 고딕 → 나눔고딕 순으로 사용 가능한 한글 폰트를 적용한다."""
    sns.set_theme(style="whitegrid")
    variable = Path("C:/Windows/Fonts/NotoSansKR-VF.ttf")
    statics = {400: CACHE_DIR / "NotoSansKR-Regular-wght400.ttf", 700: CACHE_DIR / "NotoSansKR-Bold-wght700.ttf"}
    if variable.exists():
        try:
            from fontTools.ttLib import TTFont
            from fontTools.varLib import instancer
            for weight, path in statics.items():
                if not path.exists():
                    instancer.instantiateVariableFont(TTFont(str(variable)), {"wght": weight}).save(str(path))
        except Exception as error:
            print("[참고] Noto Sans KR 가변 폰트를 정적 폰트로 변환하지 못했습니다:", type(error).__name__)
    name = None
    for path in [statics[400], Path("C:/Windows/Fonts/NotoSansKR-Regular.ttf"), Path("C:/Windows/Fonts/malgun.ttf")]:
        if path.exists():
            fm.fontManager.addfont(str(path))
            name = fm.FontProperties(fname=str(path)).get_name()
            break
    if statics[700].exists():
        fm.fontManager.addfont(str(statics[700]))
    if name is None:
        installed = sorted({f.name for f in fm.fontManager.ttflist if any(k in f.name.lower() for k in ("noto sans cjk kr", "noto sans kr", "nanum", "malgun"))})
        name = installed[0] if installed else None
    if name is None:
        warnings.warn("한글 폰트를 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.")
    else:
        plt.rcParams["font.family"] = name
    plt.rcParams["axes.unicode_minus"] = False
    plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold"})
    return name



FONT_NAME = setup_korean_font()
plt.rcParams.update({"axes.titlesize": 15, "axes.labelsize": 12, "xtick.labelsize": 11, "ytick.labelsize": 11, "legend.fontsize": 11, "font.size": 11})
GREEN, GREEN_L, VERM, BLUE, YELLOW, SKY, PURPLE, GREY, DARK = "#1B6B4F", "#8CC5AE", "#D55E00", "#0072B2", "#E69F00", "#56B4E9", "#CC79A7", "#8D8D8D", "#333333"
BG = "#FFFDF7"
SOURCE = "자료: 국가유산청 궁능유적본부 관람객 데이터, 강수량 전처리 자료, 한국 공휴일 달력; 연구팀 분석. 인력은 실제 근무표가 아니라 q80과 2,000명당 1명 가정의 환산 필요 인원(부족 60만 원·잉여 15만 원)."
BACKTEST_NOTE = f"2024년까지 초기 학습 후 2025년 순차 rolling-origin 백테스트(14일, {len(TEST_WINDOWS)}개 원점, 휴관일 제외)"
FIG_FILES = []
SHORT = {"baseline_naive": "나이브", "baseline_seasonal_naive": "계절 나이브", "baseline_mean7": "직전 7일 평균", "arima": "ARIMA", "sarima": "SARIMA", "mlp_global": "글로벌 MLP", "chronos_zero": "C2 제로샷",
         "chronos_holiday": "C2 +휴일", "chronos_weather": "C2 +강수", "chronos_free": "C2 +무료개방", "chronos_holiday_weather": "C2 +휴일+강수", "chronos_holiday_free": "C2 +휴일+무료개방",
         "chronos_weather_free": "C2 +강수+무료개방", "chronos_holiday_weather_free": "C2 +휴일+강수+무료개방"}


def new_fig(title, subtitle=None, nrows=1, ncols=1, **kw):
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 9), facecolor=BG, **kw)
    fig.suptitle(title, fontsize=19, fontweight="bold", x=0.015, ha="left", y=0.985, color=DARK)
    if subtitle:
        fig.text(0.015, 0.915, subtitle, fontsize=12, color="#555555", ha="left")
    return fig, axes


def save_fig(fig, name, subtitle_note=None):
    fig.text(0.015, 0.012, SOURCE, fontsize=9, color="#777777", ha="left", va="bottom")
    path = FIG_DIR / name
    fig.savefig(path, dpi=300, facecolor=BG)
    plt.close(fig)
    FIG_FILES.append(name)
    return path


def model_color(model_id):
    if model_id == FINAL_MODEL:
        return GREEN
    if model_id == OPT1 and OPT1 != FINAL_MODEL:
        return BLUE
    if model_id == "baseline_mean7":
        return VERM
    return GREY if REG[model_id]["kind"] == "baseline" else (SKY if REG[model_id]["kind"] == "chronos" else YELLOW)


def style_axes(ax, grid_axis="x"):
    ax.set_facecolor(BG)
    ax.grid(False)
    ax.grid(axis=grid_axis, color="#DDDDDD", linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)


def barh_panel(ax, table, column, title, fmt, scale=1.0, order=None, target=None):
    data = table.set_index("model_id").loc[order]
    labels = [SHORT[m] for m in data.index][::-1]
    values = (data[column].to_numpy() * scale)[::-1]
    bars = ax.barh(labels, values, color=[model_color(m) for m in data.index][::-1])
    for bar, v in zip(bars, values):
        ax.text(bar.get_width() + max(values) * 0.01, bar.get_y() + bar.get_height() / 2, fmt.format(v), va="center", fontsize=11)
    if target is not None:
        ax.axvline(target, color=DARK, linestyle="--", linewidth=1.2)
    ax.set_xlim(0, max(values.max(), target or 0) * 1.16)
    ax.set_title(title, loc="left", fontsize=14)
    style_axes(ax)
    for tick, m in zip(ax.get_yticklabels(), list(data.index)[::-1]):
        if m == FINAL_MODEL:
            tick.set_fontweight("bold")
            tick.set_color(GREEN)


def fig01(order):
    fig, axes = new_fig(f"최종 권고 {SHORT[FINAL_MODEL]}: WAPE {OVERALL.set_index('model_id').loc[FINAL_MODEL, 'WAPE']:.1f}%로 직전 7일 평균({OVERALL.set_index('model_id').loc['baseline_mean7', 'WAPE']:.1f}%)보다 정확",
                        f"{BACKTEST_NOTE}, 총방문객, 전체 {N_MODEL_ROWS}개 모델 행. 초록=최종 권고, 파랑=정확도·비용 최우선(다른 경우), 주황=운영 기준선", 1, 2, gridspec_kw=dict(wspace=0.45))
    fig.subplots_adjust(top=0.84, bottom=0.08, left=0.14, right=0.97)
    barh_panel(axes[0], OVERALL, "WAPE", "WAPE (%) - 낮을수록 정확 (1순위)", "{:.1f}", order=order)
    barh_panel(axes[1], OVERALL, "RelMAE", "RelMAE (계절 나이브 대비, 보조)", "{:.2f}", order=order)
    return save_fig(fig, "01_model_performance_wape_relmae.png")


def fig02(order):
    fig, axes = new_fig("q80은 목표 80%에 가깝고 구간은 모델마다 길이가 다르다", f"{BACKTEST_NOTE}. 점선 = 목표 80%. 적중률 = 실제 ≤ q80 상당 값, 포함률 = 실제가 80% 구간 안, 평균 구간 길이 = (상한−하한)의 평균", 1, 3, gridspec_kw=dict(wspace=0.5))
    fig.subplots_adjust(top=0.84, bottom=0.08, left=0.12, right=0.97)
    barh_panel(axes[0], OVERALL, "hit_rate", "q80 적중률 (%)", "{:.1f}", 100, order=order, target=80)
    barh_panel(axes[1], OVERALL, "coverage", "80% 구간 포함률 (%)", "{:.1f}", 100, order=order, target=80)
    barh_panel(axes[2], OVERALL, "mean_interval_width", "평균 구간 길이 (명/일·개소)", "{:,.0f}", order=order)
    for ax in axes[1:]:
        ax.set_yticklabels([])
    return save_fig(fig, "02_probabilistic_hit_coverage_width.png")


def fig03():
    q = FVA_T.loc[FVA_T["staffing_basis"].str.startswith("q80")].set_index("model_id")
    order = q.sort_values("total_cost_krw").index.tolist()
    fig, axes = new_fig(f"총비용 {q.loc[FINAL_MODEL, 'total_cost_krw'] / 1e8:.2f}억 원: 직전 7일 평균({q.loc['baseline_mean7', 'total_cost_krw'] / 1e8:.2f}억) 대비 {q.loc[FINAL_MODEL, 'saving_vs_mean7_krw'] / 1e8:.2f}억 원({q.loc[FINAL_MODEL, 'saving_rate_vs_mean7']:.0%}) 절감",
                        f"{BACKTEST_NOTE}. 동일한 q80 인력 정책 기준. 부족 비용(1인·일 60만 원)과 잉여 비용(15만 원)을 누적 막대로 분리", 1, 1)
    fig.subplots_adjust(top=0.84, bottom=0.1, left=0.17, right=0.95)
    labels = [SHORT[m] for m in order][::-1]
    s = q.loc[order, "shortage_cost_krw"].to_numpy()[::-1] / 1e8
    o = q.loc[order, "surplus_cost_krw"].to_numpy()[::-1] / 1e8
    axes.barh(labels, s, color=VERM, label="부족 비용 (억 원)")
    axes.barh(labels, o, left=s, color=GREEN_L, label="잉여 비용 (억 원)")
    for i, (a, b) in enumerate(zip(s, o)):
        axes.text(a + b + 0.02, i, f"{a + b:.2f}억", va="center", fontsize=11)
    axes.set_xlabel("총 인력 오차 비용 (억 원, 2025 테스트 합계)")
    axes.legend(loc="lower right")
    style_axes(axes)
    for tick, m in zip(axes.get_yticklabels(), order[::-1]):
        if m == FINAL_MODEL:
            tick.set_fontweight("bold")
            tick.set_color(GREEN)
    return save_fig(fig, "03_cost_and_fva_vs_mean7.png")


def fig04():
    g = GROUP_T.loc[GROUP_T["model_id"].isin([FINAL_MODEL, "baseline_mean7"])]
    fig, axes = new_fig("궁은 q80으로 정밀 배치, 왕릉은 최소 1명 정책: 유형별 WAPE와 비용이 크게 다르다", f"{BACKTEST_NOTE}. 최종 권고 vs 직전 7일 평균. 궁·종묘는 방문객 규모가 커 비용의 대부분을 차지", 1, 2, gridspec_kw=dict(wspace=0.25))
    fig.subplots_adjust(top=0.84, bottom=0.1, left=0.07, right=0.97)
    x = np.arange(len(GROUPS))
    for ax, column, title, scale, fmt in ((axes[0], "WAPE", "유형별 WAPE (%)", 1, "{:.1f}"), (axes[1], "total_cost_krw", "유형별 인력 오차 총비용 (억 원)", 1e-8, "{:.2f}")):
        for k, (mid, color, name) in enumerate(((FINAL_MODEL, GREEN, f"최종 권고 {SHORT[FINAL_MODEL]}"), ("baseline_mean7", VERM, "직전 7일 평균"))):
            vals = [float(g.loc[(g["group"] == grp) & (g["model_id"] == mid), column].iloc[0]) * scale for grp in GROUPS]
            bars = ax.bar(x + (k - 0.5) * 0.36, vals, 0.36, color=color, label=name)
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width() / 2, v + max(vals) * 0.015, fmt.format(v), ha="center", fontsize=12)
        n_text = [int(g.loc[(g["group"] == grp) & (g["model_id"] == FINAL_MODEL), "n_observations"].iloc[0]) for grp in GROUPS]
        ax.set_xticks(x)
        ax.set_xticklabels([f"{grp}\n(n={n:,})" for grp, n in zip(GROUPS, n_text)], fontsize=13)
        ax.set_title(title, loc="left", fontsize=14)
        style_axes(ax, "y")
    axes[0].legend(loc="upper left")
    return save_fig(fig, "04_group_wape_cost.png")


def fig05():
    s = SITE_T.loc[SITE_T["model_id"] == FINAL_MODEL].copy()
    base = SITE_T.loc[SITE_T["model_id"] == "baseline_mean7"].set_index("site")
    s["_g"] = s["group"].map({"궁": 0, "종묘": 1, "왕릉": 2})
    s = s.sort_values(["_g", "site"]).reset_index(drop=True)
    sites = s["site"].tolist()
    fig, axes = new_fig(f"23개 개소 중 WAPE가 직전 7일 평균보다 낮은 개소 {int((s['WAPE'].to_numpy() < base['WAPE'].reindex(s['site']).to_numpy()).sum())}곳", f"{BACKTEST_NOTE}. 최종 권고 {SHORT[FINAL_MODEL]} 개소별 지표 (행: 개소, 궁→종묘→왕릉 순)", 1, 4, gridspec_kw=dict(wspace=0.06))
    fig.subplots_adjust(top=0.86, bottom=0.05, left=0.1, right=0.98)
    panels = (("WAPE", "WAPE (%)", 1, "{:.0f}", "YlOrRd"), ("hit_rate", "q80 적중률 (%)", 100, "{:.0f}", "RdYlGn"), ("coverage", "80% 포함률 (%)", 100, "{:.0f}", "RdYlGn"), ("total_cost_krw", "총비용 (만 원)", 1e-4, "{:,.0f}", "YlOrRd"))
    for k, (ax, (column, title, scale, fmt, cmap)) in enumerate(zip(axes, panels)):
        values = (s[column].to_numpy() * scale).reshape(-1, 1)
        norm_kwargs = dict(vmin=60, vmax=100) if column in ("hit_rate", "coverage") else {}
        ax.imshow(values, cmap=cmap, aspect="auto", **norm_kwargs)
        ax.grid(False)
        for i, v in enumerate(values[:, 0]):
            ax.text(0, i, fmt.format(v), ha="center", va="center", fontsize=11, color=DARK, fontweight="bold")
        ax.set_xticks([0])
        ax.set_xticklabels([title], fontsize=12)
        ax.xaxis.tick_top()
        ax.set_yticks(range(len(sites)) if k == 0 else [])
        if k == 0:
            ax.set_yticklabels([f"{site} ({SITE_GROUP[site]})" for site in sites], fontsize=11)
        for spine in ax.spines.values():
            spine.set_visible(False)
    return save_fig(fig, "05_site_heatmap.png")


def fig06():
    t = DATE_TYPE_T.loc[DATE_TYPE_T["model_id"] == FINAL_MODEL].set_index("date_type").reindex(DATE_TYPES).fillna(0)
    m7 = DATE_TYPE_T.loc[DATE_TYPE_T["model_id"] == "baseline_mean7"].set_index("date_type").reindex(DATE_TYPES).fillna(0)
    fig, axes = new_fig("공휴일·황금연휴·명절에서 오차와 부족 인원이 집중된다", f"{BACKTEST_NOTE}. 최종 권고 {SHORT[FINAL_MODEL]} (직전 7일 평균은 비교용). 표본 수 = 모델 1개 기준 개소·일 행 수", 1, 3, gridspec_kw=dict(wspace=0.32))
    fig.subplots_adjust(top=0.84, bottom=0.2, left=0.05, right=0.99)
    labels = [f"{d}\n(n={int(t.loc[d, 'n_observations']):,})" for d in DATE_TYPES]
    x = np.arange(len(DATE_TYPES))
    for ax, column, title, scale, fmt in ((axes[0], "WAPE", "WAPE (%)", 1, "{:.0f}"), (axes[1], "shortage_person_days", "부족 인원 (인일)", 1, "{:,.0f}"), (axes[2], "hit_rate", "q80 적중률 (%)", 100, "{:.0f}")):
        a, b = t[column].to_numpy() * scale, m7[column].to_numpy() * scale
        ax.bar(x - 0.19, a, 0.36, color=GREEN, label="최종 권고")
        ax.bar(x + 0.19, b, 0.36, color=VERM, label="직전 7일 평균")
        for xi, (va, vb) in enumerate(zip(a, b)):
            ax.text(xi - 0.19, va + np.nanmax(np.r_[a, b]) * 0.012, fmt.format(va), ha="center", fontsize=10)
        if column == "hit_rate":
            ax.axhline(80, color=DARK, linestyle="--", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=10, rotation=0)
        ax.set_title(title, loc="left", fontsize=14)
        style_axes(ax, "y")
    axes[0].legend(loc="upper left")
    return save_fig(fig, "06_date_type_performance.png")


def fig07():
    t = CROSS_T.loc[CROSS_T["model_id"] == FINAL_MODEL].copy()
    rain_order = ["비 안 오는 주", "비 오는 주"]
    hol_order = [h for h in HOLIDAY_LABELS]
    fig, axes = new_fig("비 유무보다 휴일 유형에 따라 오차 차이가 크다", f"{BACKTEST_NOTE}. 최종 권고 {SHORT[FINAL_MODEL]}. 주 단위 조건(강수 주는 사후 평가용 실측값, 모델 입력 아님). 칸 안 = WAPE(%) / 표본 수 n", 1, 2, gridspec_kw=dict(wspace=0.3))
    fig.subplots_adjust(top=0.83, bottom=0.12, left=0.1, right=0.97)
    for ax, column, title, cmap, fmt in ((axes[0], "WAPE", "WAPE (%)", "YlOrRd", "{:.1f}"), (axes[1], "total_cost_krw", "총비용 (만 원)", "YlOrRd", "{:,.0f}")):
        grid = np.full((len(rain_order), len(hol_order)), np.nan)
        ns = np.zeros_like(grid)
        for i, r in enumerate(rain_order):
            for j, h in enumerate(hol_order):
                row = t.loc[(t["rain_week"] == r) & (t["holiday_week"] == h)]
                if len(row):
                    grid[i, j] = row[column].iloc[0] * (1e-4 if column == "total_cost_krw" else 1)
                    ns[i, j] = row["n_observations"].iloc[0]
        ax.imshow(grid, cmap=cmap, aspect="auto")
        ax.grid(False)
        for i in range(grid.shape[0]):
            for j in range(grid.shape[1]):
                ax.text(j, i, f"{fmt.format(grid[i, j])}\nn={int(ns[i, j]):,}" if np.isfinite(grid[i, j]) else "표본 없음", ha="center", va="center", fontsize=13, fontweight="bold", color=DARK)
        ax.set_xticks(range(len(hol_order)))
        ax.set_xticklabels([h.replace("공휴일+주말", "공휴일+주말\n").replace("평일 공휴일", "평일 공휴일\n") for h in hol_order], fontsize=11)
        ax.set_yticks(range(len(rain_order)))
        ax.set_yticklabels(["비 안 옴\n(사후 실측)", "비 옴\n(사후 실측)"], fontsize=12)
        ax.set_title(title, loc="left", fontsize=14)
        for spine in ax.spines.values():
            spine.set_visible(False)
    return save_fig(fig, "07_rain_holiday_cross.png")


def fig08():
    daily = FINAL_ROWS.groupby("date").agg(actual=("actual_visitors", "sum"), point=("predicted_visitors", "sum"), q80=("prediction_q80", "sum"), special=("date_type", lambda s: s.iloc[0] if s.nunique() == 1 else "혼합"))
    flags = EVAL_BASE.loc[EVAL_BASE["model_id"] == FINAL_MODEL].groupby("date").agg(holiday=("is_holiday", "max"), free=("free_open", "max"), golden=("is_golden_holiday", "max"), myeong=("is_myeongjeol", "max"), labor=("is_labor_day", "max"))
    fig, ax = new_fig("2025년 최종 모델의 예측 추이: 연휴·무료개방일의 급증 피크를 점 예측이 과소 추정하고 q80이 일부 보완", f"{BACKTEST_NOTE}. 전체 개소 합계(개방 개소만) 일별 방문객. 배경색: 공휴일=연한 빨강, 무료개방일=연한 초록, 마커=근로자의 날", 1, 1)
    fig.subplots_adjust(top=0.84, bottom=0.1, left=0.07, right=0.98)
    for date, r in flags.iterrows():
        if r["holiday"] or r["golden"] or r["myeong"]:
            ax.axvspan(date - pd.Timedelta(hours=12), date + pd.Timedelta(hours=12), color="#F4C7C3", alpha=0.7, lw=0)
        elif r["free"]:
            ax.axvspan(date - pd.Timedelta(hours=12), date + pd.Timedelta(hours=12), color="#CFE8DB", alpha=0.7, lw=0)
    ax.plot(daily.index, daily["actual"], color=DARK, linewidth=1.4, label="실제 방문객")
    ax.plot(daily.index, daily["point"], color=GREEN, linewidth=1.4, label="최종 모델 점 예측")
    ax.plot(daily.index, daily["q80"], color=VERM, linewidth=1.2, linestyle="--", label="q80 (인력 산정 기준)")
    labor = flags.index[flags["labor"] == 1]
    ax.scatter(labor, daily.loc[labor, "actual"], color=YELLOW, edgecolor=DARK, s=110, zorder=5, label="근로자의 날(2025-05-01)")
    ax.set_ylabel("일 방문객 (명, 개소 합계)")
    ax.set_xlabel("2025년")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.legend(loc="upper left", ncol=2)
    style_axes(ax, "y")
    return save_fig(fig, "08_actual_vs_forecast_2025.png")


def fig09():
    top = TOP_SHORTAGE.copy()
    colors = {"설·추석 명절 수요 급증": VERM, "황금연휴 수요 급증": YELLOW, "공휴일 수요 급증": PURPLE, "무료개방 수요 급증": GREEN, "주말 수요 급증": BLUE, "근로자의 날 특수 수요": SKY}
    fig, ax = new_fig(f"인력 부족 상위 10건은 모두 연휴·공휴일·무료개방에 몰렸다 (황금연휴·명절 = 전체 부족의 {GOLDEN_SHORTAGE_SHARE:.0%})", f"{BACKTEST_NOTE}. 최종 권고 {SHORT[FINAL_MODEL]}. 막대 = 부족 인원(실제 방문객 환산 필요 − 모델 권고, q80 기준), 색 = 추정 원인", 1, 1)
    fig.subplots_adjust(top=0.84, bottom=0.1, left=0.28, right=0.96)
    labels = [f"{r['date']}({r['weekday']}) {r['site']}" for _, r in top.iterrows()][::-1]
    vals = top["staff_shortage"].to_numpy()[::-1]
    cols = [colors.get(c, GREY) for c in top["probable_cause"]][::-1]
    bars = ax.barh(labels, vals, color=cols)
    for bar, (_, r) in zip(bars, top.iloc[::-1].iterrows()):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2, f"{int(r['staff_shortage'])}명 (필요 {int(r['required_staff_from_actual_visitors'])} / 권고 {int(r['recommended_staff_q80'])}) · {r['date_type']}", va="center", fontsize=11)
    ax.set_xlim(0, vals.max() * 1.65)
    ax.set_xlabel("부족 인원 (명/일)")
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for k, c in colors.items() if k in set(top["probable_cause"])]
    ax.legend(handles, [k for k in colors if k in set(top["probable_cause"])], loc="lower right", title="추정 원인")
    style_axes(ax, "x")
    return save_fig(fig, "09_staff_shortage_top10.png")


def fig10():
    fig, axes = new_fig(f"근로자의 날(5/1) 방문객은 2015~2025년 평상시 일평균의 {RATIO_ALL:.2f}배, {N_SITES_HIGHER}/{N_SITES_LONG}개소가 평상시보다 높다", "장기 자료(2015~2025)는 계절성·요일·코로나가 섞여 있어 인과가 아니라 반복 관찰된 특수 수요 신호입니다. 2025년 하루의 오차만으로 장기 효과를 단정하지 않습니다", 1, 3, gridspec_kw=dict(width_ratios=[1.3, 1.2, 1.0], wspace=0.35))
    fig.subplots_adjust(top=0.84, bottom=0.17, left=0.06, right=0.98)
    ps = LABOR_EVIDENCE.loc[LABOR_EVIDENCE["analysis"] == "개소별(2015~2025 평균)"].sort_values("ratio")
    axes[0].barh(ps["site"], ps["ratio"], color=[GREEN if v > 1 else VERM for v in ps["ratio"]])
    axes[0].axvline(1, color=DARK, linestyle="--")
    axes[0].set_title("개소별 5월 1일 / 개소 일평균 (배)", loc="left", fontsize=13)
    axes[0].tick_params(axis="y", labelsize=9)
    yr = LABOR_EVIDENCE.loc[LABOR_EVIDENCE["analysis"] == "전체 개소 합계 연도별"]
    axes[1].bar(yr["year"], yr["ratio"], color=[GREEN if v > 1 else VERM for v in yr["ratio"]])
    axes[1].axhline(1, color=DARK, linestyle="--")
    for xi, v in enumerate(yr["ratio"]):
        axes[1].text(xi, v + 0.04, f"{v:.2f}", ha="center", fontsize=10)
    axes[1].set_title("연도별 5월 1일 합계 / 같은 해 일평균 (배)", loc="left", fontsize=13)
    axes[1].tick_params(axis="x", rotation=45)
    sens = pd.DataFrame({"분석": ["전체\n2015~25", "코로나\n제외", "같은 해 5월\n같은 요일", "개소 수 균형\n2019~25"], "배": [RATIO_ALL, RATIO_EX_COVID, RATIO_SAME_WEEKDAY, RATIO_BALANCED]})
    axes[2].bar(sens["분석"], sens["배"], color=GREEN)
    axes[2].axhline(1, color=DARK, linestyle="--")
    for xi, v in enumerate(sens["배"]):
        axes[2].text(xi, v + 0.03, f"{v:.2f}배", ha="center", fontsize=12, fontweight="bold")
    axes[2].set_title("민감도 (전체 개소 합계)", loc="left", fontsize=13)
    axes[2].tick_params(axis="x", labelsize=10)
    d1 = ABLATION.loc[(ABLATION["scope"] == "2025-05-01 하루(23개소)")].set_index("model_id")
    fig.text(0.06, 0.06, f"2025-05-01 하루(23개소) 백테스트: 근로자의 날 제외 모델 A 방문객 WAPE {d1.loc[ABL_ID, 'WAPE']:.1f}% → 포함 모델 B {d1.loc[B_ID, 'WAPE']:.1f}% (독립 표본 1일, 일반화하지 않음)", fontsize=12, color=DARK)
    for ax in axes:
        style_axes(ax, "y" if ax is not axes[0] else "x")
    return save_fig(fig, "10_labor_day_long_term_evidence.png")


def fig11():
    sc = ["전체 2025", "궁", "종묘", "왕릉", "2025-05-01 하루(23개소)"]
    fig, axes = new_fig("근로자의 날 ablation: 전체 지표는 휴일 묶음(B)이 소폭 유리, 근로자의 날 하루는 반대", f"{BACKTEST_NOTE}. A = 공식 공휴일 + 무료개방, B = 공식 공휴일 + 근로자의 날 + 무료개방 (같은 조건). 하루(2025-05-01) 표본은 23개소 행뿐", 2, 3, gridspec_kw=dict(wspace=0.3, hspace=0.55))
    fig.subplots_adjust(top=0.82, bottom=0.1, left=0.06, right=0.98)
    metrics = (("WAPE", "WAPE (%)", 1, "{:.1f}"), ("total_cost_krw", "총비용 (만 원)", 1e-4, "{:,.0f}"), ("hit_rate", "q80 적중률 (%)", 100, "{:.1f}"), ("coverage", "80% 포함률 (%)", 100, "{:.1f}"), ("mean_interval_width", "평균 구간 길이 (명)", 1, "{:,.0f}"), ("shortage_person_days", "부족 인원 (인일)", 1, "{:,.0f}"))
    short_sc = ["전체", "궁", "종묘", "왕릉", "5/1\n하루"]
    for ax, (column, title, scale, fmt) in zip(axes.ravel(), metrics):
        x = np.arange(len(sc))
        a = [float(ABLATION.loc[(ABLATION["scope"] == s) & (ABLATION["model_id"] == ABL_ID), column].iloc[0]) * scale for s in sc]
        b = [float(ABLATION.loc[(ABLATION["scope"] == s) & (ABLATION["model_id"] == B_ID), column].iloc[0]) * scale for s in sc]
        ax.bar(x - 0.19, a, 0.36, color=GREY, label="A 근로자의 날 제외")
        ax.bar(x + 0.19, b, 0.36, color=GREEN, label="B 근로자의 날 포함")
        top = max(max(a), max(b))
        for xi, (va, vb) in enumerate(zip(a, b)):
            ax.text(xi - 0.19, va + top * 0.012, fmt.format(va), ha="center", fontsize=8.5)
            ax.text(xi + 0.19, vb + top * 0.012, fmt.format(vb), ha="center", fontsize=8.5)
        ax.set_xticks(x)
        ax.set_xticklabels(short_sc, fontsize=10)
        ax.set_title(title, loc="left", fontsize=13)
        style_axes(ax, "y")
    axes[0, 0].legend(loc="upper left", fontsize=10)
    return save_fig(fig, "11_labor_day_ablation.png")


def fig12():
    g = GOLDEN_T.copy()
    g["label"] = g["scenario"].replace({"기본 q80": "기본\nq80", "황금연휴 q80 + 1명": "q80\n+1명", "황금연휴 q80 + 2명": "q80\n+2명", "황금연휴 q90": "q90"}) + np.where(g["scenario"] == "기본 q80", "", "\n(" + g["apply_to"] + ")")
    fig, axes = new_fig("황금연휴 추가 인력: 부족 위험을 줄일수록 잉여 비용이 늘어나는 trade-off", f"2025 순차 백테스트의 황금연휴·설추석 명절 개방 행 {int(g['n_site_days'].iloc[0])}개소·일. 서비스 수준 = 부족이 없는 개소·일 비율. ★ = 운영 가능 범위(추가 ≤ 기본의 {FEASIBLE_EXTRA_RATIO:.0%}) 안 최소비용 권고", 1, 2, gridspec_kw=dict(wspace=0.25))
    fig.subplots_adjust(top=0.83, bottom=0.2, left=0.06, right=0.98)
    x = np.arange(len(g))
    ax = axes[0]
    ax.bar(x, g["shortage_cost_krw"] / 1e6, color=VERM, label="부족 비용 (백만 원)")
    ax.bar(x, g["surplus_cost_krw"] / 1e6, bottom=g["shortage_cost_krw"] / 1e6, color=GREEN_L, label="잉여 비용 (백만 원)")
    for xi, (_, r) in zip(x, g.iterrows()):
        ax.text(xi, r["total_cost_krw"] / 1e6 + g["total_cost_krw"].max() / 1e6 * 0.01, f"{r['total_cost_krw'] / 1e6:,.0f}" + (" ★" if r["is_recommended"] else ""), ha="center", fontsize=11, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels(g["label"], fontsize=10.5)
    ax.set_title("총비용 (백만 원)", loc="left", fontsize=14)
    ax.legend(loc="upper right")
    style_axes(ax, "y")
    ax = axes[1]
    ax.scatter(g["extra_person_days_vs_base"], g["service_level_no_shortage"] * 100, s=150, c=[GREEN if r else BLUE for r in g["operationally_feasible"]], zorder=3)
    for _, r in g.iterrows():
        ax.annotate(r["scenario"].replace("황금연휴 ", "") + ("" if r["scenario"] == "기본 q80" else f" ({r['apply_to']})"), (r["extra_person_days_vs_base"], r["service_level_no_shortage"] * 100), textcoords="offset points", xytext=(6, 5), fontsize=9.5)
    ax.set_xlim(-2, g["extra_person_days_vs_base"].max() * 1.35 + 3)
    ax.set_xlabel("추가 배치 인원 (인일, 기본 q80 대비)")
    ax.set_ylabel("서비스 수준 (부족 없는 개소·일 %)")
    ax.set_title("추가 인력 대비 서비스 수준 (초록 = 운영 가능 범위)", loc="left", fontsize=14)
    style_axes(ax, "both")
    return save_fig(fig, "12_golden_holiday_scenarios.png")


def fig13():
    d = F15.copy()
    d["x"] = pd.to_datetime(d["date"])
    fig, ax = new_fig(f"2026-01-01~14 예상 방문객 {int(d['predicted_total_point'].sum()):,}명 (q80 합계 {int(d['predicted_total_q80'].sum()):,}명)", "전체 개소 합계(개방 개소만), 최종 권고 모델. 2026년 실제 기상은 사용하지 않음. 빨간 배경 = 공휴일, 회색 배경 = 주말", 1, 1)
    fig.subplots_adjust(top=0.84, bottom=0.14, left=0.08, right=0.97)
    for _, r in d.iterrows():
        if r["holiday_name"]:
            ax.axvspan(r["x"] - pd.Timedelta(hours=12), r["x"] + pd.Timedelta(hours=12), color="#F4C7C3", alpha=0.8, lw=0)
            ax.text(r["x"], d["predicted_total_q80"].max() * 1.04, r["holiday_name"], ha="center", fontsize=11, color=VERM, fontweight="bold")
        elif r["x"].dayofweek >= 5:
            ax.axvspan(r["x"] - pd.Timedelta(hours=12), r["x"] + pd.Timedelta(hours=12), color="#EEEEEE", alpha=0.9, lw=0)
    ax.plot(d["x"], d["predicted_total_point"], color=GREEN, marker="o", linewidth=2.6, label="점 예측")
    ax.plot(d["x"], d["predicted_total_q80"], color=VERM, marker="s", linewidth=2.0, linestyle="--", label="q80 (인력 산정 기준)")
    for _, r in d.iterrows():
        ax.text(r["x"], r["predicted_total_point"] - d["predicted_total_q80"].max() * 0.045, f"{int(r['predicted_total_point']):,}", ha="center", fontsize=10, color=GREEN)
    ax.set_xticks(d["x"])
    ax.set_xticklabels([f"{x:%m-%d}\n({w})" for x, w in zip(d["x"], d["weekday"])], fontsize=11)
    ax.set_ylabel("일 방문객 (명, 개방 개소 합계)")
    ax.set_ylim(0, d["predicted_total_q80"].max() * 1.12)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:,.0f}"))
    ax.legend(loc="lower right")
    style_axes(ax, "y")
    return save_fig(fig, "13_forecast_2026_visitors.png")


def fig14():
    d = F15.copy()
    x = np.arange(len(d))
    fig, ax = new_fig(f"2026-01-01~14 권고 인력 합계 {int(d['final_operational_staff'].sum())} person-days: 궁이 {d['palace_staff'].sum() / d['final_operational_staff'].sum():.0%}", "날짜별 전체 권고 인원(명/일), 궁·종묘·왕릉 누적 막대. 실제 근무표 인원이 아니라 q80 기준 환산 필요 인원. 숫자 = 당일 총 권고 인원, 아래 = 당일 고수요 개소", 1, 1)
    fig.subplots_adjust(top=0.84, bottom=0.2, left=0.07, right=0.98)
    ax.bar(x, d["palace_staff"], color=GREEN, label="궁")
    ax.bar(x, d["jongmyo_staff"], bottom=d["palace_staff"], color=YELLOW, label="종묘")
    ax.bar(x, d["royal_tomb_staff"], bottom=d["palace_staff"] + d["jongmyo_staff"], color=SKY, label="왕릉")
    for xi, v in zip(x, d["final_operational_staff"]):
        ax.text(xi, v + 0.6, f"{int(v)}", ha="center", fontsize=12, fontweight="bold")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{pd.Timestamp(dt):%m-%d}\n({w})\n{(h or '')}" for dt, w, h in zip(d["date"], d["weekday"], d["holiday_name"])], fontsize=10)
    for xi, s in zip(x, d["high_demand_top5_sites"].fillna("")):
        first = s.split(", ")[0] if s else ""
        ax.text(xi, -max(d["final_operational_staff"]) * 0.2, first.replace("(", "\n("), ha="center", fontsize=9.5, color="#555555")
    ax.set_ylabel("권고 인원 (명/일)")
    ax.set_ylim(0, d["final_operational_staff"].max() * 1.12)
    ax.legend(loc="upper right", ncol=3)
    style_axes(ax, "y")
    return save_fig(fig, "14_staffing_2026_daily.png")


def fig15():
    order = [s for g in GROUPS for s in sorted(F14.loc[F14["group"] == g, "site"].unique())]
    pivot = F14.pivot(index="site", columns="date", values="final_operational_staff").reindex(order)
    closed = F14.pivot(index="site", columns="date", values="is_regular_closed").reindex(order).astype(bool)
    palace_max = int(F14.loc[F14["group"] == "궁", "final_operational_staff"].max())
    tomb_open = F14.loc[(F14["group"] == "왕릉") & ~F14["is_regular_closed"]]
    tomb_one_share = float((tomb_open["final_operational_staff"] == 1).mean())
    fig, ax = new_fig(f"2026-01-01~14 개소별 운영 배치: 궁은 하루 최대 {palace_max}명, 왕릉은 개방일의 {tomb_one_share:.0%}가 1명", "행 = 개소(궁→종묘→왕릉), 열 = 날짜, 값 = 최종 권고 인원(명). 회색 = 정기 휴관(0명). 빨간 테두리 = 왕릉 중 1명 초과 필요(q80 2,000명 초과), 주황 글씨 = 공휴일", 1, 1)
    fig.subplots_adjust(top=0.85, bottom=0.05, left=0.12, right=0.99)
    masked = np.ma.masked_where(closed.to_numpy(), pivot.to_numpy())
    cmap = plt.get_cmap("YlGn").copy()
    cmap.set_bad("#E3E3E3")
    ax.imshow(masked, cmap=cmap, aspect="auto", vmin=0, vmax=max(2, int(np.nanmax(pivot.to_numpy()))) * 1.1)
    ax.grid(False)
    holiday_dates = set(F14.loc[F14["is_holiday"] == 1, "date"])
    for i, site in enumerate(order):
        for j, date in enumerate(pivot.columns):
            if closed.loc[site, date]:
                ax.text(j, i, "휴관", ha="center", va="center", fontsize=9, color="#777777")
                continue
            v = int(pivot.loc[site, date])
            ax.text(j, i, str(v), ha="center", va="center", fontsize=12, fontweight="bold", color="white" if v >= 6 else DARK)
            if SITE_GROUP[site] == "왕릉" and v > 1:
                ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, edgecolor=VERM, linewidth=3))
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{pd.Timestamp(d):%m-%d}\n({F14.loc[F14['date'] == d, 'weekday'].iloc[0]})" for d in pivot.columns], fontsize=10.5)
    for tick, d in zip(ax.get_xticklabels(), pivot.columns):
        if d in holiday_dates:
            tick.set_color("#D55E00")
            tick.set_fontweight("bold")
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([f"{s} ({SITE_GROUP[s]})" for s in order], fontsize=10.5)
    ax.xaxis.tick_top()
    edge = 0
    for g in GROUPS[:-1]:
        edge += sum(SITE_GROUP[s] == g for s in order)
        ax.axhline(edge - 0.5, color=DARK, linewidth=1.8)
    for spine in ax.spines.values():
        spine.set_visible(False)
    return save_fig(fig, "15_staffing_2026_site_heatmap.png")


def fig16():
    o = OVERALL.set_index("model_id").loc[FINAL_MODEL]
    q = FVA_Q80.loc[FINAL_MODEL]
    fig = plt.figure(figsize=(16, 9), facecolor=BG)
    fig.suptitle("경영진 핵심 요약: 최종 모델과 2026년 14일 운영 권고", fontsize=21, fontweight="bold", x=0.015, ha="left", y=0.975, color=DARK)
    fig.text(0.015, 0.915, f"{REG[FINAL_MODEL]['label']} | {BACKTEST_NOTE} | 인력은 실제 근무표가 아니라 q80 기준 환산 필요 인원", fontsize=12, color="#555555")
    cards = [("WAPE", f"{o['WAPE']:.1f}%", f"직전 7일 평균 {OVERALL.set_index('model_id').loc['baseline_mean7', 'WAPE']:.1f}%"), ("q80 적중률", f"{o['hit_rate']:.1%}", "목표 80%"), ("80% 구간 포함률", f"{o['coverage']:.1%}", f"평균 구간 {o['mean_interval_width']:,.0f}명"),
             ("기준선 대비 절감", f"{q['saving_vs_mean7_krw'] / 1e8:.2f}억 원", f"{q['saving_rate_vs_mean7']:.0%} (총비용 {q['total_cost_krw'] / 1e8:.2f}억)"), ("2026 14일 권고", f"{int(F14['final_operational_staff'].sum()):,}", "person-days (q80 기준)"),
             ("핵심 위험", "공휴일·황금연휴", f"부족 인원의 {GOLDEN_SHORTAGE_SHARE:.0%}가 연휴·명절")]
    for i, (title, value, sub) in enumerate(cards):
        col, row = i % 3, i // 3
        ax = fig.add_axes([0.02 + col * 0.325, 0.54 - row * 0.32, 0.3, 0.28])
        ax.set_facecolor("#FFFFFF")
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_color(GREEN if i != 5 else VERM)
            spine.set_linewidth(2.5)
        ax.text(0.5, 0.78, title, ha="center", va="center", fontsize=17, color="#555555", transform=ax.transAxes)
        ax.text(0.5, 0.45, value, ha="center", va="center", fontsize=34 if len(value) < 9 else 28, fontweight="bold", color=GREEN if i != 5 else VERM, transform=ax.transAxes)
        ax.text(0.5, 0.15, sub, ha="center", va="center", fontsize=13, color="#555555", transform=ax.transAxes)
    fig.text(0.015, 0.115, f"운영 행동: ① 궁·종묘는 q80 기반 정밀 배치 ② 왕릉은 개방일 최소 1명, q80 2,000명 초과 시 증원 ③ 황금연휴·명절·무료개방일은 {GOLDEN_ADD['scenario']}({GOLDEN_ADD['apply_to']}) 검토, 근로자의 날은 특수 수요일로 관리", fontsize=12.5, color=DARK)
    fig.text(0.015, 0.05, SOURCE, fontsize=9, color="#777777")
    path = FIG_DIR / "16_executive_summary_infographic.png"
    fig.savefig(path, dpi=300, facecolor=BG)
    plt.close(fig)
    FIG_FILES.append("16_executive_summary_infographic.png")
    return path


def make_all_figures():
    order = OVERALL.sort_values("WAPE")["model_id"].tolist()
    for fn in (lambda: fig01(order), lambda: fig02(order), fig03, fig04, fig05, fig06, fig07, fig08, fig09, fig10, fig11, fig12, fig13, fig14, fig15, fig16):
        fn()
    return FIG_FILES

In [ ]:
FIG_FILES = make_all_figures()
mark("그림 16개 생성")
print("생성한 그림:", len(FIG_FILES), FIG_FILES)

## 16. 검산 (PASS/FAIL)
그림·표를 만든 뒤 저장 전에 자동 검산 항목(원점 이후 실제값 미사용, 강수 누수, 근로자의 날 생성 규칙, 무료개방, 합계 정합성, 인력 규칙, 비용 계산, 2026 행 구성, 그림 생성 등)을 실행합니다. FAIL이 하나라도 있으면 저장·완료 처리를 하지 않고 중단합니다. 저장 후에는 출력 파일을 다시 읽어 확인한 항목을 추가해 `17_validation_checks.csv`에 최종 기록합니다.

In [ ]:
def tamper_future(frames, origin):
    return {site: frame.assign(target=np.where(frame.index > origin, 1e7, frame["target"])) for site, frame in frames.items()}


def leakage_checks():
    """origin 이후 목표값(및 결측)을 극단값으로 바꿔도 예측이 변하지 않는지 확인하는 변조 실험 (Chronos-2 대표 모델, 최종 모델이 Chronos면 포함, 베이스라인)."""
    window = TEST_WINDOWS.iloc[[0]]
    origin = window["origin"].iloc[0]
    results = {}
    models = ["chronos_zero", "chronos_holiday_weather_free"] + ([FINAL_MODEL] if REG[FINAL_MODEL]["kind"] == "chronos" and FINAL_MODEL not in ("chronos_zero", "chronos_holiday_weather_free") else [])
    for model_id in models:
        for target in TARGETS:
            frames = build_target_frames(target, "test")
            a = run_chronos(get_pipeline(), REG[model_id]["covariates"], window, frames)
            b = run_chronos(get_pipeline(), REG[model_id]["covariates"], window, tamper_future(frames, origin))
            results[f"{model_id}|{target}"] = bool(np.allclose(a[QUANTILE_COLUMNS].to_numpy(), b[QUANTILE_COLUMNS].to_numpy(), rtol=1e-5))
    saved = {t: m.copy() for t, m in MATRICES.items()}
    try:
        for t in MATRICES:
            changed = saved[t].copy()
            changed.loc[changed.index > origin, :] = 1e7
            MATRICES[t] = changed
        tampered = forecast_baselines(window)
    finally:
        MATRICES.update(saved)
    original = BASELINE_PRED.loc[BASELINE_PRED["fold"] == window["fold"].iloc[0]].sort_values(["model_id", "target", "site", "horizon"])
    results["baselines"] = bool(np.allclose(original["prediction"].to_numpy(), tampered.sort_values(["model_id", "target", "site", "horizon"])["prediction"].to_numpy()))
    return results


def input_hash_now():
    return hashlib.md5("".join(hashlib.md5(p.read_bytes()).hexdigest() for p in INPUT_FILES if p.exists()).encode()).hexdigest()


def run_validations():
    info = read_info()
    checks = []

    def add(name, passed, detail=""):
        checks.append(dict(check=name, result="PASS" if bool(passed) else "FAIL", detail=str(detail)))

    leak = leakage_checks()
    add("예측에 원점 이후 실제값을 사용하지 않았는가 (원점 이후 목표값 변조 실험)", all(leak.values()), leak)
    # 강수: 2025 실측 미사용, 2026 실측 미사용
    tampered_daily = {site: frame.assign(**{c: np.where(frame.index >= TEST_START, 1, frame[c]) for c in WEATHER_BASES}) for site, frame in WEATHER_DAILY.items()}
    prob_tampered = monthly_probabilities(tampered_daily, TRAIN_END)
    same_test = all(np.allclose(prob_tampered[s][[f"{b}_prob" for b in WEATHER_BASES]].to_numpy(), COVARIATES["test"][s][WEATHER_FEATURES].to_numpy()) for s in SITES)
    site0 = SITES[0]
    used = WEATHER_DAILY[site0].loc[HISTORY_START:TRAIN_END]
    spot = used.loc[(used.index.month == 3) & (used.index.dayofweek == 2), WEATHER_BASES].mean().to_numpy()
    spot_ok = np.allclose(COVARIATES["test"][site0].loc["2025-03-12", WEATHER_FEATURES].to_numpy(dtype=float), spot)
    add("2025년 강수 실측값이 본 모델 미래 공변량에 들어가지 않았는가 (2025 실측을 모두 1로 바꿔도 테스트용 확률 불변, 2023~2024 월×요일 평균과 일치)", same_test and spot_ok, f"2025 변조 후 동일 {same_test}, 2025-03-12({site0}) 독립 재계산 일치 {spot_ok}")
    prob_final = monthly_probabilities(WEATHER_DAILY, FINAL_TRAIN_END)
    final_equal = all(np.allclose(prob_final[s][[f"{b}_prob" for b in WEATHER_BASES]].to_numpy(), COVARIATES["final"][s][WEATHER_FEATURES].to_numpy()) for s in SITES)
    add("2026년 실제 날씨(관측값)를 사용하지 않았는가 (날씨 자료 마지막 날짜 ≤ 2025-12-31, 2026 확률 = 2023~2025 기후 확률)", WEATHER["date"].max() <= pd.Timestamp("2025-12-31") and final_equal and WEATHER_DAILY[SITES[0]].index.max() <= TEST_END,
        f"날씨 자료 {WEATHER['date'].min().date()}~{WEATHER['date'].max().date()}, 2026 확률 일치 {final_equal}")
    consistency = all(np.allclose(COVARIATES[m][s]["rain_prob"].to_numpy(), (COVARIATES[m][s]["rain_1_10_prob"] + COVARIATES[m][s]["heavy_prob"]).to_numpy()) for m in ("test", "final") for s in SITES)
    backoff_text = {str(k.date()): v.sum().to_dict() for k, v in WEATHER_BACKOFF.items()}
    add("강수 묶음의 구간 정의 일관성과 백오프 단계 기록 (rain = rain_1_10 + heavy, 월×요일→월→계절→전체)", consistency and len(WEATHER_BACKOFF) >= 2 and all(float(v.to_numpy().sum()) > 0 for v in WEATHER_BACKOFF.values()), f"백오프 단계별 개소·일 수(보정 기준일별): {backoff_text}")
    # 근로자의 날·국군의 날
    labor_dates = CALENDAR.index[CALENDAR["is_labor_day"] == 1]
    gen = pd.Series(CALENDAR_2026["date"].dt.strftime("%m-%d").eq("05-01").astype(int).to_numpy(), index=CALENDAR_2026["date"])
    row_2026 = CALENDAR_2026.loc[CALENDAR_2026["date"] == pd.Timestamp("2026-05-01")].iloc[0]
    add("is_labor_day가 모든 연도의 5월 1일에만 생성되었는가 (2026 달력은 원본에 열이 없어도 날짜로 생성, 2026-05-01은 is_holiday=1 & is_labor_day=1)",
        all(d.strftime("%m-%d") == "05-01" for d in labor_dates) and len(labor_dates) >= 3 and int(gen.loc["2026-05-01"]) == 1 and int(gen.sum()) == 1 and int(row_2026["is_holiday"]) == 1 and (CALENDAR["is_labor_day"].to_numpy() == (CALENDAR.index.strftime("%m-%d") == "05-01")).all(),
        f"2023~2026-01-14 달력의 근로자의 날 {[d.strftime('%Y-%m-%d') for d in labor_dates]}, 2026 달력 생성 {int(gen.sum())}일(2026-05-01 is_holiday={int(row_2026['is_holiday'])})")
    audit = {m: {t: info[m]["covariate_audit"][t] for t in TARGETS} for m in CHRONOS_MODELS + [ABL_ID] if m in info}
    used_cols = set(sum([audit[m][t]["context_columns"] + audit[m][t]["future_columns"] for m in audit for t in TARGETS], []))
    add("국군의 날 전용 피처가 없는가 (모델 입력·공변량·MLP 피처 어디에도 없음)", not any("armed" in c.lower() or "military" in c.lower() for c in used_cols | set(ALL_COVARIATES) | set(MLP_FEATURES) | set(COVARIATES["final"][SITES[0]].columns)), f"Chronos 입력 열 {sorted(used_cols)}")
    block_models = [m for m in CHRONOS_MODELS if "is_holiday" in REG[m]["covariates"]]
    weather_models = [m for m in CHRONOS_MODELS if set(WEATHER_FEATURES) & set(REG[m]["covariates"])]
    ok_block = all(set(HOLIDAY_FEATURES) <= set(audit[m][t]["context_columns"]) and set(HOLIDAY_FEATURES) <= set(audit[m][t]["future_columns"]) for m in block_models for t in TARGETS)
    ok_weather = all(set(WEATHER_FEATURES) <= set(audit[m][t]["context_columns"]) and set(WEATHER_FEATURES) <= set(audit[m][t]["future_columns"]) for m in weather_models for t in TARGETS) and all(not (set(WEATHER_FEATURES) & set(audit[m][t]["context_columns"])) for m in CHRONOS_MODELS if m not in weather_models for t in TARGETS)
    add("휴일 묶음(is_holiday + is_labor_day)과 강수 묶음(4열)이 해당 Chronos 모델의 context·future 입력에 모두 들어갔는가", ok_block and ok_weather and len(block_models) == 4 and len(weather_models) == 4 and HOLIDAY_FEATURES == ["is_holiday", "is_labor_day"],
        f"휴일 묶음 모델 {len(block_models)}개, 강수 묶음 모델 {len(weather_models)}개, 강수 열 {WEATHER_FEATURES}")
    start, end = FESTIVAL["start"], FESTIVAL["end"]
    open_flag = VISITORS_ALL.loc[VISITORS_ALL["date"].between(start, end) & VISITORS_ALL["site"].isin(SITES)].assign(expected=lambda d: d["total"].notna().astype(int))
    got = [int(FREE_OPEN[s].loc[d]) for s, d in zip(open_flag["site"], open_flag["date"])]
    add("2025-10-29~2025-11-09 무료개방이 반영되었는가 (개방 개소는 free_open=1, 정기 휴관은 0)", got == open_flag["expected"].tolist(), f"{len(open_flag)}개소·일, 개방 {int(open_flag['expected'].sum())}행 = 1, 휴관 {int((1 - open_flag['expected']).sum())}행 = 0, 데이터 수정 {DATA_CHECK['fixed_rows']}행")
    piv = ALL_PRED.pivot_table(index=["model_id", "phase", "fold", "date", "site"], columns="target", values="prediction")
    add("total과 domestic+foreign의 정합성이 유지되는가 (2025 백테스트 전 모델 + 2026 예측)", np.allclose(piv["total"], piv["domestic"] + piv["foreign_total"]) and bool((F14["predicted_domestic"] + F14["predicted_foreign"] == F14["predicted_total_point"]).all()),
        f"백테스트 {len(piv):,}행, 2026 {len(F14)}행 (정수 반올림 후 외국인 = 총 − 내국인으로 일치)")
    closed26 = F14.loc[F14["is_regular_closed"]]
    closed_eval = EVAL_BASE.loc[(EVAL_BASE["model_id"] == FINAL_MODEL) & (EVAL_BASE["target"] == "total") & EVAL_BASE["is_closed"]]
    add("휴관일 인력이 0명인가", bool((closed26["final_operational_staff"] == 0).all()) and bool((staff_rule(to_visitors_int(closed_eval["q80_equiv"]), np.ones(len(closed_eval), dtype=bool)) == 0).all()), f"2026 휴관 {len(closed26)}개소·일, 2025 휴관 예측 {len(closed_eval)}행 모두 0명")
    open26 = F14.loc[~F14["is_regular_closed"]]
    add("개방일 최소 인력이 1명인가", bool((open26["final_operational_staff"] >= 1).all()) and bool((EVAL_STAFF["staff_q80"] >= 1).all()) and bool((EVAL_STAFF["staff_point"] >= 1).all()), f"2026 개방 {len(open26)}개소·일, 2025 평가 {len(EVAL_STAFF):,}행")
    add("방문객 3,000명의 필요 인원이 2명으로 계산되는가", int(staff_rule(np.array([3000]), np.array([False]))[0]) == 2 and (staff_rule(np.array([2000, 2001, 4000, 4001]), np.zeros(4, dtype=bool)) == np.array([1, 2, 2, 3])).all(), "3,000명 → 2명, 2,000 → 1, 2,001 → 2, 4,000 → 2, 4,001 → 3")
    add("부족·잉여 인원이 동시에 양수가 아닌가", not bool(((EVAL_STAFF["shortage_q80"] > 0) & (EVAL_STAFF["surplus_q80"] > 0)).any() or ((EVAL_STAFF["shortage_point"] > 0) & (EVAL_STAFF["surplus_point"] > 0)).any()), f"평가 {len(EVAL_STAFF):,}행")
    fin_staff = EVAL_STAFF.loc[EVAL_STAFF["model_id"] == FINAL_MODEL]
    cost_ok = bool(np.allclose(fin_staff["shortage_cost_q80"], fin_staff["shortage_q80"] * SHORTAGE_COST_KRW) and np.allclose(fin_staff["surplus_cost_q80"], fin_staff["surplus_q80"] * OVERSTAFF_COST_KRW)
                   and np.isclose(fin_staff["total_cost_q80"].sum(), FVA_Q80.loc[FINAL_MODEL, "total_cost_krw"]))
    add("부족 비용·잉여 비용 계산이 정확한가 (부족 인원 × 600,000원, 잉여 인원 × 150,000원, 표 07과 일치)", cost_ok, f"최종 모델 총비용 {fin_staff['total_cost_q80'].sum():,.0f}원")
    add("임계비율이 0.8인가", abs(CRITICAL_RATIO - 0.8) < 1e-12 and SHORTAGE_COST_KRW == 600_000 and OVERSTAFF_COST_KRW == 150_000, f"600,000 / (600,000 + 150,000) = {CRITICAL_RATIO:.4f}")
    site_sum = SITE_T.loc[SITE_T["model_id"] == FINAL_MODEL, ["n_observations", "shortage_person_days", "surplus_person_days", "total_cost_krw"]].sum()
    all_row = OVERALL.set_index("model_id").loc[FINAL_MODEL]
    add("개소별 합계와 전체 합계가 일치하는가 (2025 최종 모델, 2026 개소별 합계)", bool(np.isclose(site_sum["n_observations"], all_row["n_observations"]) and np.isclose(site_sum["shortage_person_days"], all_row["shortage_person_days"]) and np.isclose(site_sum["total_cost_krw"], all_row["total_cost_krw"]))
        and int(F16.loc[F16["row_type"] == "개소", "person_days_final"].sum()) == int(F14["final_operational_staff"].sum()) == int(F16.loc[F16["row_type"] == "전체", "person_days_final"].iloc[0]),
        f"2025 개소 합 n={int(site_sum['n_observations']):,}, 부족 {site_sum['shortage_person_days']:.0f}인일, 2026 person-days {int(F14['final_operational_staff'].sum())}")
    dates26 = sorted(F14["date"].unique())
    add("2026년 예측 날짜가 정확히 14일인가", dates26 == [d.strftime("%Y-%m-%d") for d in pd.date_range(FORECAST_START, FORECAST_END)] and len(dates26) == 14, f"{dates26[0]}~{dates26[-1]} {len(dates26)}일")
    add("23개소가 모두 포함되었는가", set(F14["site"]) == set(SITES) and F14["site"].nunique() == len(SITES) and (FAST_DEBUG or len(SITES) == 23), f"2026 개소 {F14['site'].nunique()}곳, 2025 평가 개소 {OVERALL['n_observations'].sum() > 0 and EVAL_BASE['site'].nunique()}곳")
    dup_report = {"2026 예측": int(F13.duplicated(["date", "site"]).sum()), "2026 배치표": int(F14.duplicated(["date", "site"]).sum()), "방문객": int(VISITORS_ALL.duplicated(["date", "site"]).sum()), "평가": int(EVAL.duplicated(["model_id", "target", "date", "site"]).sum())}
    add("개소별 예측 행이 중복되지 않는가 (date + site)", sum(dup_report.values()) == 0 and len(F14) == len(SITES) * 14, dup_report)
    fin_eval = EVAL_BASE.loc[(EVAL_BASE["model_id"] == FINAL_MODEL) & (EVAL_BASE["target"] == "total") & ~EVAL_BASE["is_closed"]]
    low2025 = fin_eval.loc[fin_eval["q80_equiv"] < fin_eval["prediction"] - 1e-9]
    low2026 = F13.loc[(F13["predicted_total_q80"] < F13["predicted_total_point"] - 1e-6) & ~F13["is_regular_closed"]]
    cause_ok = bool((low2025["s_up"] < 0).all())
    add("q80이 point보다 작은 사례가 있다면 원인과 보정 여부가 기록되는가", cause_ok, f"2025 평가 {len(low2025)}행({len(low2025) / max(len(fin_eval), 1):.2%}), 2026 {len(low2026)}행. 원인 = 해당 개소 보정 오차의 q80 분위수가 음수(과거 과대예측 경향)여서 q80 = max(예측 + 분위수, 0) < 예측. 보정하지 않고 conformal 값을 그대로 유지(기존 비용 검산 기준과의 일관성)")
    tables_with_cols = {"01": OVERALL, "02": GROUP_T, "03": SITE_T, "04": ORIGIN_T, "05": DATE_TYPE_T, "06": CROSS_T}
    needed = {"coverage", "mean_interval_width", "hit_rate", "WAPE", "RelMAE"}
    add("포함률과 구간 길이가 최종 표와 그림에 모두 존재하는가", all(needed <= set(t.columns) for t in tables_with_cols.values()) and (FIG_DIR / "02_probabilistic_hit_coverage_width.png").exists() and (FIG_DIR / "16_executive_summary_infographic.png").exists() and (FIG_DIR / "05_site_heatmap.png").exists(),
        f"표 01~06 모두 coverage·mean_interval_width 포함, 그림 02(세 지표)·05(개소별)·16 생성")
    from PIL import Image
    sizes = {f: Image.open(FIG_DIR / f).size for f in FIG_FILES}
    add("모든 그림 파일이 정상 생성되었는가 (16개, PNG, 16:9, 300dpi 이상)", len(FIG_FILES) == 16 and all((FIG_DIR / f).stat().st_size > 50_000 for f in FIG_FILES) and all(w >= 4800 and abs(w / h - 16 / 9) < 0.02 for w, h in sizes.values()), f"{len(FIG_FILES)}개, 해상도 예 {next(iter(sizes.values()))}")
    # 추가 검산
    add("비교 모델 행 수가 자동 계산되고 모든 행의 WAPE가 계산됨 (실제 14개)", N_MODEL_ROWS == 14 and OVERALL["WAPE"].notna().all() and set(OVERALL["model_id"]) == set(MODELS), f"모델 행 {N_MODEL_ROWS}개, 롤링 원점 {len(TEST_WINDOWS)}개")
    add("2025 순차 rolling-origin 백테스트 설계 (초기 학습 2023~2024, 14일, 원점 < 예측 시작, 약 27개 원점)", (FAST_DEBUG or len(TEST_WINDOWS) == 27) and bool((TEST_WINDOWS["origin"] < TEST_WINDOWS["test_start"]).all()) and TRAIN_END == pd.Timestamp("2024-12-31") and HORIZON == 14, f"원점 {len(TEST_WINDOWS)}개, 첫 원점 {TEST_WINDOWS['origin'].min().date()}, 마지막 예측 종료 {TEST_WINDOWS['test_end'].max().date()}")
    add("q80 보정이 2023~2024년 내부 백테스트로만 이루어짐 (2025 오차를 2026 보정에 쓰지 않음)", CAL_TEST["date"].max() <= TRAIN_END and CAL_TEST["date"].min() >= pd.Timestamp("2024-07-03"), f"보정 {CAL_TEST['date'].min().date()}~{CAL_TEST['date'].max().date()}")
    add("원본·전처리 입력 파일이 수정되지 않음 (실행 전후 파일 해시 동일)", input_hash_now() == INPUT_HASH, f"입력 해시 {INPUT_HASH[:12]}…")
    repro_ok = FAST_DEBUG or (all(abs(v["previous_wape"] - v["new_wape"]) < 0.05 for v in ABL_REPRO.values()) if ABL_REPRO else True)      # 디버그(3개 개소·일부 구간)는 기존 결과와 비교할 수 없음
    add("근로자의 날 ablation이 기존 outputs/labor_day_ablation 결과와 재현됨 (WAPE 차 < 0.05%p)", repro_ok, {k: {a: round(b, 3) for a, b in v.items()} for k, v in ABL_REPRO.items()} or "기존 결과 파일 없음")
    add("시드가 고정되어 있음 (SEED=42, MLP 시드 0~29, torch·numpy 시드)", SEED == 42 and MLP_SEEDS[0] == 0 and len(MLP_SEED_PRED["seed"].unique()) == len(MLP_SEEDS), f"SEED {SEED}, MLP 시드 {len(MLP_SEEDS)}개")
    add("2026 최종 예측에 2023~2025 전체 이력을 사용함 (MLP 학습 종료 2025-12-31, ARIMA·SARIMA 차수 선택 종료 2025-12-31, Chronos 컨텍스트 전체)",
        info["mlp_global"]["training"]["domestic|final"]["train_end"] == "2025-12-31" and all((o["cutoff"] == "2025-12-31").all() for o in FINAL_ORDERS.values()) and FINAL_WINDOW["origin"].iloc[0] == FINAL_TRAIN_END, f"최종 모델 {FINAL_MODEL}")
    add("기존 결과와의 차이가 기록됨 (직전 7일 평균 q80 비용, 기존 최종 후보 비용)", True, f"직전 7일 평균 {base_cost:,.0f}원 (기존 {REF_MEAN7:,}원, 차 {base_cost - REF_MEAN7:+,.0f}), 최종 권고 {final_cost:,.0f}원 (기존 {REF_PREV_FINAL:,}원, 차 {final_cost - REF_PREV_FINAL:+,.0f})")
    table = pd.DataFrame(checks)
    table.insert(0, "check_id", np.arange(1, len(table) + 1))
    return table


VALIDATION_PRE = run_validations()
display(VALIDATION_PRE[["check_id", "check", "result"]])
n_fail = int((VALIDATION_PRE["result"] == "FAIL").sum())
print(f"검산 {len(VALIDATION_PRE)}개 중 PASS {int((VALIDATION_PRE['result'] == 'PASS').sum())}, FAIL {n_fail}")
assert n_fail == 0, "FAIL 검산이 있어 최종 완료 처리하지 않고 중단합니다. 위 표의 detail을 확인해 원인을 수정하세요."
mark("검산")

## 17. 결과 저장 (표 · 운영 배치표 · 요약 문서 · 로그)
`outputs/final_operational_delivery/` 아래 `figures/`(그림 16개), `tables/`(분석표 `01~18` + 보조표), `staffing/`(운영팀 전달 CSV·Excel), `summary/`(요약, 모델 카드, 배치 가이드, 검증 결과), `logs/`(환경·실행 시간·재현 정보)에 저장합니다.

In [ ]:
def eok(v):
    return f"{v / 1e8:,.2f}억 원"


def won_text(v):
    return f"{v:,.0f}원 ({v / 1e8:,.2f}억 원)"


O = OVERALL.set_index("model_id")
fin_o, base_o = O.loc[FINAL_MODEL], O.loc["baseline_mean7"]
fin_q, base_q = FVA_Q80.loc[FINAL_MODEL], FVA_Q80.loc["baseline_mean7"]
golden_share_text = f"{GOLDEN_SHORTAGE_SHARE:.1%}"
TOTAL_PD = int(F14["final_operational_staff"].sum())
PD_GROUP = {g: int(F14.loc[F14["group"] == g, "final_operational_staff"].sum()) for g in GROUPS}
label_final = REG[FINAL_MODEL]["label"]


def md_table(frame, digits=2):
    head = "| " + " | ".join(str(c) for c in frame.columns) + " |\n|" + "---|" * len(frame.columns) + "\n"

    def fmt(v):
        if isinstance(v, (float, np.floating)):
            return "-" if np.isnan(v) else f"{v:,.{digits}f}"
        if isinstance(v, (int, np.integer)) and not isinstance(v, (bool, np.bool_)):
            return f"{int(v):,}"
        return str(v)
    return head + "\n".join("| " + " | ".join(fmt(v) for v in row) + " |" for row in frame.itertuples(index=False))


def build_manifest():
    d = DATE_TYPE_T.loc[DATE_TYPE_T["model_id"] == FINAL_MODEL].set_index("date_type")
    g = GROUP_T.loc[GROUP_T["model_id"] == FINAL_MODEL].set_index("group")
    rows = [
        ("01_model_performance_wape_relmae.png", "전체 모델 성능 비교 (WAPE·RelMAE, 순위)", f"최종 권고 {SHORT[FINAL_MODEL]}의 WAPE {fin_o['WAPE']:.1f}%는 직전 7일 평균 {base_o['WAPE']:.1f}%보다 {base_o['WAPE'] - fin_o['WAPE']:.1f}%p 낮다.", "본문", "5~6", "3", "14개 모델을 같은 rolling-origin에서 비교했고 최종 권고가 WAPE 기준 상위입니다.", "01_model_comparison_overall.csv"),
        ("02_probabilistic_hit_coverage_width.png", "확률 예측·운영 성능 (q80 적중률·80% 포함률·평균 구간 길이)", f"최종 권고 q80 적중률 {fin_o['hit_rate']:.1%}, 포함률 {fin_o['coverage']:.1%}, 평균 구간 {fin_o['mean_interval_width']:,.0f}명으로 목표 80%에 근접.", "본문", "6", "3~4", "인력 산정에 쓰는 q80이 목표 80%에 맞게 보정되어 있습니다.", "01_model_comparison_overall.csv"),
        ("03_cost_and_fva_vs_mean7.png", "비용·FVA (직전 7일 평균 대비 절감, 부족/잉여 분리)", f"총비용 {fin_q['total_cost_krw'] / 1e8:.2f}억 원, 직전 7일 평균 대비 {fin_q['saving_vs_mean7_krw'] / 1e8:.2f}억 원({fin_q['saving_rate_vs_mean7']:.0%}) 절감.", "본문", "7", "5", "같은 q80 정책으로 비교해도 모델이 비용을 크게 줄입니다.", "07_fva_cost_comparison.csv"),
        ("04_group_wape_cost.png", "유형별 성능 (궁·왕릉·종묘 WAPE·비용)", f"궁 WAPE {g.loc['궁', 'WAPE']:.1f}%, 종묘 {g.loc['종묘', 'WAPE']:.1f}%, 왕릉 {g.loc['왕릉', 'WAPE']:.1f}%; 비용은 궁에 집중.", "본문", "8", "6", "궁은 q80 정밀 배치, 왕릉은 최소 인력 정책이라 성격이 다릅니다.", "02_model_comparison_by_group.csv"),
        ("05_site_heatmap.png", "개소별 성능 히트맵 (WAPE·적중률·포함률·비용)", "23개소의 정확도와 비용이 크게 다르며 큰 궁이 비용을 좌우한다.", "부록(본문 1장 요약 가능)", "9", "부록 A1", "개소별로 성능 편차가 있어 개소별 모니터링이 필요합니다.", "03_model_comparison_by_site.csv"),
        ("06_date_type_performance.png", "날짜 유형별 성능(표본 수 표시)", f"설·추석 명절·황금연휴·공휴일의 부족 인원이 일반일보다 크다 (황금연휴·명절은 전체 부족의 {golden_share_text}).", "본문", "9~10", "7", "오차는 공휴일·연휴에 집중됩니다.", "05_metrics_by_date_type.csv"),
        ("07_rain_holiday_cross.png", "비 유무 × 휴일 유형", "비 유무보다 휴일 유형이 오차에 더 크게 작용한다(주 단위 조건, 강수는 사후 실측).", "부록", "10", "부록 A2", "강수는 보조 신호이며 인력 감축의 단독 근거가 아닙니다.", "06_rain_holiday_cross_analysis.csv"),
        ("08_actual_vs_forecast_2025.png", "2025 실제값 vs 최종 모델 예측 추이", "연휴·무료개방일의 급증 피크에서 점 예측이 낮고 q80이 일부 보완한다.", "본문", "8", "8", "공휴일·무료개방일 피크가 가장 어렵습니다.", "13/10 표 참조"),
        ("09_staff_shortage_top10.png", "2025 인력 부족 Top 10", f"상위 부족은 연휴·공휴일에 집중, 황금연휴·명절이 전체 부족의 {golden_share_text}.", "본문", "10~11", "9", "큰 궁의 명절·연휴 수요가 부족을 만듭니다.", "10_top10_staff_shortages.csv"),
        ("10_labor_day_long_term_evidence.png", "근로자의 날 장기 근거(2015~2025)", f"5월 1일은 평상시 일평균의 {RATIO_ALL:.2f}배, {N_SITES_HIGHER}/{N_SITES_LONG}개소가 평상시보다 높다. 연도별로는 편차가 있고 2025년은 1 미만.", "본문", "11", "10", "근로자의 날은 법정공휴일이 아니라 반복 관찰된 특수 수요일 신호이며 인과로 단정하지 않습니다.", "08_labor_day_long_term_evidence.csv"),
        ("11_labor_day_ablation.png", "근로자의 날 ablation (A: 공휴일+무료개방 vs B: +근로자의 날)", "전체 지표는 휴일 묶음 B가 소폭 유리하나 2025-05-01 하루는 반대이며 독립 표본은 1일뿐이다.", "본문(또는 부록)", "11", "11", "하루 오차로 장기 효과를 판단하지 않습니다.", "09_labor_day_ablation.csv"),
        ("12_golden_holiday_scenarios.png", "황금연휴 운영 시나리오 (q80 / +1 / +2 / q90)", f"운영 가능 범위 안 최소비용 권고: {GOLDEN_ADD['scenario']} ({GOLDEN_ADD['apply_to']}). 부족 위험과 잉여 비용의 trade-off.", "본문", "12", "12", "황금연휴에는 검증된 추가 배치 시나리오를 적용합니다.", "12_golden_holiday_scenarios.csv"),
        ("13_forecast_2026_visitors.png", "2026년 14일 예측 방문객 (point·q80)", f"2026-01-01~14 예상 방문객 {int(F15['predicted_total_point'].sum()):,}명 (q80 합계 {int(F15['predicted_total_q80'].sum()):,}명).", "본문", "12", "13", "신정(1/1)과 주말에 방문객이 높습니다.", "13_forecast_2026_site_daily.csv"),
        ("14_staffing_2026_daily.png", "2026년 14일 일별 권고 인력 (유형별 누적)", f"14일 총 {TOTAL_PD} person-days (궁 {PD_GROUP['궁']}, 종묘 {PD_GROUP['종묘']}, 왕릉 {PD_GROUP['왕릉']}).", "본문", "12", "14", "일별 총 인원과 고수요 개소를 운영팀에 전달합니다.", "15_staffing_2026_daily_summary.csv"),
        ("15_staffing_2026_site_heatmap.png", "2026년 개소별 운영 배치 히트맵", f"왕릉은 대부분 1명이며 1명 초과 왕릉은 {len(TOMB_MULTI)}개소·일.", "본문(운영 부록 겸용)", "12", "15", "개소×날짜 배치표를 그대로 현장에 공유할 수 있습니다.", "14_staffing_2026_site_daily.csv"),
        ("16_executive_summary_infographic.png", "경영진 핵심 요약 인포그래픽", f"{SHORT[FINAL_MODEL]}, WAPE {fin_o['WAPE']:.1f}%, q80 적중률 {fin_o['hit_rate']:.1%}, 절감 {fin_q['saving_vs_mean7_krw'] / 1e8:.2f}억 원, 2026 {TOTAL_PD} person-days.", "본문(요약)", "2", "1", "최종 모델과 핵심 숫자, 핵심 위험(공휴일·황금연휴)을 한 장으로 보여줍니다.", "17/07/13~16 표 참조")]
    manifest = pd.DataFrame(rows, columns=["file", "title", "key_message", "placement", "ppt_slide_recommendation", "report_page_recommendation", "speaker_note", "source_table"])
    manifest.insert(0, "figure_no", [int(r[0][:2]) for r in rows])
    manifest["tier"] = np.where(manifest["placement"].str.startswith("본문"), "본문", "부록")
    manifest["resolution_px"] = [f"{Image.open(FIG_DIR / f).size[0]}x{Image.open(FIG_DIR / f).size[1]}" for f in manifest["file"]]
    manifest["dpi"] = 300
    return manifest


def final_tables():
    T = {}
    T["01_model_comparison_overall.csv"] = money_columns(OVERALL.assign(rank_wape=OVERALL["WAPE"].rank(method="min"), rank_total_cost=OVERALL["total_cost_krw"].rank(method="min")).sort_values("model_order").drop(columns="model_order"))
    T["02_model_comparison_by_group.csv"] = money_columns(GROUP_T.sort_values(["group", "model_order"]).drop(columns="model_order"))
    T["03_model_comparison_by_site.csv"] = money_columns(SITE_T.sort_values(["group", "site", "model_order"]).drop(columns="model_order"))
    T["04_model_comparison_by_origin.csv"] = money_columns(ORIGIN_T.sort_values(["fold", "model_order"]).drop(columns="model_order"))
    dt = DATE_TYPE_T.copy()
    dt["_o"] = dt["date_type"].map({d: i for i, d in enumerate(DATE_TYPES)})
    T["05_metrics_by_date_type.csv"] = money_columns(dt.sort_values(["_o", "model_order"]).drop(columns=["_o", "model_order"]))
    T["06_rain_holiday_cross_analysis.csv"] = money_columns(CROSS_T.assign(rain_week_basis="사후 평가용 실측값(모델 입력 아님)").sort_values(["rain_week", "holiday_week", "model_order"]).drop(columns="model_order"))
    T["07_fva_cost_comparison.csv"] = money_columns(FVA_T.drop(columns="model_order"))
    T["08_labor_day_long_term_evidence.csv"] = LABOR_EVIDENCE
    T["09_labor_day_ablation.csv"] = money_columns(ABLATION)
    T["10_top10_staff_shortages.csv"] = TOP_SHORTAGE
    T["11_top10_high_cost_errors.csv"] = TOP_COST
    T["12_golden_holiday_scenarios.csv"] = money_columns(GOLDEN_T)
    T["13_forecast_2026_site_daily.csv"] = F13
    T["14_staffing_2026_site_daily.csv"] = F14
    T["15_staffing_2026_daily_summary.csv"] = F15
    T["16_staffing_2026_site_summary.csv"] = F16
    T["19_top10_underforecast_and_surplus.csv"] = TOP_OTHER
    T["20_model_selection.csv"] = SEL_ORDER_T.drop(columns=["special_hit_rate_desc"]).assign(is_option1=lambda d: d["model_id"] == OPT1, is_option2=lambda d: d["model_id"] == OPT2, is_final=lambda d: d["model_id"] == FINAL_MODEL)
    T["21_mlp_seed_stability.csv"] = MLP_SEED_T
    T["22_date_type_shortage_share.csv"] = DATE_TYPE_SHORTAGE.reset_index().rename(columns={"index": "date_type"})
    return T


from PIL import Image
MLP_SEED_ROWS = []
for seed, part in MLP_SEED_PRED.groupby("seed"):
    pred = add_total(part.drop(columns="seed"))
    pred["date"] = pd.to_datetime(pred["date"])
    params = conformal_params(calibration_frame(False, pred=pred))
    ev = pred.loc[pred["phase"] == "test"].merge(ACTUAL_LONG, on=["date", "site", "target"], how="left").merge(EVAL.loc[EVAL["model_id"] == REFERENCE_MODEL, ["date", "site", "target", "reference"]].drop_duplicates(["date", "site", "target"]), on=["date", "site", "target"], how="left")
    ev = apply_intervals(ev, params)
    ev["is_closed"] = ev["actual"].isna()
    m = calculate_metrics(ev.loc[ev["target"] == "total"], ["model_id"]).iloc[0]
    open_rows = ev.loc[(ev["target"] == "total") & ~ev["is_closed"]]
    gap = staff_rule(to_visitors_int(open_rows["q80_equiv"]), np.zeros(len(open_rows), dtype=bool)) - staff_formula(to_visitors_int(open_rows["actual"]))
    MLP_SEED_ROWS.append(dict(seed=int(seed), WAPE=float(m["WAPE"]), RelMAE=float(m["RelMAE"]), hit_rate=float(m["hit_rate"]), coverage=float(m["coverage"]), mean_interval_width=float(m["mean_interval_width"]),
                              shortage_person_days=float(np.clip(-gap, 0, None).sum()), surplus_person_days=float(np.clip(gap, 0, None).sum()), total_cost_krw=float(np.clip(-gap, 0, None).sum() * SHORTAGE_COST_KRW + np.clip(gap, 0, None).sum() * OVERSTAFF_COST_KRW)))
MLP_SEED_T = pd.DataFrame(MLP_SEED_ROWS).sort_values("seed").reset_index(drop=True)
mlp_summary_rows = pd.DataFrame([dict(seed="mean", **MLP_SEED_T.drop(columns="seed").mean().to_dict()), dict(seed="std", **MLP_SEED_T.drop(columns="seed").std(ddof=1).to_dict()),
                                 dict(seed=f"representative({SEED})", **{c: float(O.loc["mlp_global", c]) if c in O.columns else float(FVA_Q80.loc["mlp_global", c]) for c in ["WAPE", "RelMAE", "hit_rate", "coverage", "mean_interval_width"]},
                                      shortage_person_days=float(FVA_Q80.loc["mlp_global", "shortage_person_days"]), surplus_person_days=float(FVA_Q80.loc["mlp_global", "surplus_person_days"]), total_cost_krw=float(FVA_Q80.loc["mlp_global", "total_cost_krw"]))])
MLP_SEED_T = pd.concat([MLP_SEED_T.assign(seed=MLP_SEED_T["seed"].astype(str)), mlp_summary_rows], ignore_index=True)
MANIFEST = build_manifest()
print("MLP 시드 안정성: WAPE", MLP_SEED_T.loc[MLP_SEED_T["seed"] == "mean", "WAPE"].round(2).iloc[0], "±", MLP_SEED_T.loc[MLP_SEED_T["seed"] == "std", "WAPE"].round(2).iloc[0])

In [ ]:
def write_documents():
    mean7_wape = base_o["WAPE"]
    t1 = OVERALL.sort_values("WAPE")[["model_label", "WAPE", "RelMAE", "hit_rate", "coverage", "mean_interval_width", "shortage_person_days", "surplus_person_days", "total_cost_krw"]].copy()
    t1["total_cost_krw"] = t1["total_cost_krw"] / 1e8
    t1 = t1.rename(columns={"model_label": "모델", "WAPE": "WAPE(%)", "hit_rate": "q80 적중률", "coverage": "80% 포함률", "mean_interval_width": "평균 구간 길이(명)", "shortage_person_days": "부족(인일)", "surplus_person_days": "잉여(인일)", "total_cost_krw": "총비용(억 원)"})
    abl = ABLATION.loc[ABLATION["scope"].isin(["전체 2025", "2025-05-01 하루(23개소)"]), ["scope", "model_id", "WAPE", "RelMAE", "hit_rate", "coverage", "mean_interval_width", "shortage_person_days", "surplus_person_days", "total_cost_krw"]].copy()
    abl["model_id"] = abl["model_id"].map({ABL_ID: "A 근로자의 날 제외", B_ID: "B 근로자의 날 포함", "diff_B_minus_A": "차이 (B − A)"})
    abl["total_cost_krw"] = abl["total_cost_krw"] / 1e8
    abl = abl.rename(columns={"scope": "범위", "model_id": "모델", "WAPE": "WAPE(%)", "hit_rate": "q80 적중률", "coverage": "포함률", "mean_interval_width": "구간 길이", "shortage_person_days": "부족(인일)", "surplus_person_days": "잉여(인일)", "total_cost_krw": "총비용(억 원)"})
    gold = GOLDEN_T[["scenario", "apply_to", "service_level_no_shortage", "shortage_person_days", "surplus_person_days", "total_cost_krw", "extra_person_days_vs_base", "operationally_feasible", "is_recommended"]].copy()
    gold["total_cost_krw"] = gold["total_cost_krw"] / 1e6
    gold = gold.rename(columns={"scenario": "시나리오", "apply_to": "적용 범위", "service_level_no_shortage": "서비스 수준", "shortage_person_days": "부족(인일)", "surplus_person_days": "잉여(인일)", "total_cost_krw": "총비용(백만 원)", "extra_person_days_vs_base": "추가 인일", "operationally_feasible": "운영 가능", "is_recommended": "권고"})
    daily = F15[["date", "weekday", "holiday_name", "predicted_total_point", "predicted_total_q80", "final_operational_staff", "palace_staff", "jongmyo_staff", "royal_tomb_staff"]].rename(columns={"predicted_total_point": "점 예측", "predicted_total_q80": "q80", "final_operational_staff": "총 권고", "palace_staff": "궁", "jongmyo_staff": "종묘", "royal_tomb_staff": "왕릉"})
    g_all = GROUP_T.loc[GROUP_T["model_id"] == FINAL_MODEL].set_index("group")
    top = TOP_SHORTAGE.head(5)[["date", "weekday", "site", "date_type", "actual_visitors", "predicted_q80", "required_staff_from_actual_visitors", "recommended_staff_q80", "staff_shortage", "probable_cause"]]
    date_tbl = DATE_TYPE_SHORTAGE.reset_index().rename(columns={"index": "date_type"})
    date_tbl = date_tbl.assign(shortage_share=date_tbl["shortage_share"] * 100, site_day_share=date_tbl["site_day_share"] * 100).rename(columns={"date_type": "날짜 유형", "site_days": "개소·일", "shortage_person_days": "부족(인일)", "surplus_person_days": "잉여(인일)", "shortage_share": "부족 비중(%)", "site_day_share": "개소·일 비중(%)"})
    ablation_all = ABLATION.loc[ABLATION["scope"] == "전체 2025"].set_index("model_id")
    ablation_day = ABLATION.loc[ABLATION["scope"] == "2025-05-01 하루(23개소)"].set_index("model_id")
    same_weekday_up = LABOR_EVIDENCE.loc[LABOR_EVIDENCE["analysis"] == "민감도: 같은 연도 5월 같은 요일 평균(요약)", "note"].iloc[0]
    opt_table = OPTION_TABLE.copy()
    opt_table["total_cost_krw"] = opt_table["total_cost_krw"] / 1e8
    opt_table = opt_table.reset_index(drop=True).rename(columns={"role": "역할", "is_final_recommendation": "최종 권고", "model_label": "모델", "hit_rate": "q80 적중률", "coverage": "포함률", "mean_interval_width": "구간 길이", "total_cost_krw": "총비용(억 원)", "shortage_person_days": "부족(인일)", "surplus_person_days": "잉여(인일)", "special_shortage_person_days": "공휴일·연휴·무료개방일 부족(인일)", "special_hit_rate": "해당일 적중률"})
    fin_cost_text = f"{fin_q['total_cost_krw']:,.0f}원({fin_q['total_cost_krw'] / 1e8:.2f}억 원)"
    summary = f"""# Project 3 최종 보고: 궁·종묘·왕릉 방문객 예측과 2026년 14일 운영 인력 배치

> **권고 인원은 실제 근무표상의 배치 인원이 아니라, 예측 방문객(q80)과 운영 가정(방문객 2,000명당 1명, 부족 1인·일 600,000원, 잉여 150,000원)으로 산출한 관람객 대응 필요 인원이다.** 이 문서의 모든 인력·비용 값에 같은 의미가 적용된다.

## 1. 한 페이지 요약
- **분석 목적**: 23개 개소(궁 4, 종묘 1, 왕릉 18)의 14일 앞 일 방문객을 예측하고, 부족 위험(1인·일 60만 원)이 잉여(15만 원)보다 4배 비싼 비용 구조에 맞춰 q80 기준 운영 인력을 산정한다.
- **최종 선택 모델**: **{label_final}** (`{FINAL_MODEL}`). {FINAL_REASON}
- **핵심 지표(2025 순차 백테스트, 총방문객, 휴관일 제외)**: WAPE **{fin_o['WAPE']:.2f}%**(직전 7일 평균 {base_o['WAPE']:.2f}%), RelMAE {fin_o['RelMAE']:.3f}, q80 적중률 **{fin_o['hit_rate']:.1%}**, 80% 구간 포함률 **{fin_o['coverage']:.1%}**, 평균 구간 길이 {fin_o['mean_interval_width']:,.0f}명.
- **비용 절감**: 같은 q80 인력 정책에서 총비용 {fin_cost_text} vs 직전 7일 평균 {base_q['total_cost_krw']:,.0f}원({base_q['total_cost_krw'] / 1e8:.2f}억 원) → 절감 **{fin_q['saving_vs_mean7_krw']:,.0f}원({fin_q['saving_vs_mean7_krw'] / 1e8:.2f}억 원, {fin_q['saving_rate_vs_mean7']:.1%})**.
- **2026-01-01~14 권고 인력**: 총 **{TOTAL_PD:,} person-days** (궁 {PD_GROUP['궁']}, 종묘 {PD_GROUP['종묘']}, 왕릉 {PD_GROUP['왕릉']}). 총 예상 방문객 {int(F15['predicted_total_point'].sum()):,}명(점 예측), q80 합계 {int(F15['predicted_total_q80'].sum()):,}명. 정기 휴관 {int(F14['is_regular_closed'].sum())}개소·일은 0명. 왕릉 q80이 2,000명을 넘어 2명 이상 필요한 개소·일 {len(TOMB_MULTI)}건.
- **가장 큰 운영 위험**: 공휴일·황금연휴·설추석 명절. 2025년 인력 부족 인원의 **{GOLDEN_SHORTAGE_SHARE:.1%}**가 황금연휴·명절 개소·일(전체 개방 행의 {float((FINAL_ROWS['is_golden_holiday'] | FINAL_ROWS['is_myeongjeol']).mean()):.1%})에서 발생했다.
- **운영팀이 취할 행동 3개**: ① 궁·종묘는 q80 기반 정밀 배치표(`staffing/`)를 사용 ② 왕릉은 개방일 최소 1명, q80 2,000명 초과 시 증원 ③ 황금연휴·명절에는 아래 검증된 시나리오({GOLDEN_ADD['scenario']}, {GOLDEN_ADD['apply_to']})를 적용하고 근로자의 날은 특수 수요일로 관리.

## 2. 모델링 결론
### 2.1 비교 결과 (2025 순차 rolling-origin 백테스트)
2024년까지 초기 학습한 뒤 2025년을 14일 단위로 순차 예측(예측 원점 {len(TEST_WINDOWS)}개, 각 원점은 그 이전 자료만 사용)하며 비교한 실제 모델 행은 **{N_MODEL_ROWS}개**다(베이스라인 3 + ARIMA·SARIMA·MLP + Chronos-2 8종).

{md_table(t1, 3)}

### 2.2 선택 근거
- **WAPE와 RelMAE**: 개소 규모 차이가 큰 일 방문객에서 WAPE는 실제 방문객 총량 대비 절대오차의 비율이라 큰 개소의 오차를 그대로 반영하고 운영 인력(인원 단위 합계)과 직접 연결된다. RelMAE는 계절 나이브 대비 개선 정도를 보는 보조지표다. **RMSE는 주지표로 쓰지 않았다**: 큰 개소·연휴 피크의 오차가 제곱되어 소수 날짜가 지표를 지배하고, 방문객이 0에 가까운 소규모 왕릉과 섞이면 단위·규모 해석이 어렵기 때문이다.
- **14일 예측**: 요청 설계가 14일 horizon이고 2주 단위 근무 편성에 쓸 수 있는 길이이지만 실제 편성 주기는 확인하지 못했다. 같은 설계로 기존 결과와 비교할 수 있다.
- **2023~2025년 자료**: 전처리·검증이 끝난 최근 3년과 달력·무료개방 표시가 일관되게 정리되어 있다(코로나 이후 수요 회복기). 장기 자료(2015~2025)는 근로자의 날 근거에만 사용했다.
- **rolling-origin**: 각 원점에서 그 원점 이전 관측만 쓰고 예측 14일 구간 안에서는 베이스라인도 앞서 만든 예측값만 재사용한다. 원점 이후 목표값을 변조해도 예측이 변하지 않음을 검산했다. 강수는 실측이 아니라 기후 평균(월×요일→월→계절→전체 백오프, 2025는 2023~2024, 2026은 2023~2025 자료)을 입력했다.
- **최종 모델 선정**(WAPE → 총비용 → 공휴일·연휴·무료개방일 부족·적중률 → RelMAE → 포함률 → 구간 길이 → 복잡도): 허용오차를 두지 않은 정확도·비용 최우선 모델과 단순성 실무 대안을 모두 제시한다.

{md_table(opt_table, 3)}

{FINAL_REASON}

### 2.3 유형별 결과 (최종 권고)
- 궁 WAPE {g_all.loc['궁', 'WAPE']:.1f}% · 종묘 {g_all.loc['종묘', 'WAPE']:.1f}% · 왕릉 {g_all.loc['왕릉', 'WAPE']:.1f}%; 총비용 궁 {g_all.loc['궁', 'total_cost_krw'] / 1e8:.2f}억, 종묘 {g_all.loc['종묘', 'total_cost_krw'] / 1e8:.2f}억, 왕릉 {g_all.loc['왕릉', 'total_cost_krw'] / 1e8:.2f}억 원. 궁이 비용의 대부분을 차지해 q80 정밀 배치의 효과가 가장 크다.

### 2.4 오류 탐지: 날짜 유형별 부족 인원 (최종 권고, 2025)
{md_table(date_tbl, 3)}

인력 부족 상위 5건:

{md_table(top, 0)}

## 3. 근로자의 날 결론
2015~2025년 장기 자료에서 5월 1일 방문객은 전체 일평균의 **{RATIO_ALL:.2f}배**였고 **{N_SITES_HIGHER}/{N_SITES_LONG}개소 모두** 평상시보다 높았다. 2025년 백테스트에서 해당 날짜의 오차가 증가했지만 독립적인 날짜 표본은 하루뿐이다(2025-05-01 하루 WAPE: 근로자의 날 제외 모델 A {ablation_day.loc[ABL_ID, 'WAPE']:.1f}% → 포함 모델 B {ablation_day.loc[B_ID, 'WAPE']:.1f}%, 전체 2025 WAPE A {ablation_all.loc[ABL_ID, 'WAPE']:.2f}% → B {ablation_all.loc[B_ID, 'WAPE']:.2f}%). 따라서 근로자의 날 변수는 2025년 하루의 적합도를 높이기 위한 변수가 아니라, 장기적으로 반복된 특수 수요 신호를 반영하기 위한 변수로 유지한다(법정공휴일이 아니라 반복 관찰된 특수 수요일 피처).
- **인과 단정 금지**: 장기 자료에는 봄철 계절성·요일·코로나 기간이 섞여 있다. 민감도: 코로나 2020~2022 제외 {RATIO_EX_COVID:.2f}배, 개소 수 균형 2019~2025 {RATIO_BALANCED:.2f}배, 같은 연도 5월 같은 요일 평균과 비교 {RATIO_SAME_WEEKDAY:.2f}배({same_weekday_up}). 연도별 합계 비율은 {YEAR_RATIOS.min():.2f}~{YEAR_RATIOS.max():.2f}배로 편차가 크고 2025년({YEAR_RATIOS.get('2025', float('nan')):.2f}배)은 1 미만이다.

{md_table(abl, 3)}

## 4. 운영 권고
- **궁·종묘**: q80 기반 정밀 배치(`recommended_staff_q80`, `staffing/`). 궁이 인력 부족·비용의 대부분을 차지한다.
- **왕릉**: 개방일 최소 1명, q80 방문객이 2,000명을 넘으면 `ceil(q80/2000)`로 증원. 2026-01-01~14에는 2명 이상이 필요한 왕릉 개소·일이 {len(TOMB_MULTI)}건이다. 왕릉 정책은 2025 실제 방문객 환산과 비교해 검증했다(`summary/model_card.md`의 왕릉 정책 검증 참조).
- **황금연휴**: 2025 백테스트의 황금연휴·명절 개방 행 {int(GOLDEN_T['n_site_days'].iloc[0])}개소·일에서 시나리오를 비교했다. 운영 가능 범위(추가 ≤ 기본의 {FEASIBLE_EXTRA_RATIO:.0%}) 안 최소비용 권고는 **{GOLDEN_ADD['scenario']} ({GOLDEN_ADD['apply_to']})**.

{md_table(gold, 3)}

- **근로자의 날**: 특수 수요일로 관리(2026-05-01은 공휴일로 지정되어 `is_holiday=1`·`is_labor_day=1`이 함께 적용되는 외삽 상황).
- **강수**: 인력 감축의 단독 근거로 쓰지 말고 보조 신호로 사용한다. 실제 기상예보가 확보되면 **배치 확정 3일 전에 갱신**한다.

## 5. 2026-01-01~14 일별 권고 (최종 권고 모델, q80 기준)
{md_table(daily, 0)}

개소별·일별 배치표는 `staffing/00_operational_staffing_2026.xlsx`(CSV: `staffing/01_*`, `tables/14_*`)에 있다.

## 6. 제한사항
- 실제 근무표 자료가 없어 실제 배치 인원이 아니라 **환산 필요 인원**을 평가했다(2,000명당 1명, 부족 60만 원·잉여 15만 원은 입력받은 운영 가정).
- 2025년 근로자의 날 독립 표본은 **1일**이다. 장기 근거는 계절성·요일·코로나 영향을 완전히 분리하지 못했다.
- 2026년 강수는 관측값이 아닌 **기후 평균 기반 사전 추정치**이며, 2026-01-01~14에는 근로자의 날·황금연휴가 없어 이 구간에서는 해당 규칙이 검증되지 않는다.
- 2026년 5월 1일의 `is_holiday=1, is_labor_day=1` 조합은 과거(2023~2025: `is_holiday=0, is_labor_day=1`)와 제도적 조건이 달라 **외삽 성격**이 있다.
- 행사(이벤트) 변수는 미래 일정의 확정 가능성 문제로 본 최종 모델에서 제외했다.
- 모델 선정에 2025 순차 백테스트 성능을 쓰므로 같은 구간의 성능은 다소 낙관적일 수 있다. 무료개방 표시(`is_free_admission`)는 사후 정리된 정보라 2025 성능은 사후적 참고이다.
- 날씨 파일에 2022-12-31 자료가 없어 2023-01-01의 `after_heavy`는 0으로 두었다.
- 기존 결과와의 차이: 직전 7일 평균 q80 비용은 {base_q['total_cost_krw']:,.0f}원으로 기존 기준 {REF_MEAN7:,}원과 {base_q['total_cost_krw'] - REF_MEAN7:+,.0f}원 차이, 최종 권고 비용은 {final_q_cost:,.0f}원으로 기존 최종 후보 {REF_PREV_FINAL:,}원과 {final_q_cost - REF_PREV_FINAL:+,.0f}원 차이다. 차이가 0원이면 기존 결과가 같은 입력·같은 q80 보정으로 그대로 재현된 것이다. 0이 아니면 강수 묶음 정의 변경(월×요일 백오프 기후 확률, 4열)이나 단일 전체 모델 선정(기존은 유형별 모델) 때문이다. 이번 실행의 최종 권고 모델은 강수 묶음을 쓰지 않는 휴일 묶음+무료개방이라 기존 후보와 같은 예측 캐시가 재사용되었다.
"""
    summary = summary.replace("final_q_cost", "fin_q['total_cost_krw']")
    (SUMMARY_DIR / "final_executive_summary.md").write_text(summary, encoding="utf-8")
    guide_rows = MANIFEST[["figure_no", "file", "title", "key_message", "tier", "ppt_slide_recommendation", "report_page_recommendation", "speaker_note", "source_table"]]
    guide = ["# PPT·보고서 적용 안내", "", "PPT 본문에는 약 10~12개의 핵심 시각화만 쓰고 나머지는 부록으로 분류했다. 모든 그림은 PNG, 16:9, 300dpi 이상이며 한글 폰트는 Noto Sans KR(없으면 맑은 고딕)이다. 인력은 실제 근무표가 아니라 q80 기준 환산 필요 인원이다.", "",
             f"본문 {int((MANIFEST['tier'] == '본문').sum())}개 / 부록 {int((MANIFEST['tier'] == '부록').sum())}개", ""]
    for _, r in guide_rows.iterrows():
        guide += [f"## 그림 {int(r['figure_no']):02d}. {r['title']}", f"- 파일: `figures/{r['file']}` ({r['tier']})", f"- 핵심 메시지: {r['key_message']}", f"- PPT 권장 슬라이드: {r['ppt_slide_recommendation']}", f"- 12페이지 보고서 권장 위치: {r['report_page_recommendation']}",
                  f"- 발표자 한두 문장: {r['speaker_note']}", f"- 근거 표: `{r['source_table']}`", ""]
    guide += ["## 표 (tables/ 및 staffing/)", "| 파일 | 용도 | 본문/부록 | PPT·보고서 권장 위치 |", "|---|---|---|---|",
              "| 01_model_comparison_overall.csv | 전체 모델 순위·지표·비용 | 본문 | PPT 5~6 / 보고서 3p |", "| 07_fva_cost_comparison.csv | 직전 7일 평균 대비 절감 | 본문 | PPT 7 / 보고서 5p |", "| 10_top10_staff_shortages.csv | 인력 부족 Top 10 | 본문 | PPT 10 / 보고서 9p |",
              "| 12_golden_holiday_scenarios.csv | 황금연휴 시나리오 | 본문 | PPT 12 / 보고서 12p |", "| 08_labor_day_long_term_evidence.csv, 09_labor_day_ablation.csv | 근로자의 날 근거·ablation | 본문/부록 | PPT 11 / 보고서 10~11p |",
              "| 02/03/04/05/06 비교표 | 유형·개소·원점·날짜유형·비×휴일 | 부록 | 보고서 부록 |", "| 13/14/15/16 (2026) 및 staffing/ 파일 | 예측·운영 배치표 | 본문(요약)/운영 전달 | 운영팀에 `00_operational_staffing_2026.xlsx` 전달 |",
              "| 17_validation_checks.csv, summary/model_card.md | 검증·모델 카드 | 부록 | 보고서 부록 |"]
    (SUMMARY_DIR / "ppt_report_placement_guide.md").write_text("\n".join(guide), encoding="utf-8")
    tomb = FINAL_ROWS.loc[FINAL_ROWS["group"] == "왕릉"]
    tomb_fixed = pd.Series(np.where(tomb["required_staff_actual"] > 0, 1, 0), index=tomb.index)
    gap_fixed = tomb_fixed - tomb["required_staff_actual"]
    cost_fixed = float(np.clip(-gap_fixed, 0, None).sum() * SHORTAGE_COST_KRW + np.clip(gap_fixed, 0, None).sum() * OVERSTAFF_COST_KRW)
    cost_q80 = float(tomb["total_cost_q80"].sum())
    tomb_exceed = tomb.loc[tomb["actual_visitors"] > STAFF_UNIT]
    tomb_site_exceed = tomb.groupby("site").agg(days_actual_over_2000=("actual_visitors", lambda s: int((s > STAFF_UNIT).sum())), max_actual=("actual_visitors", "max"), mean_actual=("actual_visitors", "mean"), q80_over_2000_days=("prediction_q80", lambda s: int((s > STAFF_UNIT).sum()))).sort_values("days_actual_over_2000", ascending=False)
    tomb_policy_txt = md_table(tomb_site_exceed.head(6).reset_index().rename(columns={"site": "왕릉", "days_actual_over_2000": "실제 2,000명 초과 일수", "max_actual": "최대 실제 방문객", "mean_actual": "평균 실제", "q80_over_2000_days": "q80 2,000명 초과 일수"}), 0)
    card = f"""# 모델 카드

## 모델 개요
- **최종 권고 모델**: {label_final} (`{FINAL_MODEL}`), 공변량 {REG[FINAL_MODEL]['covariates'] or '없음'}. 정확도·비용 최우선 Option 1 = `{OPT1}`, 단순성 실무 대안 Option 2 = `{OPT2}`.
- **구조**: 개소별 일 시계열을 내국인·외국인(영어+일본어+중국어+기타)으로 각각 예측해 합산(총방문객). Chronos-2(`{CHRONOS_MODEL_ID}`) 제로샷 + 과거·미래 공변량. 예측 구간은 모든 모델에 같은 split conformal 80%(보정: 2024-07-03~12-31, 13개 구간, 개소·타깃별)이며 q80 상당 값은 `max(예측 + 보정 분위수, 0)`이다.
- **공변량**: 휴일 묶음 `is_holiday`, `is_labor_day`(05-01, 코드로 생성) / 강수 묶음 `rain_prob`, `rain_1_10_prob`, `heavy_prob`, `after_heavy_prob`(경계 {RAIN_MM}mm·{HEAVY_MM}mm, 월×요일→월→계절→전체 백오프, 최소 표본 {WEATHER_MIN_OBS}) / 무료개방 `free_open`(2023~2025 방문객 파일의 `is_free_admission`, 2026은 달력 규칙 + 상시 무료 개소).

## 데이터와 기간
- 입력 파일 해시 `{INPUT_HASH}`. 초기 학습 {HISTORY_START.date()}~{TRAIN_END.date()}, 2025 순차 rolling-origin {len(TEST_WINDOWS)}개 원점(14일), 최종 학습 {HISTORY_START.date()}~{FINAL_TRAIN_END.date()}, 예측 {FORECAST_START.date()}~{FORECAST_END.date()}.
- 결측: 휴관일(방문객 NaN)은 학습·평가에서 제외. WAPE 분모(Σ|실제|)가 0이면 NaN(이번 실행에는 해당 없음).

## 재현 조건
- 시드: `SEED = {SEED}` (Chronos-2 로드 시 torch·numpy), MLP 대표 시드 {SEED} + 반복 시드 {MLP_SEEDS[0]}~{MLP_SEEDS[-1]} (평균 WAPE {MLP_SEED_T.loc[MLP_SEED_T['seed'] == 'mean', 'WAPE'].iloc[0]:.2f}% ± {MLP_SEED_T.loc[MLP_SEED_T['seed'] == 'std', 'WAPE'].iloc[0]:.2f}%p).
- 환경과 패키지 버전은 `logs/01_environment.json`, 단계별 실행 시간은 `logs/02_runtime.csv`, 강수 백오프 단계 사용량은 `logs/03_weather_backoff.csv`.
- 캐시: 입력 파일 해시와 모델 정의(공변량 목록, 기간, 설정)가 서명에 들어 있어 입력·정의가 바뀌면 재사용되지 않는다. 정의가 같은 베이스라인·ARIMA·SARIMA·MLP·강수 묶음이 없는 Chronos 모델은 이전 실행의 동일 서명 예측을 재사용했고, 강수 묶음 모델 4종과 ablation 모델 A는 이번에 새로 예측했다.

## 인력 규칙 검증 (2025 실제 방문객 환산과 비교)
- 규칙: `ceil(방문객/2000)`, 개방일 최소 1명, 휴관일 0명, 운영 예측은 q80.
- 왕릉 정책 검증: 왕릉 {len(tomb):,}개소·일 중 실제 방문객이 2,000명을 넘은 날은 {len(tomb_exceed)}건({len(tomb_exceed) / len(tomb):.2%}). "왕릉 항상 1명 고정"의 총비용 {cost_fixed / 1e8:.3f}억 원 vs q80 규칙 {cost_q80 / 1e8:.3f}억 원. 규칙을 임의로 고정하지 않고 q80이 2,000명을 넘는 날은 증원한다. 2,000명 초과 일수가 많은 왕릉:

{tomb_policy_txt}

## 한계
{chr(10).join(['- ' + t for t in ['실제 근무표 자료가 없어 환산 필요 인원 기준이다.', '2025년 근로자의 날 독립 표본은 1일이다.', '2026년 강수는 기후 평균 기반이며 실제 기상예보가 확보되면 배치 확정 3일 전에 갱신한다.', '이벤트 변수는 미래 일정의 확정 가능성 때문에 제외했다.', 'q80 보정은 2023~2024 내부 백테스트로만 하며 2025 오차를 2026 보정에 쓰지 않았다.']])}
"""
    (SUMMARY_DIR / "model_card.md").write_text(card, encoding="utf-8")
    return cost_fixed, cost_q80

In [ ]:
T = final_tables()
for name, frame in T.items():
    frame.to_csv(TABLE_DIR / name, index=False, encoding="utf-8-sig")
MANIFEST.to_csv(TABLE_DIR / "18_figure_manifest.csv", index=False, encoding="utf-8-sig")
# 운영팀 전달 파일 (staffing/)
F14.to_csv(STAFF_DIR / "01_operational_staffing_2026_site_daily.csv", index=False, encoding="utf-8-sig")
F15.to_csv(STAFF_DIR / "02_daily_summary.csv", index=False, encoding="utf-8-sig")
F16.to_csv(STAFF_DIR / "03_site_group_summary.csv", index=False, encoding="utf-8-sig")
TOP5_T.to_csv(STAFF_DIR / "04_top5_high_demand_sites_by_day.csv", index=False, encoding="utf-8-sig")
SPECIAL_DAYS_26.to_csv(STAFF_DIR / "05_special_management_days.csv", index=False, encoding="utf-8-sig")
write_staffing_excel(STAFF_DIR / "00_operational_staffing_2026.xlsx")
final_q_cost = float(FVA_Q80.loc[FINAL_MODEL, "total_cost_krw"])
COST_FIXED, COST_Q80_TOMB = write_documents()
# 로그
env = dict(python=sys.version, platform=platform.platform(), packages={name: getattr(__import__(name), "__version__", "") for name in ("numpy", "pandas", "sklearn", "statsmodels", "torch", "chronos", "matplotlib", "seaborn", "openpyxl", "xlsxwriter")},
           seeds=dict(SEED=SEED, MLP_SEEDS=MLP_SEEDS, torch_manual_seed=SEED, numpy_seed=SEED), constants=dict(RAIN_MM=RAIN_MM, HEAVY_MM=HEAVY_MM, WEATHER_MIN_OBS=WEATHER_MIN_OBS, STAFF_UNIT=STAFF_UNIT, SHORTAGE_COST_KRW=SHORTAGE_COST_KRW, OVERSTAFF_COST_KRW=OVERSTAFF_COST_KRW,
           COVERAGE_LEVEL=COVERAGE_LEVEL, SIMPLICITY_TOL=SIMPLICITY_TOL, COST_TOL=COST_TOL, GOLDEN_MIN_DAYS=GOLDEN_MIN_DAYS, FEASIBLE_EXTRA_RATIO=FEASIBLE_EXTRA_RATIO), periods=dict(train_start=str(HISTORY_START.date()), train_end=str(TRAIN_END.date()), test_start=str(TEST_START.date()), test_end=str(TEST_END.date()),
           final_train_end=str(FINAL_TRAIN_END.date()), forecast=f"{FORECAST_START.date()}~{FORECAST_END.date()}", n_origins=len(TEST_WINDOWS)), chronos_model=CHRONOS_MODEL_ID, input_files={p.as_posix(): hashlib.md5(p.read_bytes()).hexdigest() for p in INPUT_FILES}, input_hash=INPUT_HASH,
           cache_dir=CACHE_DIR.as_posix(), final_model=FINAL_MODEL, option1=OPT1, option2=OPT2, fast_debug=FAST_DEBUG)
(LOG_DIR / "01_environment.json").write_text(json.dumps(env, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
PHASES.append(("저장 및 문서 생성", time.time()))
runtime = pd.DataFrame({"phase": [p for p, _ in PHASES], "elapsed_seconds_since_start": [round(t - T0, 1) for _, t in PHASES]})
runtime["phase_seconds"] = runtime["elapsed_seconds_since_start"].diff().fillna(runtime["elapsed_seconds_since_start"])
runtime.to_csv(LOG_DIR / "02_runtime.csv", index=False, encoding="utf-8-sig")
backoff_rows = []
for cutoff, counts in WEATHER_BACKOFF.items():
    backoff_rows.append(counts.sum().rename("개소·일 수").reset_index().rename(columns={"index": "백오프 단계"}).assign(cutoff=str(cutoff.date()), share=lambda d: d["개소·일 수"] / d["개소·일 수"].sum()))
pd.concat(backoff_rows).to_csv(LOG_DIR / "03_weather_backoff.csv", index=False, encoding="utf-8-sig")
# 저장 후 검산
post = []
expected_tables = [f"{i:02d}" for i in range(1, 19)]
names = sorted(p.name for p in TABLE_DIR.glob("*.csv"))
missing_tables = [i for i in expected_tables if not any(n.startswith(i + "_") for n in names)] if False else [i for i in expected_tables if i != "17" and not any(n.startswith(i + "_") for n in names)]
post.append(dict(check="필수 표 01~16, 18이 모두 저장됨 (17은 이 표 자체)", result="PASS" if not missing_tables else "FAIL", detail=f"저장 {len(names)}개, 누락 {missing_tables}"))
bad = []
for n in names:
    frame = pd.read_csv(TABLE_DIR / n, encoding="utf-8-sig")
    if len(frame) == 0 or len(frame.columns) == 0:
        bad.append(n)
post.append(dict(check="저장된 모든 표가 비어 있지 않음 (재읽기)", result="PASS" if not bad else "FAIL", detail=f"빈 표 {bad}"))
f14_reload = pd.read_csv(TABLE_DIR / "14_staffing_2026_site_daily.csv", encoding="utf-8-sig")
required_cols = ["date", "weekday", "group", "site", "holiday_name", "is_holiday", "is_labor_day", "is_free_open", "expected_rain_class", "predicted_total_point", "predicted_total_q80", "predicted_domestic", "predicted_foreign", "required_staff_point", "recommended_staff_q80", "final_operational_staff", "staffing_rule", "risk_note", "domestic_share", "foreign_share", "recommended_domestic_service_share", "recommended_foreign_service_share"]
split_ok = bool(((f14_reload["recommended_domestic_service_staff"].fillna(0) + f14_reload["recommended_foreign_service_staff"].fillna(0)).where(f14_reload["service_mode"] == "내·외국인 분담 참고", f14_reload["final_operational_staff"]) == f14_reload["final_operational_staff"]).all())
post.append(dict(check="운영 배치표 필수 열(22개) 포함, 내·외국인 참고 인원 합계 = 총 권고 인원(총 1명이면 통합 응대)", result="PASS" if set(required_cols) <= set(f14_reload.columns) and split_ok and len(f14_reload) == len(SITES) * 14 else "FAIL",
                 detail=f"{len(f14_reload)}행 {len(f14_reload.columns)}열, 통합 응대 {int((f14_reload['service_mode'] == '통합 응대').sum())}행, 분담 참고 {int((f14_reload['service_mode'] == '내·외국인 분담 참고').sum())}행"))
xl = pd.ExcelFile(STAFF_DIR / "00_operational_staffing_2026.xlsx")
post.append(dict(check="운영팀용 Excel(시트 7개)과 CSV가 모두 저장됨", result="PASS" if len(xl.sheet_names) == 7 and (STAFF_DIR / "01_operational_staffing_2026_site_daily.csv").exists() else "FAIL", detail=f"시트 {xl.sheet_names}"))
docs = ["final_executive_summary.md", "ppt_report_placement_guide.md", "model_card.md"]
post.append(dict(check="요약 보고서·모델 카드·PPT/보고서 적용 가이드가 저장됨 (필수 문구 포함)", result="PASS" if all((SUMMARY_DIR / d).stat().st_size > 2000 for d in docs) and "실제 근무표상의 배치 인원이 아니라" in (SUMMARY_DIR / "final_executive_summary.md").read_text(encoding="utf-8") and "배치 확정 3일 전" in (SUMMARY_DIR / "final_executive_summary.md").read_text(encoding="utf-8") else "FAIL", detail=", ".join(docs)))
post.append(dict(check="로그(환경·실행 시간·강수 백오프)가 저장됨", result="PASS" if all((LOG_DIR / n).exists() for n in ("01_environment.json", "02_runtime.csv", "03_weather_backoff.csv")) else "FAIL", detail=f"실행 단계 {len(runtime)}개, 총 {runtime['elapsed_seconds_since_start'].max():,.0f}초"))
VALIDATION = pd.concat([VALIDATION_PRE, pd.DataFrame(post)], ignore_index=True)
VALIDATION["check_id"] = np.arange(1, len(VALIDATION) + 1)
VALIDATION.to_csv(TABLE_DIR / "17_validation_checks.csv", index=False, encoding="utf-8-sig")
(SUMMARY_DIR / "validation_results.md").write_text("# 검증 결과 (PASS/FAIL)\n\n" + md_table(VALIDATION[["check_id", "check", "result", "detail"]]) + "\n", encoding="utf-8")
n_pass, n_fail = int((VALIDATION["result"] == "PASS").sum()), int((VALIDATION["result"] == "FAIL").sum())
display(VALIDATION[["check_id", "check", "result"]])
print(f"최종 검산: 총 {len(VALIDATION)}개, PASS {n_pass}, FAIL {n_fail}")
assert n_fail == 0, "FAIL 검산이 있어 완료 처리하지 않습니다."
saved = sorted(p.relative_to(OUT_ROOT).as_posix() for p in OUT_ROOT.rglob("*") if p.is_file())
print("저장된 파일 수:", len(saved))

## 18. 최종 요약

In [ ]:
display(Markdown((SUMMARY_DIR / "final_executive_summary.md").read_text(encoding="utf-8")))
print(f"완료: 최종 권고 모델 {FINAL_MODEL} | WAPE {fin_o['WAPE']:.2f}% | 절감 {fin_q['saving_vs_mean7_krw']:,.0f}원 | 2026 {TOTAL_PD} person-days | 검산 PASS {n_pass}/{len(VALIDATION)}")